# YOLOv5-3D for PCB Defect Detection - Full Source

This notebook contains the complete source code for the YOLOv5-3D PCB defect detection project. It physically writes all required scripts to the local environment so it can run on any machine without hurdles.


## 1. Environment Setup
Create necessary directories to hold the modules.


In [ ]:
!mkdir -p utils models data scripts
!pip install -U albumentations


## Requirements


In [ ]:
%%writefile requirements.txt
torch>=2.0.0
torchvision>=0.15.0
numpy>=1.23.0,<2.0.0
scipy>=1.9.0
opencv-python>=4.7.0
Pillow>=9.0.0
albumentations==2.0.8
PyYAML>=6.0
tqdm>=4.64.0
pandas>=1.5.0
matplotlib>=3.6.0
seaborn>=0.12.0
onnx>=1.13.0
onnxruntime>=1.14.0
onnxslim>=0.1.8
tensorboard>=2.12.0


In [ ]:
!pip install -r requirements.txt


## 2. Core Project Files
Writing all necessary modules cell by cell.


### app.py
Definition and code for `app.py`.


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import cv2
import numpy as np
import os
import sys
from PIL import Image
import zipfile
import tempfile
from collections import OrderedDict

# Route Python to our local workspace to import the architectural modules
sys.path.insert(0, os.path.abspath(os.path.dirname(__file__)))
from models.yolo import attempt_load
from utils.metrics import non_max_suppression

# Configure Streamlit Page
st.set_page_config(layout="wide", page_title="Multi-Modal YOLOv5-3D Defect Detection")

# Inject Custom CSS for Clean Layout
st.markdown("""
    <style>
    /* Styling to match the clean horizontal radio layout */
    div.row-widget.stRadio > div {
        flex-direction: row;
        justify-content: flex-start;
        gap: 2rem;
    }
    div.row-widget.stRadio > div > label {
        cursor: pointer;
        padding: 5px 15px;
        border-radius: 5px;
        transition: background-color 0.3s;
    }
    div.row-widget.stRadio > div > label:hover {
        background-color: #f0f2f6;
    }
    /* Hide the sidebar completely as requested */
    [data-testid="collapsedControl"] {
        display: none;
    }
    section[data-testid="stSidebar"] {
        display: none;
    }
    </style>
""", unsafe_allow_html=True)

# Main Title matching layout style (Centered)
st.markdown("<h1 style='text-align: center;'>Identifying Defective PCBs through Multi-Modal Imaging</h1>", unsafe_allow_html=True)
st.markdown("<h4 style='text-align: center; color: gray;'>Bi-Directional Surface Defect Recognition with Geometric and Photometric Priors</h4>", unsafe_allow_html=True)
st.write("")

# Top Navigation Tabs
tab = st.radio("Navigation", ["🏠 Home", "⚙️ Prediction", "ℹ️ About"], horizontal=True, label_visibility="collapsed")
st.markdown("---")

# ================= MODEL LOADING FUNCTION =================
@st.cache_resource
def load_model(model_path):
    from models.yolo import YOLOv5_3D
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    try:
        model = attempt_load(model_path, map_location=device)
    except Exception as e:
        ckpt = torch.load(model_path, map_location=device, weights_only=False)
        state_dict = ckpt['ema' if ckpt.get('ema') else 'model'] if isinstance(ckpt, dict) and ('ema' in ckpt or 'model' in ckpt) else ckpt
        
        if isinstance(state_dict, OrderedDict) or isinstance(state_dict, dict):
            model = YOLOv5_3D(nc=12, input_size=1024)
            clean_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] if k.startswith('module.') else k
                clean_state_dict[name] = v
            model.load_state_dict(clean_state_dict, strict=False)
            model.to(device)
        else:
            raise RuntimeError(f"Unknown PyTorch model trace format: {type(state_dict)}")
            
    model.float()
    model.eval()
    return model, device

# ================= PROCESSING FUNCTION =================
def process_extracted_file(filepath, is_depth=False):
    """Safely handles the image/npy and forces 1024x1024."""
    try:
        if filepath.endswith('.npy'):
            img = np.load(filepath)
            img = np.array(img, dtype=np.float32)
            if is_depth and len(img.shape) == 3:
                img = img[:, :, 0]
            elif not is_depth:
                if len(img.shape) == 3 and img.shape[0] == 3:
                    img = np.transpose(img, (1, 2, 0))
                elif len(img.shape) == 2:
                    img = np.stack((img,)*3, axis=-1)
            
            if float(np.max(img)) <= 1.0:
                img = img * 255.0
            img = np.clip(img, 0, 255).astype(np.uint8)
        else:
            img = Image.open(filepath).convert('L' if is_depth else 'RGB')
            img = np.array(img, dtype=np.uint8)
            
        img = cv2.resize(img, (1024, 1024))
        return img
    except Exception as e:
        st.error(f"Error processing {filepath}: {e}")
        return np.zeros((1024, 1024, 3 if not is_depth else 1), dtype=np.uint8)

# Constants
names = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper', 'Missing_solder', 'Short_top', 'Excess_solder', 'Insufficient_solder', 'Missing_component', 'Spike']
colors = [(255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29), (207, 210, 49), (72, 249, 10), (46, 214, 255), (28, 115, 255), (140, 70, 255), (210, 83, 237), (255, 83, 172), (134, 134, 134)]


# ================= TABS LOGIC =================

if tab == "🏠 Home":
    st.markdown("<h3 style='text-align: center;'>Architecture Overview</h3>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center;'>Our system integrates RGB, Depth, and Illumination topographic features into a Bilinear Cross-Attention fusion module, breaking traditional 2D defect detection ceilings.</p>", unsafe_allow_html=True)
    
    # Check for architecture diagram
    arch_image = "Architecture.png"
    feed_image = "Images Feed into the Model.png"
    
    col1, col2, col3 = st.columns([1, 8, 1])
    with col2:
        if os.path.exists(feed_image):
            st.image(feed_image, use_container_width=True, caption="Input Data Streams into the Model")
            st.write("")
        
        if os.path.exists(arch_image):
            st.image(arch_image, use_container_width=True, caption="Cross-Attention Multi-Modal YOLOv5-3D Architecture")
        elif not os.path.exists(feed_image):
            st.info("💡 Place 'Architecture.png' and 'Images Feed into the Model.png' in the root directory to display the pipeline diagram here.")


elif tab == "⚙️ Prediction":
    st.markdown("### ⚙️ Engine Setup")
    model_path_input = st.text_input("Model Weights Path (e.g., C:/path/to/best.pt)", value="", help="Provide the absolute path to your trained .pt model weights.")
    
    conf_thres = st.slider("Confidence Threshold", 0.0, 1.0, 0.60, 0.05)
    iou_thres = st.slider("IoU Threshold", 0.0, 1.0, 0.45, 0.05)
    
    st.markdown("### 📦 Upload Data Package")
    st.markdown("Please upload a `.zip` file containing your 4 test items (RGB Image, Depth Array/Image, Illumination Array/Image, and Label).")
    
    zip_upload = st.file_uploader("Upload .zip package", type=["zip"])
    
    if st.button("🚀 Analyze Defect Topography", use_container_width=True):
        if not model_path_input or not os.path.exists(model_path_input):
            st.error("❌ Invalid model path. Please provide a valid independent path to your weights file.")
        elif not zip_upload:
            st.error("❌ Please upload the .zip test package first.")
        else:
            with st.spinner("Initializing Neural Engine..."):
                model, device = load_model(model_path_input)
                
            with st.spinner("Extracting & Processing Package..."):
                with tempfile.TemporaryDirectory() as tmpdir:
                    with zipfile.ZipFile(zip_upload, "r") as zip_ref:
                        zip_ref.extractall(tmpdir)
                        
                    rgb_path, depth_path, illum_path = None, None, None
                    
                    # Smart auto-detection of the unpacked files based on filenames
                    for root, _, files in os.walk(tmpdir):
                        for file in files:
                            lower_name = file.lower()
                            full_path = os.path.join(root, file)
                            
                            if "depth" in lower_name:
                                depth_path = full_path
                            elif "illum" in lower_name:
                                illum_path = full_path
                            elif lower_name.endswith(('.jpg', '.png', '.jpeg', '.npy')) and "label" not in lower_name and ".txt" not in lower_name:
                                rgb_path = full_path
                    
                    if not rgb_path or not depth_path or not illum_path:
                        st.error("❌ Failed to automatically identify all 3 components from the ZIP. Please ensure filenames clearly contain 'depth' and 'illumination'.")
                    else:
                        rgb_img = process_extracted_file(rgb_path, is_depth=False)
                        depth_img = process_extracted_file(depth_path, is_depth=True)
                        illum_img = process_extracted_file(illum_path, is_depth=False)
                        
                        # Show Input Previews
                        st.markdown("##### 🔍 Detected Topographic Maps")
                        p_c1, p_c2, p_c3 = st.columns(3)
                        p_c1.image(rgb_img, caption="RGB Color Base", use_container_width=True)
                        p_c2.image(depth_img, caption="Z-Axis Depth Mapping", use_container_width=True, clamp=True)
                        p_c3.image(illum_img, caption="Photometric Illumination", use_container_width=True)
                        
                        st.markdown("---")
                        
                        with st.spinner("Processing Cross-Attention Inference..."):
                            # Inference Prep
                            t_rgb = torch.from_numpy(rgb_img.transpose(2, 0, 1)).float().unsqueeze(0) / 255.0
                            t_dep = torch.from_numpy(depth_img).float().unsqueeze(0).unsqueeze(0) / 255.0
                            t_ill = torch.from_numpy(illum_img.transpose(2, 0, 1)).float().unsqueeze(0) / 255.0
                            
                            t_rgb, t_dep, t_ill = t_rgb.to(device), t_dep.to(device), t_ill.to(device)
                            
                            # Prediction
                            with torch.no_grad():
                                pred = model(t_rgb, t_dep, t_ill)[0]
                                
                            pred = non_max_suppression(pred, conf_thres, iou_thres)[0]
                            
                            res_img = rgb_img.copy()
                            
                            st.markdown("<h3 style='text-align: center;'>🎯 Diagnostic Output</h3>", unsafe_allow_html=True)
                            r_c1, r_c2 = st.columns([2, 1])
                            
                            with r_c1:
                                if pred is not None and len(pred):
                                    for *xyxy, conf, cls in pred:
                                        class_idx = int(cls)
                                        class_name = names[class_idx]
                                        color = colors[class_idx]
                                        
                                        c1, c2 = (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3]))
                                        cv2.rectangle(res_img, c1, c2, color, 4)
                                        cv2.putText(res_img, f"{class_name} {conf:.2f}", (c1[0], max(0, c1[1] - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 3)
                                        
                                        with r_c2:
                                            st.error(f"**Defect Found:** `{class_name}` \n\n**Confidence:** `{conf:.2%}`")
                                else:
                                    with r_c2:
                                        st.success("✅ **Board Verified:** Safe. No topological anomalies detected.")
                                        st.balloons()
                                        
                                st.image(res_img, caption="Final Rendered Structural Mapping", use_container_width=True)


elif tab == "ℹ️ About":
    st.markdown("### 👨‍💻 About the Project")
    st.markdown("""
    Welcome! I'm an engineer passionate about bridging the gap between hardware manufacturing and cutting-edge artificial intelligence.
    
    **The Problem:**
    Standard visual inspection in modern electronics focuses almost entirely on the flat, back-side of Printed Circuit Boards (PCBs). When checking the complex top-surface, traditional 2D cameras are blind to volumetric anomalies—like microscopic solder spikes or excess solder bulges—because they look identically reflective from a top-down view.
    
    **My Solution:**
    I engineered this **Multi-Modal YOLOv5-3D** architecture. By forcing the neural network to look at three simultaneous inputs:
    1. Standard Color (RGB)
    2. Physical Topography (Depth)
    3. Reflective Shadowing (Illumination)
    
    The custom *Bilinear Cross-Attention Fusion module* acts as a central brain, merging these three spatial domains to understand true 3D space, completely shattering historical accuracy ceilings.
    
    *Thank you for exploring this interface! Feel free to run a test package in the Prediction tab to see the engine in action.*
    """)



### create_test_packs.py
Definition and code for `create_test_packs.py`.


In [ ]:
%%writefile create_test_packs.py
import os
import zipfile

src_dir = r"D:\Data_Tuning\PCB_final\test"
images_dir = os.path.join(src_dir, "images")
depth_dir = os.path.join(src_dir, "depth")
illum_dir = os.path.join(src_dir, "illumination")
labels_dir = os.path.join(src_dir, "labels")

out_dir = r"d:\yolov5modfication\Test Packs"
os.makedirs(out_dir, exist_ok=True)

# Find 5 common stems
valid_stems = []
if os.path.exists(images_dir):
    for f in os.listdir(images_dir):
        if f.endswith('.jpg') or f.endswith('.png'):
            stem = os.path.splitext(f)[0]
            if os.path.exists(os.path.join(depth_dir, stem + ".npy")) and \
               os.path.exists(os.path.join(illum_dir, stem + ".npy")):
                valid_stems.append(stem)
            if len(valid_stems) >= 5:
                break

if not valid_stems:
    print("Could not find matching RGB, Depth, and Illumination files!")
else:
    for i, stem in enumerate(valid_stems):
        zip_path = os.path.join(out_dir, f"Test_Pack_{i+1}_{stem}.zip")
        with zipfile.ZipFile(zip_path, 'w') as zf:
            # Image
            img_file = stem + ".jpg" if os.path.exists(os.path.join(images_dir, stem + ".jpg")) else stem + ".png"
            zf.write(os.path.join(images_dir, img_file), img_file)
            
            # Depth - We force 'depth' into the filename so the Streamlit app can auto-detect it
            zf.write(os.path.join(depth_dir, stem + ".npy"), f"depth_{stem}.npy")
            
            # Illumination - We force 'illum' into the filename
            zf.write(os.path.join(illum_dir, stem + ".npy"), f"illum_{stem}.npy")
            
            # Label
            if os.path.exists(os.path.join(labels_dir, stem + ".txt")):
                zf.write(os.path.join(labels_dir, stem + ".txt"), f"label_{stem}.txt")
                
        print(f"Created: {zip_path}")



### Dataengineering-1.py
Definition and code for `Dataengineering-1.py`.


In [ ]:
%%writefile Dataengineering-1.py
"""
YOLO Dataset Analyzer
Dataset: D:\RES-SAC-2025-027\ModifcationProcess\yolov8 Training on Converted XML to TXT\pcb top
Structure:
    pcb top/
    ├── images/
    ├── labels/
    └── data.yaml
"""

import os
import sys
import glob
import yaml
import random
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter

try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    import matplotlib
    matplotlib.use("TkAgg")  # or "Agg" if no GUI needed
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

try:
    from PIL import Image
    HAS_PIL = True
except ImportError:
    HAS_PIL = False


# ═══════════════════════════════════════════════════════
#   CONFIGURE YOUR DATASET PATH HERE
# ═══════════════════════════════════════════════════════
DATASET_ROOT = r"C:\Data_Tuning\PCBtop"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "analysis_output")
# ═══════════════════════════════════════════════════════

IMAGES_DIR = os.path.join(DATASET_ROOT, "images")
LABELS_DIR = os.path.join(DATASET_ROOT, "labels")
YAML_PATH  = os.path.join(DATASET_ROOT, "data.yaml")

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


# ──────────────────────────────────────────────
# 1. UTILITIES
# ──────────────────────────────────────────────
def load_class_names(yaml_path: str) -> dict:
    if not os.path.isfile(yaml_path):
        return {}
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    if "names" not in data:
        return {}
    names = data["names"]
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    elif isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {}


def parse_yolo_label(label_path: str) -> list:
    annotations = []
    if not os.path.isfile(label_path):
        return annotations
    with open(label_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            annotations.append({
                "class_id": int(parts[0]),
                "x_center": float(parts[1]),
                "y_center": float(parts[2]),
                "width":    float(parts[3]),
                "height":   float(parts[4]),
            })
    return annotations


def get_image_size(image_path: str):
    if HAS_PIL:
        try:
            with Image.open(image_path) as im:
                return im.size
        except Exception:
            pass
    if HAS_CV2:
        img = cv2.imread(image_path)
        if img is not None:
            h, w = img.shape[:2]
            return w, h
    return None, None


# ──────────────────────────────────────────────
# 2. DATASET SCANNER
# ──────────────────────────────────────────────
def scan_dataset(images_dir: str, labels_dir: str):
    image_files = sorted([
        f for f in glob.glob(os.path.join(images_dir, "*"))
        if Path(f).suffix.lower() in IMG_EXTENSIONS
    ])

    stats = {
        "total_images":          len(image_files),
        "images_with_labels":    0,
        "images_without_labels": 0,
        "empty_labels":          0,
        "total_instances":       0,
        "class_instance_count":  Counter(),
        "class_image_count":     Counter(),
        "bbox_widths":           defaultdict(list),
        "bbox_heights":          defaultdict(list),
        "bbox_areas":            defaultdict(list),
        "bbox_aspect_ratios":    defaultdict(list),
        "instances_per_image":   [],
        "image_widths":          [],
        "image_heights":         [],
        "per_image_details":     [],
        "corrupted_labels":      [],
        "missing_labels":        [],
    }

    print(f"\n  Scanning {len(image_files)} images...")

    for i, img_path in enumerate(image_files):
        if (i + 1) % 200 == 0 or (i + 1) == len(image_files):
            print(f"    Processed {i + 1}/{len(image_files)} images", end="\r")

        stem = Path(img_path).stem
        label_path = os.path.join(labels_dir, stem + ".txt")

        img_w, img_h = get_image_size(img_path)
        if img_w:
            stats["image_widths"].append(img_w)
            stats["image_heights"].append(img_h)

        # No label file
        if not os.path.isfile(label_path):
            stats["images_without_labels"] += 1
            stats["instances_per_image"].append(0)
            stats["missing_labels"].append(img_path)
            continue

        annotations = parse_yolo_label(label_path)
        stats["images_with_labels"] += 1

        # Empty label
        if len(annotations) == 0:
            stats["empty_labels"] += 1
            stats["instances_per_image"].append(0)
            continue

        stats["instances_per_image"].append(len(annotations))
        stats["total_instances"] += len(annotations)

        classes_in_image = set()
        for ann in annotations:
            cid = ann["class_id"]
            stats["class_instance_count"][cid] += 1
            classes_in_image.add(cid)
            stats["bbox_widths"][cid].append(ann["width"])
            stats["bbox_heights"][cid].append(ann["height"])
            stats["bbox_areas"][cid].append(ann["width"] * ann["height"])
            if ann["height"] > 0:
                stats["bbox_aspect_ratios"][cid].append(ann["width"] / ann["height"])

        for cid in classes_in_image:
            stats["class_image_count"][cid] += 1

        stats["per_image_details"].append({
            "image_path":  img_path,
            "label_path":  label_path,
            "annotations": annotations,
            "img_w":       img_w,
            "img_h":       img_h,
        })

    print()  # newline after progress
    return stats


# ──────────────────────────────────────────────
# 3. CONSOLE REPORT
# ──────────────────────────────────────────────
def print_report(stats: dict, class_names: dict):
    sep = "=" * 85
    thin = "-" * 85

    print(f"\n{sep}")
    print(f"  📁 DATASET ANALYSIS REPORT")
    print(f"  📂 {DATASET_ROOT}")
    print(sep)
    print(f"  Total images              : {stats['total_images']}")
    print(f"  Images with labels        : {stats['images_with_labels']}")
    print(f"  Images without labels     : {stats['images_without_labels']}")
    print(f"  Empty label files         : {stats['empty_labels']}")
    print(f"  Total object instances    : {stats['total_instances']}")
    print(f"  Unique classes found      : {len(stats['class_instance_count'])}")

    # Instances per image
    if stats["instances_per_image"]:
        arr = np.array(stats["instances_per_image"])
        print(f"\n  📊 Instances per image:")
        print(f"    Min    : {arr.min()}")
        print(f"    Max    : {arr.max()}")
        print(f"    Mean   : {arr.mean():.2f}")
        print(f"    Median : {np.median(arr):.1f}")
        print(f"    Std    : {arr.std():.2f}")

    # Image resolution
    if stats["image_widths"]:
        w = np.array(stats["image_widths"])
        h = np.array(stats["image_heights"])
        print(f"\n  🖼️  Image resolution (pixels):")
        print(f"    Width  — Min: {w.min():>5}  Max: {w.max():>5}  Mean: {w.mean():>7.0f}")
        print(f"    Height — Min: {h.min():>5}  Max: {h.max():>5}  Mean: {h.mean():>7.0f}")
        unique_res = set(zip(w.tolist(), h.tolist()))
        if len(unique_res) <= 10:
            print(f"    Unique resolutions: {sorted(unique_res)}")

    # ── Per-class table ──
    all_classes = sorted(stats["class_instance_count"].keys())
    if all_classes:
        print(f"\n  📋 PER-CLASS BREAKDOWN")
        print(f"  {thin}")
        print(f"  {'ID':<5} {'Class Name':<28} {'Instances':>10} {'Images':>8} "
              f"{'AvgW':>7} {'AvgH':>7} {'AvgArea':>10} {'AvgAR':>7}")
        print(f"  {thin}")
        for cid in all_classes:
            name  = class_names.get(cid, f"class_{cid}")
            inst  = stats["class_instance_count"][cid]
            imgs  = stats["class_image_count"][cid]
            avg_w = np.mean(stats["bbox_widths"][cid])
            avg_h = np.mean(stats["bbox_heights"][cid])
            avg_a = np.mean(stats["bbox_areas"][cid])
            avg_r = np.mean(stats["bbox_aspect_ratios"].get(cid, [0]))
            print(f"  {cid:<5} {name:<28} {inst:>10} {imgs:>8} "
                  f"{avg_w:>7.4f} {avg_h:>7.4f} {avg_a:>10.6f} {avg_r:>7.2f}")
        print(f"  {thin}")
        print(f"  {'':5} {'TOTAL':<28} {stats['total_instances']:>10} {'':>8}")

    # ── Size categories ──
    if all_classes:
        print(f"\n  📏 BBOX SIZE CATEGORIES (COCO-style on normalized area)")
        print(f"  {thin}")
        print(f"  {'ID':<5} {'Class Name':<28} {'Small':>8} {'Medium':>8} {'Large':>8}")
        print(f"  {thin}")
        for cid in all_classes:
            name  = class_names.get(cid, f"class_{cid}")
            areas = np.array(stats["bbox_areas"][cid])
            small  = int(np.sum(areas < 0.01))     # < 1% of image
            medium = int(np.sum((areas >= 0.01) & (areas < 0.09)))  # 1-9%
            large  = int(np.sum(areas >= 0.09))     # > 9% of image
            print(f"  {cid:<5} {name:<28} {small:>8} {medium:>8} {large:>8}")

    # ── Imbalance ──
    if len(all_classes) > 1:
        values  = [stats["class_instance_count"][c] for c in all_classes]
        total   = sum(values)
        max_cnt = max(values)
        min_cnt = min(values)
        cv      = np.std(values) / np.mean(values) * 100

        print(f"\n  ⚖️  CLASS IMBALANCE ANALYSIS")
        print(f"  {thin}")
        print(f"  Imbalance Ratio (max/min)  : {max_cnt / min_cnt:.2f}")
        print(f"  Coefficient of Variation   : {cv:.1f}%")
        status = "✅ Balanced" if cv < 30 else ("⚠️  Moderate" if cv < 60 else "🔴 Highly imbalanced")
        print(f"  Status                     : {status}\n")

        print(f"  {'ID':<5} {'Class Name':<28} {'Count':>8} {'%':>7}  Distribution")
        print(f"  {thin}")
        for cid in all_classes:
            name  = class_names.get(cid, f"class_{cid}")
            cnt   = stats["class_instance_count"][cid]
            pct   = cnt / total * 100
            ratio = cnt / max_cnt
            bar   = "█" * int(ratio * 40) + "░" * (40 - int(ratio * 40))
            print(f"  {cid:<5} {name:<28} {cnt:>8} {pct:>6.1f}%  {bar}")

    # ── Missing labels ──
    if stats["missing_labels"]:
        print(f"\n  ⚠️  Images WITHOUT label files ({len(stats['missing_labels'])}):")
        for p in stats["missing_labels"][:10]:
            print(f"    • {Path(p).name}")
        if len(stats["missing_labels"]) > 10:
            print(f"    ... and {len(stats['missing_labels']) - 10} more")

    print(f"\n{sep}\n")


# ──────────────────────────────────────────────
# 4. PLOTS
# ──────────────────────────────────────────────
def generate_plots(stats: dict, class_names: dict, output_dir: str):
    if not HAS_PLT:
        print("  [WARN] matplotlib not installed — run: pip install matplotlib")
        return
    if not stats["class_instance_count"]:
        print("  [WARN] No annotations found — skipping plots.")
        return

    os.makedirs(output_dir, exist_ok=True)
    colors = plt.cm.tab20.colors
    all_classes = sorted(stats["class_instance_count"].keys())
    names = [class_names.get(c, f"class_{c}") for c in all_classes]
    bar_colors = [colors[i % len(colors)] for i in range(len(all_classes))]

    # ── 1. Class distribution ──
    fig, axes = plt.subplots(1, 2, figsize=(16, max(6, len(all_classes) * 0.55)))
    fig.suptitle("Class Distribution — PCB Top Dataset", fontsize=16, fontweight="bold")

    inst = [stats["class_instance_count"][c] for c in all_classes]
    imgs = [stats["class_image_count"][c]    for c in all_classes]

    axes[0].barh(names, inst, color=bar_colors, edgecolor="black", linewidth=0.5)
    axes[0].set_xlabel("Bounding-box Instances")
    axes[0].set_title("Instances per Class")
    for i, v in enumerate(inst):
        axes[0].text(v + max(inst) * 0.01, i, str(v), va="center", fontsize=9)

    axes[1].barh(names, imgs, color=bar_colors, edgecolor="black", linewidth=0.5)
    axes[1].set_xlabel("Number of Images")
    axes[1].set_title("Images Containing Class")
    for i, v in enumerate(imgs):
        axes[1].text(v + max(imgs) * 0.01, i, str(v), va="center", fontsize=9)

    plt.tight_layout()
    p = os.path.join(output_dir, "01_class_distribution.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p}")
    plt.close()

    # ── 2. Pie chart ──
    fig, ax = plt.subplots(figsize=(9, 9))
    wedges, texts, autotexts = ax.pie(
        inst, labels=names, autopct="%1.1f%%",
        colors=bar_colors, startangle=90, pctdistance=0.82
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title("Instance Distribution (Pie)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    p = os.path.join(output_dir, "02_class_pie.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p}")
    plt.close()

    # ── 3. BBox size box-plots ──
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, len(all_classes) * 0.55)))
    fig.suptitle("BBox Size Distribution (normalized YOLO coords)", fontsize=16, fontweight="bold")

    for ax, key, title, xlabel in [
        (axes[0], "bbox_widths",  "Widths",  "Normalized Width"),
        (axes[1], "bbox_heights", "Heights", "Normalized Height"),
        (axes[2], "bbox_areas",   "Areas",   "Normalized Area (w×h)"),
    ]:
        bp = ax.boxplot(
            [stats[key][c] for c in all_classes],
            vert=False, labels=names, patch_artist=True,
            flierprops=dict(marker=".", markersize=3, alpha=0.4)
        )
        for patch, col in zip(bp["boxes"], bar_colors):
            patch.set_facecolor(col)
            patch.set_alpha(0.8)
        ax.set_xlabel(xlabel)
        ax.set_title(title)

    plt.tight_layout()
    p = os.path.join(output_dir, "03_bbox_sizes.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p}")
    plt.close()

    # ── 4. Instances per image histogram ──
    fig, ax = plt.subplots(figsize=(10, 5))
    arr = np.array(stats["instances_per_image"])
    max_val = int(arr.max())
    bins = range(0, max_val + 2)
    ax.hist(arr, bins=bins, edgecolor="black", alpha=0.75, color="steelblue")
    ax.axvline(arr.mean(), color="red", ls="--", lw=2, label=f"Mean = {arr.mean():.1f}")
    ax.axvline(np.median(arr), color="orange", ls="--", lw=2, label=f"Median = {np.median(arr):.1f}")
    ax.set_xlabel("Objects per Image")
    ax.set_ylabel("Number of Images")
    ax.set_title("Instances per Image Distribution")
    ax.legend()
    plt.tight_layout()
    p = os.path.join(output_dir, "04_instances_per_image.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p}")
    plt.close()

    # ── 5. BBox center heatmap ──
    cx_all, cy_all = [], []
    for d in stats["per_image_details"]:
        for a in d["annotations"]:
            cx_all.append(a["x_center"])
            cy_all.append(a["y_center"])
    if cx_all:
        fig, ax = plt.subplots(figsize=(7, 7))
        hm, _, _ = np.histogram2d(cx_all, cy_all, bins=50, range=[[0, 1], [0, 1]])
        im = ax.imshow(hm.T, origin="upper", extent=[0, 1, 1, 0], cmap="YlOrRd", aspect="equal")
        plt.colorbar(im, ax=ax, label="Count")
        ax.set_xlabel("x_center (normalized)")
        ax.set_ylabel("y_center (normalized)")
        ax.set_title("BBox Center Heatmap")
        plt.tight_layout()
        p = os.path.join(output_dir, "05_center_heatmap.png")
        plt.savefig(p, dpi=150, bbox_inches="tight")
        print(f"  ✅ {p}")
        plt.close()

    # ── 6. Width vs Height scatter ──
    fig, ax = plt.subplots(figsize=(8, 8))
    for i, cid in enumerate(all_classes):
        ax.scatter(
            stats["bbox_widths"][cid], stats["bbox_heights"][cid],
            s=10, alpha=0.35, color=colors[i % len(colors)],
            label=class_names.get(cid, f"class_{cid}")
        )
    ax.set_xlabel("Normalized Width")
    ax.set_ylabel("Normalized Height")
    ax.set_title("BBox Width vs Height (per class)")
    ax.legend(markerscale=4, fontsize=8, loc="upper right")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Square")
    plt.tight_layout()
    p = os.path.join(output_dir, "06_width_vs_height.png")
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  ✅ {p}")
    plt.close()

    # ── 7. Per-class center heatmaps ──
    if len(all_classes) <= 20:
        cols = min(4, len(all_classes))
        rows = (len(all_classes) + cols - 1) // cols
        fig, axes_grid = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
        fig.suptitle("Per-Class BBox Center Heatmaps", fontsize=16, fontweight="bold")

        if not isinstance(axes_grid, np.ndarray):
            axes_grid = np.array([[axes_grid]])
        axes_grid = np.atleast_2d(axes_grid)

        for idx, cid in enumerate(all_classes):
            r, c = divmod(idx, cols)
            ax = axes_grid[r][c] if rows > 1 else axes_grid[0][c]
            cx = [a["x_center"] for d in stats["per_image_details"] for a in d["annotations"] if a["class_id"] == cid]
            cy = [a["y_center"] for d in stats["per_image_details"] for a in d["annotations"] if a["class_id"] == cid]
            if cx:
                hm, _, _ = np.histogram2d(cx, cy, bins=30, range=[[0, 1], [0, 1]])
                ax.imshow(hm.T, origin="upper", extent=[0, 1, 1, 0], cmap="YlOrRd", aspect="equal")
            name = class_names.get(cid, f"class_{cid}")
            ax.set_title(f"{name} ({len(cx)})", fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])

        for idx in range(len(all_classes), rows * cols):
            r, c = divmod(idx, cols)
            ax = axes_grid[r][c] if rows > 1 else axes_grid[0][c]
            ax.axis("off")

        plt.tight_layout()
        p = os.path.join(output_dir, "07_per_class_heatmaps.png")
        plt.savefig(p, dpi=150, bbox_inches="tight")
        print(f"  ✅ {p}")
        plt.close()

    # ── 8. Sample annotated images ──
    if HAS_CV2 and stats["per_image_details"]:
        n_samples = min(9, len(stats["per_image_details"]))
        samples = random.sample(stats["per_image_details"], n_samples)
        cols = 3
        rows = (n_samples + cols - 1) // cols
        fig, axes_grid = plt.subplots(rows, cols, figsize=(6 * cols, 6 * rows))
        fig.suptitle("Sample Annotated Images — PCB Top", fontsize=16, fontweight="bold")

        if not isinstance(axes_grid, np.ndarray):
            axes_grid = np.array([[axes_grid]])
        axes_grid = np.atleast_2d(axes_grid)
        if axes_grid.ndim == 1:
            axes_grid = axes_grid[np.newaxis, :]

        for idx, detail in enumerate(samples):
            r, c = divmod(idx, cols)
            ax = axes_grid[r][c]
            img = cv2.imread(detail["image_path"])
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ih, iw = img.shape[:2]
            ax.imshow(img)
            for ann in detail["annotations"]:
                cx  = ann["x_center"] * iw
                cy  = ann["y_center"] * ih
                bw  = ann["width"]    * iw
                bh  = ann["height"]   * ih
                x1  = cx - bw / 2
                y1  = cy - bh / 2
                cid = ann["class_id"]
                col = colors[cid % len(colors)]
                rect = patches.Rectangle(
                    (x1, y1), bw, bh, linewidth=2,
                    edgecolor=col, facecolor="none"
                )
                ax.add_patch(rect)
                lbl = class_names.get(cid, f"cls_{cid}")
                ax.text(x1, y1 - 3, lbl, fontsize=7, color="white",
                        bbox=dict(facecolor=col, alpha=0.75, pad=1))
            ax.set_title(Path(detail["image_path"]).name, fontsize=8)
            ax.axis("off")

        for idx in range(n_samples, rows * cols):
            r, c = divmod(idx, cols)
            axes_grid[r][c].axis("off")

        plt.tight_layout()
        p = os.path.join(output_dir, "08_sample_images.png")
        plt.savefig(p, dpi=150, bbox_inches="tight")
        print(f"  ✅ {p}")
        plt.close()


# ──────────────────────────────────────────────
# 5. MAIN
# ──────────────────────────────────────────────
def main():
    print("\n" + "=" * 85)
    print("  YOLO DATASET ANALYZER")
    print("=" * 85)
    print(f"  Dataset : {DATASET_ROOT}")
    print(f"  Images  : {IMAGES_DIR}")
    print(f"  Labels  : {LABELS_DIR}")
    print(f"  YAML    : {YAML_PATH}")
    print(f"  Output  : {OUTPUT_DIR}")

    # ── Validate paths ──
    if not os.path.isdir(DATASET_ROOT):
        sys.exit(f"\n  ❌ Dataset root not found: {DATASET_ROOT}")
    if not os.path.isdir(IMAGES_DIR):
        sys.exit(f"\n  ❌ images/ folder not found: {IMAGES_DIR}")
    if not os.path.isdir(LABELS_DIR):
        sys.exit(f"\n  ❌ labels/ folder not found: {LABELS_DIR}")

    # ── Load class names ──
    class_names = load_class_names(YAML_PATH)
    if class_names:
        print(f"\n  ✅ Loaded {len(class_names)} classes from data.yaml:")
        for cid, name in sorted(class_names.items()):
            print(f"      {cid}: {name}")
    else:
        print("\n  ⚠️  No class names loaded — using numeric IDs.")

    # ── Scan ──
    stats = scan_dataset(IMAGES_DIR, LABELS_DIR)

    # ── Report ──
    print_report(stats, class_names)

    # ── Plots ──
    print(f"  📊 Generating plots → {OUTPUT_DIR}/")
    generate_plots(stats, class_names, OUTPUT_DIR)

    print(f"\n  🎉 Analysis complete! Check: {OUTPUT_DIR}\n")


if __name__ == "__main__":
    main()


### debug_inference.py
Definition and code for `debug_inference.py`.


In [ ]:
%%writefile debug_inference.py
"""Analyze state dict structures for weight transfer."""
import torch
import torch.nn as nn
import sys
import pathlib
sys.path.insert(0, '.')

# Fix PosixPath on Windows
pathlib.PosixPath = pathlib.WindowsPath

# Add stub classes for unpickling
import models.yolo as yolo_module
import models.common as common_module
for cls_name in ['C3', 'Conv', 'SPPF', 'Bottleneck', 'Concat']:
    if hasattr(common_module, cls_name) and not hasattr(yolo_module, cls_name):
        setattr(yolo_module, cls_name, getattr(common_module, cls_name))
if not hasattr(yolo_module, 'Detect'):
    setattr(yolo_module, 'Detect', yolo_module.Detections)

# Load pretrained YOLOv5s
print("=" * 60)
print("PRETRAINED YOLOv5s STATE DICT")
print("=" * 60)
ckpt = torch.load('weights/yolov5s.pt', map_location='cpu', weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys())}")

pretrained = ckpt['model'] if 'model' in ckpt else ckpt
if hasattr(pretrained, 'state_dict'):
    pretrained_sd = pretrained.float().state_dict()
elif hasattr(pretrained, 'float'):
    pretrained_sd = pretrained.float().state_dict()
elif isinstance(pretrained, dict):
    pretrained_sd = pretrained
else:
    pretrained_sd = pretrained

print(f"\nPretrained layers: {len(pretrained_sd)}")
for k, v in list(pretrained_sd.items())[:50]:
    print(f"  {k:55s} {str(v.shape):20s}")
print(f"  ... ({len(pretrained_sd)} total)")

# Load YOLOv5-3D
print("\n" + "=" * 60)
print("YOLOv5-3D STATE DICT")
print("=" * 60)
from models.yolo import YOLOv5_3D
model = YOLOv5_3D(nc=12, input_size=640, freeze_backbone=False)
model_sd = model.state_dict()

print(f"\nModel layers: {len(model_sd)}")
for k, v in list(model_sd.items())[:50]:
    print(f"  {k:55s} {str(v.shape):20s}")
print(f"  ... ({len(model_sd)} total)")



### detect.py
Definition and code for `detect.py`.


In [ ]:
%%writefile detect.py
"""
YOLOv5-3D Detection Script
Run inference on images with illumination-depth support.
"""

import os
import sys
import argparse
import time
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import torch
from torch.amp import autocast

# Add project root to path
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.yolo import YOLOv5_3D
from utils.metrics import non_max_suppression
from utils.plots import plot_detection_results
from utils.general import colorstr, check_img_size, scale_coords


class Detector:
    """
    YOLOv5-3D Detector class for inference.
    
    Supports:
    - RGB only inference (no illumination-depth)
    - RGB + Illumination + Depth inference
    - Batch inference
    - Video inference
    """
    
    def __init__(
        self,
        weights: str,
        device: str = '',
        imgsz: int = 640,
        conf_thres: float = 0.25,
        iou_thres: float = 0.45,
        max_det: int = 300,
        amp: bool = True,
        classes: Optional[List[str]] = None
    ):
        """
        Initialize Detector.
        
        Args:
            weights: Path to model weights
            device: Device (cuda:0, cpu)
            imgsz: Inference image size
            conf_thres: Confidence threshold
            iou_thres: IoU threshold for NMS
            max_det: Maximum detections per image
            amp: Use automatic mixed precision
            classes: List of class names
        """
        self.device = self._select_device(device)
        self.imgsz = check_img_size(imgsz)
        self.conf_thres = conf_thres
        self.iou_thres = iou_thres
        self.max_det = max_det
        self.amp = amp
        self.classes = classes
        
        # Load model
        print(f"{colorstr('blue')}Loading model...{colorstr('reset')}")
        self.model = self._load_model(weights)
        self.model.eval()
        
        # Class names
        if classes:
            self.names = classes
        else:
            # Default PCB defect classes
            self.names = ['missing_hole', 'mouse_bite', 'open_circuit', 
                         'short', 'spur', 'spurious_copper']
        
        # Warmup
        self._warmup()
        
        print(f"{colorstr('green')}Model loaded successfully!{colorstr('reset')}")
    
    def _select_device(self, device: str) -> torch.device:
        """Select device."""
        if device:
            return torch.device(device)
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    def _load_model(self, weights: str) -> YOLOv5_3D:
        """Load model from weights."""
        ckpt = torch.load(weights, map_location=self.device)
        
        if 'model' in ckpt:
            state_dict = ckpt['model']
            if hasattr(state_dict, 'state_dict'):
                state_dict = state_dict.state_dict()
            
            # Get number of classes from checkpoint
            nc = ckpt.get('hyp', {}).get('nc', 6)
            if isinstance(state_dict, dict) and 'detect.nc' in state_dict:
                nc = state_dict['detect.nc']
            
            model = YOLOv5_3D(nc=nc).to(self.device)
            model.load_state_dict(state_dict)
        else:
            # Assume it's a state dict
            model = YOLOv5_3D().to(self.device)
            model.load_state_dict(ckpt)
        
        return model
    
    def _warmup(self):
        """Warmup model."""
        print(f"{colorstr('yellow')}Warming up model...{colorstr('reset')}")
        
        with torch.no_grad():
            # RGB only warmup
            dummy_rgb = torch.zeros(1, 3, self.imgsz, self.imgsz).to(self.device)
            dummy_depth = torch.zeros(1, 1, self.imgsz, self.imgsz).to(self.device)
            dummy_illum = torch.zeros(1, 3, self.imgsz, self.imgsz).to(self.device)
            
            with autocast('cuda', enabled=self.amp):
                _ = self.model(dummy_rgb, dummy_depth, dummy_illum)
        
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    def preprocess_image(self, image: np.ndarray) -> torch.Tensor:
        """
        Preprocess image for inference.
        
        Args:
            image: BGR image (OpenCV format)
        
        Returns:
            Preprocessed tensor
        """
        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Resize
        h, w = image.shape[:2]
        r = self.imgsz / max(h, w)
        if r != 1:
            new_shape = (int(w * r), int(h * r))
            image = cv2.resize(image, new_shape, interpolation=cv2.INTER_LINEAR)
        
        # Pad to square
        h, w = image.shape[:2]
        pad_h = self.imgsz - h
        pad_w = self.imgsz - w
        
        if pad_h > 0 or pad_w > 0:
            image = cv2.copyMakeBorder(
                image, 0, pad_h, 0, pad_w, 
                cv2.BORDER_CONSTANT, value=(114, 114, 114)
            )
        
        # Normalize and convert to tensor
        image = image.transpose(2, 0, 1)  # HWC -> CHW
        image = np.ascontiguousarray(image, dtype=np.float32)
        image = image / 255.0
        
        return torch.from_numpy(image).unsqueeze(0).to(self.device)
    
    def load_depth(self, depth_path: str) -> torch.Tensor:
        """Load depth map from .npy file."""
        if not os.path.exists(depth_path):
            return torch.zeros(1, 1, self.imgsz, self.imgsz).to(self.device)
        
        depth = np.load(depth_path)
        
        # Handle different formats
        if depth.ndim == 2:
            depth = depth[:, :, np.newaxis]
        elif depth.ndim == 3 and depth.shape[0] == 1:
            depth = depth.transpose(1, 2, 0)
        
        # Normalize
        if depth.max() > 0:
            depth = depth / depth.max()
        
        # Resize
        depth = cv2.resize(depth, (self.imgsz, self.imgsz))
        
        # Convert to tensor
        depth = depth.transpose(2, 0, 1)
        depth = np.ascontiguousarray(depth, dtype=np.float32)
        
        return torch.from_numpy(depth).unsqueeze(0).to(self.device)
    
    def load_illumination(self, illum_path: str) -> torch.Tensor:
        """Load illumination map from .npy file."""
        if not os.path.exists(illum_path):
            return torch.zeros(1, 3, self.imgsz, self.imgsz).to(self.device)
        
        illum = np.load(illum_path)
        
        # Handle different formats
        if illum.ndim == 2:
            illum = np.stack([illum] * 3, axis=-1)
        elif illum.ndim == 3 and illum.shape[0] == 3:
            illum = illum.transpose(1, 2, 0)
        
        # Normalize
        if illum.max() > 0:
            illum = illum / illum.max()
        
        # Resize
        illum = cv2.resize(illum, (self.imgsz, self.imgsz))
        
        # Convert to tensor
        illum = illum.transpose(2, 0, 1)
        illum = np.ascontiguousarray(illum, dtype=np.float32)
        
        return torch.from_numpy(illum).unsqueeze(0).to(self.device)
    
    @torch.no_grad()
    def predict(
        self,
        image: np.ndarray,
        depth: Optional[np.ndarray] = None,
        illumination: Optional[np.ndarray] = None
    ) -> Tuple[np.ndarray, float]:
        """
        Run inference on a single image.
        
        Args:
            image: BGR image (OpenCV format)
            depth: Optional depth map (H, W, 1)
            illumination: Optional illumination map (H, W, 3)
        
        Returns:
            detections: (N, 6) [x1, y1, x2, y2, conf, cls]
            inference_time: Inference time in ms
        """
        # Preprocess
        img_tensor = self.preprocess_image(image)
        
        # Load auxiliary data if provided
        if depth is not None:
            if isinstance(depth, np.ndarray):
                depth_tensor = torch.from_numpy(depth).float().to(self.device)
                if depth_tensor.ndim == 2:
                    depth_tensor = depth_tensor.unsqueeze(0).unsqueeze(0)
                elif depth_tensor.ndim == 3:
                    depth_tensor = depth_tensor.permute(2, 0, 1).unsqueeze(0)
            else:
                depth_tensor = torch.zeros(1, 1, self.imgsz, self.imgsz).to(self.device)
        else:
            depth_tensor = torch.zeros(1, 1, self.imgsz, self.imgsz).to(self.device)
        
        if illumination is not None:
            if isinstance(illumination, np.ndarray):
                illum_tensor = torch.from_numpy(illumination).float().to(self.device)
                if illum_tensor.ndim == 3:
                    illum_tensor = illum_tensor.permute(2, 0, 1).unsqueeze(0)
            else:
                illum_tensor = torch.zeros(1, 3, self.imgsz, self.imgsz).to(self.device)
        else:
            illum_tensor = torch.zeros(1, 3, self.imgsz, self.imgsz).to(self.device)
        
        # Inference
        start_time = time.time()
        
        with autocast('cuda', enabled=self.amp):
            outputs = self.model(img_tensor, depth_tensor, illum_tensor)
        
        inference_time = (time.time() - start_time) * 1000
        
        # Post-process
        pred = outputs[0] if isinstance(outputs, tuple) else outputs
        pred = pred.cpu().numpy()
        detections = non_max_suppression(
            pred, 
            self.conf_thres, 
            self.iou_thres,
            max_det=self.max_det
        )[0]
        
        # Scale boxes to original image size
        if detections.shape[0] > 0:
            h, w = image.shape[:2]
            detections[:, [0, 2]] *= w / self.imgsz
            detections[:, [1, 3]] *= h / self.imgsz
        
        return detections, inference_time
    
    def predict_from_files(
        self,
        image_path: str,
        depth_path: Optional[str] = None,
        illum_path: Optional[str] = None
    ) -> Tuple[np.ndarray, float]:
        """
        Run inference from file paths.
        
        Args:
            image_path: Path to image file
            depth_path: Path to depth .npy file
            illum_path: Path to illumination .npy file
        
        Returns:
            detections: (N, 6) [x1, y1, x2, y2, conf, cls]
            inference_time: Inference time in ms
        """
        # Load image
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Could not load image: {image_path}")
        
        # Load depth and illumination
        depth = self.load_depth(depth_path) if depth_path else None
        illum = self.load_illumination(illum_path) if illum_path else None
        
        return self.predict(image, depth, illum)


def run_inference(
    source: str,
    weights: str,
    depth_dir: Optional[str] = None,
    illum_dir: Optional[str] = None,
    output: str = 'runs/detect',
    imgsz: int = 640,
    conf_thres: float = 0.25,
    iou_thres: float = 0.45,
    device: str = '',
    save_txt: bool = False,
    save_img: bool = True,
    classes: Optional[List[str]] = None
):
    """
    Run inference on images or directory.
    
    Args:
        source: Image file or directory
        weights: Model weights path
        depth_dir: Directory containing depth .npy files
        illum_dir: Directory containing illumination .npy files
        output: Output directory
        imgsz: Image size
        conf_thres: Confidence threshold
        iou_thres: IoU threshold
        device: Device
        save_txt: Save results as txt
        save_img: Save annotated images
        classes: Class names
    """
    # Initialize detector
    detector = Detector(
        weights=weights,
        device=device,
        imgsz=imgsz,
        conf_thres=conf_thres,
        iou_thres=iou_thres,
        classes=classes
    )
    
    # Setup output
    output_dir = Path(output)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Get image files
    source = Path(source)
    if source.is_file():
        image_files = [source]
    else:
        image_files = list(source.glob('*.jpg')) + \
                      list(source.glob('*.png')) + \
                      list(source.glob('*.jpeg'))
    
    print(f"\n{colorstr('blue')}Processing {len(image_files)} images...{colorstr('reset')}\n")
    
    total_time = 0
    total_detections = 0
    
    for img_path in image_files:
        # Get auxiliary paths
        depth_path = None
        illum_path = None
        
        if depth_dir:
            depth_path = Path(depth_dir) / f"{img_path.stem}.npy"
            if not depth_path.exists():
                depth_path = None
        
        if illum_dir:
            illum_path = Path(illum_dir) / f"{img_path.stem}.npy"
            if not illum_path.exists():
                illum_path = None
        
        # Run inference
        detections, inf_time = detector.predict_from_files(
            str(img_path),
            str(depth_path) if depth_path else None,
            str(illum_path) if illum_path else None
        )
        
        total_time += inf_time
        total_detections += len(detections)
        
        # Print results
        print(f"{img_path.name}: {len(detections)} detections, {inf_time:.1f}ms")
        
        # Save txt
        if save_txt:
            txt_path = output_dir / f"{img_path.stem}.txt"
            with open(txt_path, 'w') as f:
                for det in detections:
                    # Save in YOLO format
                    x1, y1, x2, y2, conf, cls = det
                    img = cv2.imread(str(img_path))
                    h, w = img.shape[:2]
                    x_center = (x1 + x2) / 2 / w
                    y_center = (y1 + y2) / 2 / h
                    bw = (x2 - x1) / w
                    bh = (y2 - y1) / h
                    f.write(f"{int(cls)} {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f} {conf:.4f}\n")
        
        # Save image
        if save_img:
            img = cv2.imread(str(img_path))
            out_path = output_dir / img_path.name
            
            # Draw detections
            for det in detections:
                x1, y1, x2, y2, conf, cls = det
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                cls = int(cls)
                
                # Draw box
                color = (0, 255, 0)  # Green
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                # Draw label
                label = f"{detector.names[cls]} {conf:.2f}"
                cv2.putText(img, label, (x1, y1 - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            
            cv2.imwrite(str(out_path), img)
    
    # Summary
    print(f"\n{colorstr('green')}Inference Complete!{colorstr('reset')}")
    print(f"  Total images: {len(image_files)}")
    print(f"  Total detections: {total_detections}")
    print(f"  Average time: {total_time / len(image_files):.1f}ms")
    print(f"  Results saved to: {output_dir}")


def main():
    parser = argparse.ArgumentParser(description='YOLOv5-3D Detection')
    
    parser.add_argument('--source', type=str, required=True, 
                       help='Image file or directory')
    parser.add_argument('--weights', type=str, required=True, 
                       help='Model weights path')
    parser.add_argument('--depth-dir', type=str, default=None, 
                       help='Directory containing depth .npy files')
    parser.add_argument('--illum-dir', type=str, default=None, 
                       help='Directory containing illumination .npy files')
    parser.add_argument('--output', type=str, default='runs/detect', 
                       help='Output directory')
    parser.add_argument('--imgsz', type=int, default=640, 
                       help='Image size')
    parser.add_argument('--conf-thres', type=float, default=0.25, 
                       help='Confidence threshold')
    parser.add_argument('--iou-thres', type=float, default=0.45, 
                       help='IoU threshold for NMS')
    parser.add_argument('--device', type=str, default='', 
                       help='Device (cuda:0, cpu)')
    parser.add_argument('--save-txt', action='store_true', 
                       help='Save results as txt')
    parser.add_argument('--save-img', action='store_true', default=True,
                       help='Save annotated images')
    
    args = parser.parse_args()
    
    run_inference(
        source=args.source,
        weights=args.weights,
        depth_dir=args.depth_dir,
        illum_dir=args.illum_dir,
        output=args.output,
        imgsz=args.imgsz,
        conf_thres=args.conf_thres,
        iou_thres=args.iou_thres,
        device=args.device,
        save_txt=args.save_txt,
        save_img=args.save_img
    )


if __name__ == '__main__':
    main()



### export.py
Definition and code for `export.py`.


In [ ]:
%%writefile export.py
"""
Export Utilities for YOLOv5-3D
Export to ONNX, TensorRT, TFLite for edge deployment.
"""

import os
import sys
import argparse
import warnings
from pathlib import Path
from typing import Optional, Tuple

import torch
import torch.nn as nn
from torch.amp import autocast

# Add project root to path
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.yolo import YOLOv5_3D
from models.common import Conv
from utils.general import colorstr


def export_onnx(
    model: nn.Module,
    save_path: str,
    imgsz: int = 640,
    simplify: bool = True,
    dynamic: bool = False,
    opset: int = 12,
    device: torch.device = None
) -> str:
    """
    Export model to ONNX format.
    
    Args:
        model: Model to export
        save_path: Path to save ONNX model
        imgsz: Image size
        simplify: Simplify ONNX model
        dynamic: Use dynamic input shapes
        opset: ONNX opset version
        device: Device
    
    Returns:
        Path to exported model
    """
    device = device or torch.device('cpu')
    model = model.to(device)
    model.eval()
    
    # Create dummy inputs
    img = torch.zeros(1, 3, imgsz, imgsz).to(device)
    depth = torch.zeros(1, 1, imgsz, imgsz).to(device)
    illum = torch.zeros(1, 3, imgsz, imgsz).to(device)
    
    # Input names and dynamic axes
    input_names = ['images', 'depth', 'illumination']
    output_names = ['output']
    
    dynamic_axes = None
    if dynamic:
        dynamic_axes = {
            'images': {0: 'batch', 2: 'height', 3: 'width'},
            'depth': {0: 'batch', 2: 'height', 3: 'width'},
            'illumination': {0: 'batch', 2: 'height', 3: 'width'},
            'output': {0: 'batch', 1: 'num_detections'}
        }
    
    # Export
    print(f"{colorstr('blue')}Exporting to ONNX...{colorstr('reset')}")
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        torch.onnx.export(
            model,
            (img, depth, illum),
            save_path,
            verbose=False,
            opset_version=opset,
            input_names=input_names,
            output_names=output_names,
            dynamic_axes=dynamic_axes
        )
    
    # Simplify
    if simplify:
        try:
            import onnx
            from onnxsim import simplify as onnx_simplify
            
            print(f"{colorstr('yellow')}Simplifying ONNX model...{colorstr('reset')}")
            
            onnx_model = onnx.load(save_path)
            onnx_model, _ = onnx_simplify(onnx_model)
            onnx.save(onnx_model, save_path)
            
            print(f"{colorstr('green')}ONNX model simplified!{colorstr('reset')}")
        except ImportError:
            print(f"{colorstr('yellow')}onnx-simplifier not found, skipping simplification{colorstr('reset')}")
    
    print(f"{colorstr('green')}ONNX export complete: {save_path}{colorstr('reset')}")
    
    # Verify
    try:
        import onnx
        onnx_model = onnx.load(save_path)
        onnx.checker.check_model(onnx_model)
        print(f"{colorstr('green')}ONNX model verified!{colorstr('reset')}")
    except ImportError:
        pass
    
    return save_path


def export_torchscript(
    model: nn.Module,
    save_path: str,
    imgsz: int = 640,
    device: torch.device = None
) -> str:
    """
    Export model to TorchScript format.
    
    Args:
        model: Model to export
        save_path: Path to save TorchScript model
        imgsz: Image size
        device: Device
    
    Returns:
        Path to exported model
    """
    device = device or torch.device('cpu')
    model = model.to(device)
    model.eval()
    
    # Create dummy inputs
    img = torch.zeros(1, 3, imgsz, imgsz).to(device)
    depth = torch.zeros(1, 1, imgsz, imgsz).to(device)
    illum = torch.zeros(1, 3, imgsz, imgsz).to(device)
    
    print(f"{colorstr('blue')}Exporting to TorchScript...{colorstr('reset')}")
    
    # Trace
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        traced_model = torch.jit.trace(model, (img, depth, illum))
    
    # Save
    traced_model.save(save_path)
    
    print(f"{colorstr('green')}TorchScript export complete: {save_path}{colorstr('reset')}")
    
    return save_path


def export_coreml(
    model: nn.Module,
    save_path: str,
    imgsz: int = 640
) -> str:
    """
    Export model to CoreML format (for iOS).
    
    Args:
        model: Model to export
        save_path: Path to save CoreML model
        imgsz: Image size
    
    Returns:
        Path to exported model
    """
    try:
        import coremltools as ct
    except ImportError:
        print(f"{colorstr('red')}coremltools not installed. Run: pip install coremltools{colorstr('reset')}")
        return None
    
    device = torch.device('cpu')
    model = model.to(device)
    model.eval()
    
    # Create dummy inputs
    img = torch.zeros(1, 3, imgsz, imgsz).to(device)
    depth = torch.zeros(1, 1, imgsz, imgsz).to(device)
    illum = torch.zeros(1, 3, imgsz, imgsz).to(device)
    
    print(f"{colorstr('blue')}Exporting to CoreML...{colorstr('reset')}")
    
    # Trace
    traced_model = torch.jit.trace(model, (img, depth, illum))
    
    # Convert to CoreML
    mlmodel = ct.convert(
        traced_model,
        inputs=[
            ct.ImageType(name="image", shape=img.shape, scale=1/255.0),
            ct.ImageType(name="depth", shape=depth.shape, scale=1/255.0),
            ct.ImageType(name="illumination", shape=illum.shape, scale=1/255.0),
        ]
    )
    
    # Save
    mlmodel.save(save_path)
    
    print(f"{colorstr('green')}CoreML export complete: {save_path}{colorstr('reset')}")
    
    return save_path


def export_tensorrt(
    onnx_path: str,
    save_path: str,
    fp16: bool = True,
    int8: bool = False,
    workspace: int = 4
) -> str:
    """
    Convert ONNX model to TensorRT engine.
    
    Args:
        onnx_path: Path to ONNX model
        save_path: Path to save TensorRT engine
        fp16: Use FP16 precision
        int8: Use INT8 precision
        workspace: GPU workspace size in GB
    
    Returns:
        Path to exported model
    """
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit
    except ImportError:
        print(f"{colorstr('red')}TensorRT not installed.{colorstr('reset')}")
        return None
    
    print(f"{colorstr('blue')}Converting to TensorRT...{colorstr('reset')}")
    
    # Logger
    TRT_LOGGER = trt.Logger(trt.Logger.INFO)
    
    # Create builder
    builder = trt.Builder(TRT_LOGGER)
    builder.max_workspace_size = workspace << 30
    
    # Create network
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    
    # Parse ONNX
    parser = trt.OnnxParser(network, TRT_LOGGER)
    with open(onnx_path, 'rb') as f:
        parser.parse(f.read())
    
    # Config
    config = builder.create_builder_config()
    
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print(f"{colorstr('yellow')}Using FP16 precision{colorstr('reset')}")
    
    if int8:
        config.set_flag(trt.BuilderFlag.INT8)
        print(f"{colorstr('yellow')}Using INT8 precision{colorstr('reset')}")
    
    # Build engine
    engine = builder.build_engine(network, config)
    
    # Save
    with open(save_path, 'wb') as f:
        f.write(engine.serialize())
    
    print(f"{colorstr('green')}TensorRT export complete: {save_path}{colorstr('reset')}")
    
    return save_path


def export_tflite(
    model: nn.Module,
    save_path: str,
    imgsz: int = 640,
    quantize: bool = False
) -> str:
    """
    Export model to TFLite format (for edge devices).
    
    Args:
        model: Model to export
        save_path: Path to save TFLite model
        imgsz: Image size
        quantize: Apply INT8 quantization
    
    Returns:
        Path to exported model
    """
    try:
        import tensorflow as tf
    except ImportError:
        print(f"{colorstr('red')}TensorFlow not installed. Run: pip install tensorflow{colorstr('reset')}")
        return None
    
    import onnx
    from onnx_tf.backend import prepare
    
    # First export to ONNX
    onnx_path = save_path.replace('.tflite', '.onnx')
    export_onnx(model, onnx_path, imgsz)
    
    print(f"{colorstr('blue')}Converting to TFLite...{colorstr('reset')}")
    
    # Convert ONNX to TF
    onnx_model = onnx.load(onnx_path)
    tf_rep = prepare(onnx_model)
    
    # Save TF model
    tf_path = save_path.replace('.tflite', '_tf')
    tf_rep.export_graph(tf_path)
    
    # Convert to TFLite
    converter = tf.lite.TFLiteConverter.from_saved_model(tf_path)
    
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        print(f"{colorstr('yellow')}Applying INT8 quantization{colorstr('reset')}")
    
    tflite_model = converter.convert()
    
    # Save
    with open(save_path, 'wb') as f:
        f.write(tflite_model)
    
    print(f"{colorstr('green')}TFLite export complete: {save_path}{colorstr('reset')}")
    
    return save_path


def export_all(
    weights: str,
    output_dir: str,
    imgsz: int = 640,
    device: str = '',
    include: Tuple[str] = ('onnx', 'torchscript')
):
    """
    Export model to all specified formats.
    
    Args:
        weights: Path to model weights
        output_dir: Output directory
        imgsz: Image size
        device: Device
        include: Formats to export
    """
    device = torch.device(device if device else ('cuda' if torch.cuda.is_available() else 'cpu'))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Load model
    print(f"{colorstr('blue')}Loading model from {weights}...{colorstr('reset')}")
    
    ckpt = torch.load(weights, map_location=device)
    if 'model' in ckpt:
        state_dict = ckpt['model']
        if hasattr(state_dict, 'state_dict'):
            state_dict = state_dict.state_dict()
        nc = ckpt.get('nc', 6)
    else:
        state_dict = ckpt
        nc = 6
    
    model = YOLOv5_3D(nc=nc).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    
    base_name = Path(weights).stem
    
    # Export
    exports = {
        'onnx': lambda: export_onnx(model, output_dir / f'{base_name}.onnx', imgsz, device=device),
        'torchscript': lambda: export_torchscript(model, output_dir / f'{base_name}.torchscript', imgsz, device=device),
        'coreml': lambda: export_coreml(model, output_dir / f'{base_name}.mlmodel', imgsz),
        'tflite': lambda: export_tflite(model, output_dir / f'{base_name}.tflite', imgsz),
    }
    
    for fmt in include:
        if fmt in exports:
            try:
                exports[fmt]()
            except Exception as e:
                print(f"{colorstr('red')}Failed to export {fmt}: {e}{colorstr('reset')}")
        else:
            print(f"{colorstr('yellow')}Unknown format: {fmt}{colorstr('reset')}")
    
    print(f"\n{colorstr('green')}Export complete!{colorstr('reset')}")
    print(f"  Output directory: {output_dir}")


def main():
    parser = argparse.ArgumentParser(description='YOLOv5-3D Export')
    
    parser.add_argument('--weights', type=str, required=True, 
                       help='Model weights path')
    parser.add_argument('--output', type=str, default='runs/export', 
                       help='Output directory')
    parser.add_argument('--imgsz', type=int, default=640, 
                       help='Image size')
    parser.add_argument('--device', type=str, default='', 
                       help='Device')
    parser.add_argument('--include', type=str, nargs='+', 
                       default=['onnx', 'torchscript'],
                       choices=['onnx', 'torchscript', 'coreml', 'tflite', 'tensorrt'],
                       help='Export formats')
    
    args = parser.parse_args()
    
    export_all(
        weights=args.weights,
        output_dir=args.output,
        imgsz=args.imgsz,
        device=args.device,
        include=args.include
    )


if __name__ == '__main__':
    main()



### generate_pdf.py
Definition and code for `generate_pdf.py`.


In [ ]:
%%writefile generate_pdf.py
import sys
import os

try:
    import markdown
    import pdfkit
except ImportError:
    print("Required packages not found. Installing...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "markdown", "pdfkit"])
    import markdown
    import pdfkit

# File paths
md_file = r"C:\Users\HP\.gemini\antigravity\brain\fb0ddee7-da8a-42d2-998f-0a12a2e03afd\academic_research_paper.md"
html_file = r"C:\Users\HP\.gemini\antigravity\brain\fb0ddee7-da8a-42d2-998f-0a12a2e03afd\academic_research_paper.html"
pdf_file = r"C:\Users\HP\.gemini\antigravity\brain\fb0ddee7-da8a-42d2-998f-0a12a2e03afd\academic_research_paper.pdf"

print(f"Reading markdown from: {md_file}")
with open(md_file, "r", encoding="utf-8") as f:
    text = f.read()

# Convert Markdown to HTML
print("Converting to HTML...")
html = markdown.markdown(text, extensions=['extra', 'tables', 'fenced_code'])

# Wrap in basic HTML structure for better PDF styling
styled_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>
    body {{ font-family: Arial, sans-serif; line-height: 1.6; margin: 40px; color: #333; }}
    h1 {{ color: #2c3e50; border-bottom: 2px solid #eee; padding-bottom: 10px; }}
    h2 {{ color: #2980b9; margin-top: 30px; border-bottom: 1px solid #eee; padding-bottom: 5px; }}
    h3 {{ color: #16a085; }}
    code {{ background-color: #f8f9fa; padding: 2px 4px; border-radius: 4px; font-family: monospace; }}
    pre {{ background-color: #f8f9fa; padding: 15px; border-radius: 5px; overflow-x: auto; }}
    table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
    th {{ background-color: #f2f2f2; }}
    .mermaid {{ background-color: #f8f9fa; padding: 10px; border: 1px solid #ddd; }}
</style>
</head>
<body>
{html}
</body>
</html>
"""

with open(html_file, "w", encoding="utf-8") as f:
    f.write(styled_html)

# Convert HTML to PDF (Note: Requires wkhtmltopdf to be installed on the system)
print("Attempting PDF conversion...")
try:
    # Try basic conversion
    pdfkit.from_file(html_file, pdf_file)
    print(f"Success! PDF generated at: {pdf_file}")
except Exception as e:
    print(f"Error during PDF generation: {e}")
    print("\nNote: For perfect PDF generation, please ensure wkhtmltopdf is installed on your Windows machine.")
    print("You can download it from: https://wkhtmltopdf.org/downloads.html")
    print("\nAlternatively, you can open the generated HTML file in Chrome/Edge and select 'Print -> Save as PDF':")
    print(f"HTML File: {html_file}")



### kaggle_eval.py
Definition and code for `kaggle_eval.py`.


In [ ]:
%%writefile kaggle_eval.py
import os
import sys
import argparse
import ast
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Add project root to path so we can import models and utils
sys.path.append(os.getcwd())

from models.yolo import YOLOv5_3D
from utils.datasets import DataHandler
from utils.metrics import non_max_suppression, box_iou, compute_ap
from utils.general import select_device

def compute_map_per_class(stats, num_classes, class_names):
    """Computes per-class mAP given collected stats."""
    if len(stats) == 0:
        return {}

    correct = np.concatenate([s[0] for s in stats], axis=0) if stats[0][0].size > 0 else np.zeros((0, 10))
    confidences = np.concatenate([s[1] for s in stats]) if len(stats[0][1]) > 0 else np.array([])
    pred_classes = np.concatenate([s[2] for s in stats]) if len(stats[0][2]) > 0 else np.array([])
    target_classes = np.concatenate([np.array(s[3]) for s in stats if len(s[3]) > 0]) if any(len(s[3]) > 0 for s in stats) else np.array([])

    if len(confidences) == 0:
        return {}

    i = np.argsort(-confidences)
    confidences = confidences[i]
    pred_classes = pred_classes[i]
    correct = correct[i]

    unique_classes = np.unique(np.concatenate([target_classes, pred_classes])) if len(target_classes) > 0 else np.unique(pred_classes)
    
    class_metrics = {}
    
    for cls in unique_classes:
        cls_mask = pred_classes == cls
        n_pred = cls_mask.sum()
        n_gt = (target_classes == cls).sum()

        if n_gt == 0 or n_pred == 0:
            continue

        tp_50 = correct[cls_mask, 0].cumsum()
        fp_50 = (~correct[cls_mask, 0]).cumsum()
        
        recall_50 = tp_50 / (n_gt + 1e-16)
        precision_50 = tp_50 / (tp_50 + fp_50 + 1e-16)

        ap_per_thresh = []
        for j in range(correct.shape[1]):
            tp = correct[cls_mask, j].cumsum()
            fp = (~correct[cls_mask, j]).cumsum()
            recall = tp / (n_gt + 1e-16)
            precision = tp / (tp + fp + 1e-16)
            ap_per_thresh.append(compute_ap(recall, precision))
            
        ap_per_thresh = np.array(ap_per_thresh)
        cls_name = class_names[int(cls)] if int(cls) < len(class_names) else f"Class_{int(cls)}"
        
        class_metrics[cls_name] = {
            'mAP@0.5': ap_per_thresh[0],
            'mAP@0.5:0.95': ap_per_thresh.mean(),
            'precision': precision_50[-1] if len(precision_50) else 0.0,
            'recall': recall_50[-1] if len(recall_50) else 0.0
        }
        
    return class_metrics

def plot_confusion_matrix(cm, class_names, save_dir):
    """Plots and saves the confusion matrix using Seaborn."""
    plt.figure(figsize=(12, 10))
    # Normalize by row (true class)
    cm_norm = cm / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)
    # Background class is the last row/col for YOLO
    names = class_names + ['background'] 
    
    sns.heatmap(cm_norm, annot=cm, fmt='.0f', cmap='Blues', 
                xticklabels=names, yticklabels=names,
                cbar_kws={'label': 'Normalized True Class'})
    plt.title('Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig(save_dir / 'confusion_matrix.png', dpi=300)
    plt.close()

def plot_pr_curve(px, py, ap, save_dir, class_names):
    """Plot Precision-Recall curve."""
    fig, ax = plt.subplots(1, 1, figsize=(9, 6), tight_layout=True)
    py = np.stack(py, axis=1)

    for i, y in enumerate(py.T):
        ax.plot(px, y, linewidth=1, label=f'{class_names[i]} {ap[i, 0]:.3f}')

    ax.plot(px, py.mean(1), linewidth=3, color='blue', label=f'all classes {ap[:, 0].mean():.3f} mAP@0.5')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left')
    fig.savefig(save_dir / 'PR_curve.png', dpi=250)
    plt.close()

def plot_mc_curve(px, py, save_dir, class_names, xlabel='Confidence', ylabel='Metric'):
    """Plot Metric-Confidence curve."""
    fig, ax = plt.subplots(1, 1, figsize=(9, 6), tight_layout=True)
    py = np.stack(py, axis=1)

    for i, y in enumerate(py.T):
        ax.plot(px, y, linewidth=1, label=f'{class_names[i]}')

    ax.plot(px, py.mean(1), linewidth=3, color='blue', label=f'all classes {py.mean(1).max():.2f} at {px[py.mean(1).argmax()]:.3f}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.04, 1), loc='upper left')
    fig.savefig(save_dir / f'{ylabel}_curve.png', dpi=250)
    plt.close()

def run_evaluation(data_dir, weights, imgsz=640, batch_size=8, conf_thres=0.001, augment=False):
    iou_thres = 0.6
    print(f"Loading weights from {weights}...")
    device = select_device('')
    
    # Load Model Checkpoint
    ckpt = torch.load(weights, map_location=device, weights_only=False)
    state_dict = ckpt['model'] if 'model' in ckpt else ckpt
    
    print(f"Loading validation dataset from {data_dir}...")
    data_handler = DataHandler(data_dir=data_dir, batch_size=batch_size, img_size=imgsz)
    val_loader = data_handler.val_loader
    class_names = data_handler.classes
    num_classes = len(class_names)
    
    # We must instantiate based on checkpoint keys or assume YOLOv5_3D defaults
    model = YOLOv5_3D(nc=num_classes, freeze_backbone=False).to(device)
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    
    # Trackers
    stats = []
    iouv = np.linspace(0.5, 0.95, 10)
    niou = iouv.shape[0]
    
    # Confusion Matrix (N+1 x N+1 to include background)
    cm = np.zeros((num_classes + 1, num_classes + 1))
    
    print("Evaluating predictions...")
    with torch.no_grad():
        for batch in tqdm(val_loader):
            images = batch['images'].to(device).float()
            depths = batch['depths'].to(device).float()
            illuminations = batch['illuminations'].to(device).float()
            targets = batch['targets'].to(device)
            
            nb, _, img_h, img_w = images.shape
            
            if augment:
                all_preds = []
                
                # 1. Base Image Forward Pass
                base_out = model(images, depths, illuminations)
                all_preds.append(base_out[0] if isinstance(base_out, tuple) else base_out)
                
                # 2. Custom SAHI (Slicing Aided Hyper Inference) - Five Overlapping 640x640 Crops
                # Forces the model to see microscopic 3D defects at native training resolution
                crops = [
                    (0, 0, 640, 640),
                    (img_w - 640, 0, img_w, 640),
                    (0, img_h - 640, 640, img_h),
                    (img_w - 640, img_h - 640, img_w, img_h),
                    (img_w // 2 - 320, img_h // 2 - 320, img_w // 2 + 320, img_h // 2 + 320)
                ]
                
                for x1, y1, x2, y2 in crops:
                    img_c = images[:, :, y1:y2, x1:x2]
                    dep_c = depths[:, :, y1:y2, x1:x2]
                    ill_c = illuminations[:, :, y1:y2, x1:x2]
                    
                    c_out = model(img_c, dep_c, ill_c)
                    c_out = c_out[0] if isinstance(c_out, tuple) else c_out
                    
                    # Shift predicted center coordinates (x, y) back to absolute 1024x1024 scale
                    c_out[..., 0] += x1
                    c_out[..., 1] += y1
                    all_preds.append(c_out)
                
                # 3. Standard Horizontal Flip TTA (on the full image)
                img_f = torch.flip(images, [3])
                dep_f = torch.flip(depths, [3])
                ill_f = torch.flip(illuminations, [3])
                f_out = model(img_f, dep_f, ill_f)
                f_out = f_out[0] if isinstance(f_out, tuple) else f_out
                f_out[..., 0] = img_w - f_out[..., 0]
                all_preds.append(f_out)
                
                # Merge all 7 forward passes for unified NMS
                pred_tensor = torch.cat(all_preds, 1).cpu().numpy()
            else:
                # Standard Forward pass
                outputs = model(images, depths, illuminations)
                inference_out = outputs[0] if isinstance(outputs, tuple) else outputs
                pred_tensor = inference_out.cpu().numpy()
                
            for si in range(nb):
                pred = pred_tensor[si]
                mask = targets[:, 0] == si
                t = targets[mask].cpu().numpy()
                nl = len(t)
                tcls = t[:, 1].astype(int).tolist() if nl else []
                
                # NMS
                from train import apply_nms # Reuse the optimized NMS from train.py
                det = apply_nms(pred, conf_thres, iou_thres, img_w, img_h)
                
                if len(det) == 0:
                    if nl:
                        stats.append((np.zeros((0, niou), dtype=bool), np.array([]), np.array([]), tcls))
                        for tc in tcls:
                            cm[tc, num_classes] += 1 # False negative (predicted background)
                    continue
                
                det_box = det[:, :4]
                det_conf = det[:, 4]
                det_cls = det[:, 5].astype(int)
                
                if nl:
                    labels = t.copy()
                    labels_xyxy = np.zeros((nl, 5))
                    labels_xyxy[:, 0] = labels[:, 1]
                    labels_xyxy[:, 1] = (labels[:, 2] - labels[:, 4] / 2) * img_w
                    labels_xyxy[:, 2] = (labels[:, 3] - labels[:, 5] / 2) * img_h
                    labels_xyxy[:, 3] = (labels[:, 2] + labels[:, 4] / 2) * img_w
                    labels_xyxy[:, 4] = (labels[:, 3] + labels[:, 5] / 2) * img_h
                    
                    iou_mat = box_iou(det_box, labels_xyxy[:, 1:])
                    
                    correct = np.zeros((len(det), niou), dtype=bool)
                    matched_gt = []
                    
                    for j in range(len(det)):
                        max_iou = 0
                        best_k = -1
                        for k in range(nl):
                            if det_cls[j] == int(labels_xyxy[k, 0]):
                                if iou_mat[j, k] > max_iou and k not in matched_gt:
                                    max_iou = iou_mat[j, k]
                                    best_k = k
                        
                        correct[j] = max_iou >= iouv
                        
                        # Populate Confusion Matrix based on IoU > 0.5
                        if max_iou >= 0.5:
                            cm[int(labels_xyxy[best_k, 0]), det_cls[j]] += 1
                            matched_gt.append(best_k)
                        else:
                            cm[num_classes, det_cls[j]] += 1 # False positive (true background)
                    
                    # Any GT not matched is a False Negative
                    for k in range(nl):
                        if k not in matched_gt:
                            cm[int(labels_xyxy[k, 0]), num_classes] += 1
                            
                    stats.append((correct, det_conf, det_cls, tcls))
                else:
                    stats.append((np.zeros((len(det), niou), dtype=bool), det_conf, det_cls, tcls))
                    for dc in det_cls:
                        cm[num_classes, dc] += 1 # FP

    # Create output directory relative to the weights file
    out_dir = Path(weights).parent / "evaluation_graphs"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print("\n--- Generating Metrics ---")
    class_metrics = compute_map_per_class(stats, num_classes, class_names)
    
    from utils.metrics import ap_per_class
    # Compute the raw curves from the collected stats
    # stats contains (correct, conf, pred_cls, target_cls)
    stats_concat = [np.concatenate(x, 0) for x in zip(*stats)]
    p, r, f1, ap, unique_classes = ap_per_class(*stats_concat)
    
    # Save Metrics to CSV
    df = pd.DataFrame.from_dict(class_metrics, orient='index')
    df.loc['MEAN'] = df.mean()
    df.to_csv(out_dir / 'per_class_metrics.csv')
    print(df.to_string())
    
    print("\n--- Generating Graphs ---")
    # 1. Confusion Matrix
    plot_confusion_matrix(cm, class_names, out_dir)
    print(f"Confusion matrix saved to {out_dir / 'confusion_matrix.png'}")
    
    # 2. Per-Class mAP Bar Chart
    plt.figure(figsize=(12, 6))
    df_no_mean = df.drop('MEAN')
    df_no_mean[['mAP@0.5', 'mAP@0.5:0.95']].plot(kind='bar', ax=plt.gca(), colormap='viridis')
    plt.title('mAP by Defect Class')
    plt.ylabel('mAP Score')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(out_dir / 'map_per_class.png', dpi=300)
    plt.close()
    print(f"mAP Bar Chart saved to {out_dir / 'map_per_class.png'}")
    
    # 3. Ultralytics Curves
    # ap_per_class generates curves across 1000 points from 0 to 1
    px = np.linspace(0, 1, 1000)
    plot_pr_curve(px, p, ap, out_dir, class_names)
    plot_mc_curve(px, f1, out_dir, class_names, ylabel='F1')
    plot_mc_curve(px, p, out_dir, class_names, ylabel='Precision')
    plot_mc_curve(px, r, out_dir, class_names, ylabel='Recall')
    print("Standard PR, F1, P, and R curves generated successfully.")

    print(f"\nEvaluation Complete! All graphs saved to {out_dir}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data-dir', type=str, required=True, help='Dataset path')
    parser.add_argument('--weights', type=str, required=True, help='Weights path')
    parser.add_argument('--imgsz', type=int, default=640)
    parser.add_argument('--conf_thres', type=float, default=0.001, help='Confidence threshold')
    parser.add_argument('--augment', action='store_true', help='Use Test-Time Augmentation (TTA)')
    opt = parser.parse_args()
    
    run_evaluation(data_dir=opt.data_dir, weights=opt.weights, imgsz=opt.imgsz, conf_thres=opt.conf_thres, augment=opt.augment)



### plot_training_curves.py
Definition and code for `plot_training_curves.py`.


In [ ]:
%%writefile plot_training_curves.py
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_theme(style="whitegrid")

# Define paths
csv_path = Path(r"D:\model3dyolo\yolov5modfication\runs\train\kaggle_yolo3d_final\results.csv")
out_dir = csv_path.parent / "evaluation_graphs"
out_dir.mkdir(parents=True, exist_ok=True)

# Read the CSV
print(f"Reading data from {csv_path}...")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

print("Columns found:", df.columns.tolist())

# Helper function to save individual plots
def save_plot(filename):
    save_path = out_dir / filename
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    plt.close()

# 1. Total Loss (Train vs Val)
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'], label='Train Total Loss', color='blue', linewidth=2)
plt.plot(df['epoch'], df['val_loss'], label='Val Total Loss', color='orange', linewidth=2)
plt.title('Overall Loss Progression', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=12)
plt.ylim(0, df['val_loss'][20:].max() * 1.2) # Ignore first 20 epochs for scale
save_plot('chart_1_overall_loss.png')

# 2. Component Losses (Box, Obj, Cls)
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['box_loss'], label='Box (Localization)', color='red', linewidth=2)
plt.plot(df['epoch'], df['obj_loss'], label='Obj (Confidence)', color='green', linewidth=2)
plt.plot(df['epoch'], df['cls_loss'], label='Cls (Classification)', color='purple', linewidth=2)
plt.title('Training Loss Components', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss Component', fontsize=12)
plt.legend(fontsize=12)
save_plot('chart_2_loss_components.png')

# 3. Precision vs Recall
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['precision'], label='Precision', color='teal', linewidth=2)
plt.plot(df['epoch'], df['recall'], label='Recall', color='magenta', linewidth=2)
plt.title('Precision and Recall', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.legend(fontsize=12)
plt.ylim(0, 1.0)
save_plot('chart_3_precision_recall.png')

# 4. Mean Average Precision (mAP)
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['mAP50'], label='mAP@0.5', color='blue', linewidth=2)
plt.plot(df['epoch'], df['mAP50_95'], label='mAP@0.5:0.95', color='crimson', linewidth=2)
plt.title('Mean Average Precision (mAP)', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('mAP Score', fontsize=12)
plt.legend(fontsize=12)
save_plot('chart_4_map.png')

# 5. F1 Score
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['f1'], label='F1-Score', color='darkgreen', linewidth=2)
plt.title('F1 Score Progression', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.legend(fontsize=12)
plt.ylim(0, 1.0)
save_plot('chart_5_f1_score.png')

# 6. Learning Rate Schedule
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['lr'], label='Learning Rate', color='grey', linewidth=2, linestyle='--')
plt.title('Learning Rate Decay (Cosine Annealing)', fontsize=16)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.yscale('log')
plt.legend(fontsize=12)
save_plot('chart_6_learning_rate.png')

print("\nAll individual charts have been generated successfully!")



### train.py
Definition and code for `train.py`.


In [ ]:
%%writefile train.py
"""
YOLOv5-3D Training Script
Main training loop with mixed precision, gradient accumulation, and all metrics.
"""

import os
import sys
import argparse
import math
import time
import datetime
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast
from torch.utils.tensorboard import SummaryWriter
import numpy as np
from tqdm import tqdm

# Add project root to path
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.yolo import YOLOv5_3D, attempt_load, fuse_model
from utils.datasets import DataHandler, PCBDataset, create_dataloaders
from utils.loss import ComputeLoss, ComputeLossAux
from utils.metrics import MetricTracker, ConfusionMatrix, non_max_suppression, box_iou, xywh2xyxy
from utils.plots import TrainingVisualizer
from utils.general import (
    increment_path, colorstr, check_dataset, 
    one_cycle, initialize_weights, set_logging, get_latest_run,
    select_device
)


def train(
    data_dir: str,
    weights: str = '',
    epochs: int = 100,
    batch_size: int = 8,
    imgsz: int = 640,
    lr0: float = 0.01,
    lrf: float = 0.2,
    momentum: float = 0.937,
    weight_decay: float = 0.0005,
    warmup_epochs: int = 1,
    warmup_momentum: float = 0.8,
    warmup_bias_lr: float = 0.1,
    optimizer: str = 'AdamW',
    device: str = '',
    workers: int = 4,
    project: str = 'runs/train',
    name: str = 'exp',
    exist_ok: bool = False,
    pretrained: bool = False,
    freeze_backbone: bool = True,
    freeze_illum_fraction: float = 0.33,
    freeze_neck: bool = False,
    amp: bool = True,
    gradient_accumulation_steps: int = 1,
    save_period: int = -1,
    patience: int = 50,
    rect: bool = False,
    cos_lr: bool = True,
    conf_thres: float = 0.001,  # COCO standard for mAP eval
    iou_thres: float = 0.7,
    verbose: bool = True,
    **kwargs
):
    """
    Train YOLOv5-3D model.
    """
    # Setup
    set_logging()
    device = select_device(device)
    
    # Don't freeze backbone when training from scratch (no pretrained weights)
    if not weights and not pretrained:
        freeze_backbone = False
    
    # Save directory
    save_dir = increment_path(Path(project) / name, exist_ok=exist_ok)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Hyperparameters
    hyp = {
        'lr0': 0.001,          # Lowered from 0.01 for fine-tuning pre-trained weights
        'lrf': 0.1,            # 10% of lr0 for final LR
        'momentum': 0.937,
        'weight_decay': 0.0005,
        'warmup_epochs': 3.0,  # Extended from 1.0 to protect pre-trained weights
        'warmup_momentum': 0.8,
        'warmup_bias_lr': 0.1,
        'box': 0.05,
        'cls': 0.5,
        'obj': 2.5,            # Increased massive penalty for inventing boxes (False Positives)
        'cls_pw': 1.0,
        'obj_pw': 0.5,         # Reduced from 2.0 to stop pushing model to guess
        'fl_gamma': 2.0,       # Maximize focal loss to suppress easy background pixels
        'anchor_t': 4.0,
        'label_smoothing': 0.0,
    }
    
    # Log settings
    print_args(vars())
    with open(save_dir / 'hyp.yaml', 'w') as f:
        import yaml
        yaml.dump(hyp, f, sort_keys=False)
    
    # Tensorboard
    writer = SummaryWriter(str(save_dir / 'tensorboard'))
    
    # ============ DATA ============
    print(f"\n{colorstr('blue')}Setting up data...{colorstr('reset')}")
    
    data_handler = DataHandler(
        data_dir=data_dir,
        batch_size=batch_size,
        img_size=imgsz,
        num_workers=workers
    )
    
    train_loader = data_handler.train_loader
    val_loader = data_handler.val_loader
    num_classes = data_handler.num_classes
    class_names = data_handler.classes
    
    # Save data config
    with open(save_dir / 'data.yaml', 'w') as f:
        import yaml
        yaml.dump({
            'path': data_dir,
            'nc': num_classes,
            'names': class_names
        }, f)
    
    # ============ MODEL ============
    print(f"\n{colorstr('blue')}Building model...{colorstr('reset')}")
    
    model = YOLOv5_3D(
        nc=num_classes,
        input_size=imgsz,
        pretrained=pretrained or weights,
        freeze_backbone=freeze_backbone,
        use_illum_attention=True
    ).to(device)
    
    # Load specific weights if provided
    if weights and not pretrained:
        surgical = kwargs.get('surgical_transfer', False)
        load_pretrained_weights(model, weights, device, surgical_transfer=surgical)
    
    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\n{colorstr('green')}Model Info:{colorstr('reset')}")
    print(f"  Total parameters: {n_params:,}")
    print(f"  Trainable parameters: {n_trainable:,}")
    print(f"  Frozen parameters: {n_params - n_trainable:,}")

    # ===== DEBUG: Print anchor sizes =====
    print(f"\n{colorstr('yellow')}Anchor sizes per layer:{colorstr('reset')}")
    m = model.model[-1] if hasattr(model, 'model') else model.detect
    for i, anchors in enumerate(m.anchors):
        print(f"  Layer {i}: {anchors}")
    # ===== END DEBUG =====
    
    # ============ OPTIMIZER ============
    print(f"\n{colorstr('blue')}Setting up optimizer...{colorstr('reset')}")
    
    # Separate parameters for different LR
    pg0, pg1, pg2 = [], [], []  # optimizer parameter groups
    
    for k, v in model.named_modules():
        if hasattr(v, 'bias') and isinstance(v.bias, nn.Parameter):
            pg2.append(v.bias)  # biases
        if isinstance(v, nn.BatchNorm2d):
            pg0.append(v.weight)  # no decay
        elif hasattr(v, 'weight') and isinstance(v.weight, nn.Parameter):
            pg1.append(v.weight)  # apply decay
    
    if optimizer.lower() == 'adamw':
        optim_class = optim.AdamW
        optimizer = optim_class(pg0, lr=hyp['lr0'], weight_decay=0.0, betas=(hyp['momentum'], 0.999))
    else:
        optim_class = optim.SGD
        optimizer = optim_class(pg0, lr=hyp['lr0'], momentum=hyp['momentum'], weight_decay=0.0)
    
    optimizer.add_param_group({'params': pg1, 'weight_decay': hyp['weight_decay']})
    optimizer.add_param_group({'params': pg2, 'weight_decay': 0.0})
    
    print(f"  Optimizer: {optimizer.__class__.__name__}")
    print(f"  Parameter groups: {len(pg0)} BN, {len(pg1)} weight, {len(pg2)} bias")
    
    # Learning rate scheduler
    lf = one_cycle(1, hyp['lrf'], epochs) if cos_lr else lambda x: (1 - x / epochs) * (1.0 - hyp['lrf']) + hyp['lrf']
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
    
    # ============ CLASS WEIGHTS ============
    from utils.metrics import compute_class_weights
    print(f"\n{colorstr('blue')}Computing class weights...{colorstr('reset')}")
    class_weights = compute_class_weights(data_handler.train_loader.dataset).to(device) * num_classes
    print(f"  Class weights: {class_weights.cpu().numpy().round(3)}")
    
    # ============ LOSS ============
    compute_loss = ComputeLoss(model, hyp, class_weights=class_weights)
    
    # ============ AMP ============
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    
    # ============ VISUALIZATION ============
    visualizer = TrainingVisualizer(save_dir, class_names)
    
    # ============ TRAINING ============
    print(f"\n{colorstr('green')}Starting training...{colorstr('reset')}")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Image size: {imgsz}")
    print(f"  Device: {device}")
    print(f"  AMP: {amp}")
    print(f"  Save dir: {save_dir}")
    print()
    
    start_time = time.time()
    best_fitness = 0.0
    epochs_without_improvement = 0  # Track for early stopping
    
    nb = len(train_loader)
    nw = max(round(hyp['warmup_epochs'] * nb), 100)  # warmup iterations
    
    # The ID Encoder (Depth/Illumination) is now required immediately from Epoch 1
    # to feed queries to the Phase 7 Cross-Attention Transformer. 
    # The legacy Phase 3 freeze protocol has been permanently disabled.
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        
        epoch_start = time.time()
        
        # Metrics
        mloss = torch.zeros(3, device=device)  # box, obj, cls
        pbar = tqdm(enumerate(train_loader), total=nb, 
                    desc=f'{epoch + 1}/{epochs}', bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}')
        
        optimizer.zero_grad()
        
        for i, batch in pbar:
            # Warmup
            ni = i + nb * epoch
            if ni <= nw:
                xi = [0, nw]  # x interp
                for j, x in enumerate(optimizer.param_groups):
                    # Bias lr falls from 0.1 to lr0, all other lrs rise from 0.0 to lr0
                    x['lr'] = np.interp(
                        ni, xi, 
                        [hyp['warmup_bias_lr'] if j == 2 else 0.0, x['initial_lr'] * lf(epoch)]
                    )
                    if 'momentum' in x:
                        x['momentum'] = np.interp(ni, xi, [hyp['warmup_momentum'], hyp['momentum']])
            
            # Forward pass
            images = batch['images'].to(device, non_blocking=True).float()
            depths = batch['depths'].to(device, non_blocking=True).float()
            illuminations = batch['illuminations'].to(device, non_blocking=True).float()
            targets = batch['targets'].to(device)
            
            with autocast('cuda', enabled=amp):
                outputs = model(images, depths, illuminations)
                loss, loss_items = compute_loss(outputs, targets)
            
            # Skip bad batches that produce NaN/Inf loss
            if not torch.isfinite(loss):
                if verbose and i < 5:  # Only warn for first few
                    print(f"\n  [WARN] Skipping batch {i}: NaN/Inf loss")
                optimizer.zero_grad()
                continue
            
            # Scale loss for gradient accumulation
            loss = loss / gradient_accumulation_steps
            
            # Backward
            scaler.scale(loss).backward()
            
            # Optimize
            if ni % gradient_accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            # Update metrics (use nan_to_num for safety)
            loss_items = torch.nan_to_num(loss_items, nan=0.0)
            mloss = (mloss * i + loss_items) / (i + 1)
            
            mem = torch.cuda.memory_reserved() / 1E9 if torch.cuda.is_available() else 0
            pbar.set_postfix({
                'loss': f'{mloss.sum():.4f}',
                'box': f'{mloss[0]:.4f}',
                'obj': f'{mloss[1]:.4f}',
                'cls': f'{mloss[2]:.4f}',
                'mem': f'{mem:.1f}GB'
            })
        
        # Scheduler step
        scheduler.step()
        
        # ============ VALIDATION ============
        metrics = validate(
            model, val_loader, compute_loss, device, 
            num_classes, conf_thres, iou_thres, amp
        )
        
        # Update visualizer
        visualizer.update({
            'train_loss': mloss.sum().item(),
            'val_loss': metrics['loss'],
            'mAP50': metrics['mAP50'],
            'mAP50_95': metrics['mAP50-95'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'box_loss': mloss[0].item(),
            'obj_loss': mloss[1].item(),
            'cls_loss': mloss[2].item(),
        }, lr=optimizer.param_groups[0]['lr'])
        
        # Tensorboard
        writer.add_scalar('Loss/train', mloss.sum().item(), epoch)
        writer.add_scalar('Loss/val', metrics['loss'], epoch)
        writer.add_scalar('mAP50', metrics['mAP50'], epoch)
        writer.add_scalar('mAP50-95', metrics['mAP50-95'], epoch)
        writer.add_scalar('LR', optimizer.param_groups[0]['lr'], epoch)
        
        # Print epoch summary
        epoch_time = time.time() - epoch_start
        print(f"  Epoch {epoch + 1}/{epochs} | "
              f"Train Loss: {mloss.sum().item():.4f} | "
              f"Val Loss: {metrics['loss']:.4f} | "
              f"mAP@0.5: {metrics['mAP50']:.4f} | "
              f"mAP@0.5:0.95: {metrics['mAP50-95']:.4f} | "
              f"Time: {epoch_time:.1f}s")
        
        # Save checkpoint
        fitness = metrics['mAP50-95']  # Use mAP@0.5:0.95 as fitness
        is_best = fitness > best_fitness
        best_fitness = max(fitness, best_fitness)
        
        # Save periodic checkpoint
        if (save_period > 0 and epoch % save_period == 0) or epoch == epochs - 1:
            save_checkpoint({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_fitness': best_fitness,
                'hyp': hyp,
            }, save_dir / f'epoch_{epoch}.pt')
        
        # Save best
        if is_best:
            save_checkpoint({
                'epoch': epoch,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'best_fitness': best_fitness,
                'hyp': hyp,
            }, save_dir / 'best.pt')
            epochs_without_improvement = 0
            
        else:
            epochs_without_improvement += 1
            if patience > 0 and epochs_without_improvement >= patience:
                print(f"\n{colorstr('red')}Early Stopping triggered.{colorstr('reset')}")
                print(f"  Validation mAP did not improve for {patience} consecutive epochs.")
                break
    
    # Save final
    save_checkpoint({
        'epoch': epochs,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_fitness': best_fitness,
        'hyp': hyp,
    }, save_dir / 'last.pt')
    
    # Final visualization
    visualizer.save_final()
    
    # Training summary
    total_time = time.time() - start_time
    print(f"\n{colorstr('green')}Training complete!{colorstr('reset')}")
    print(f"  Total time: {total_time / 3600:.2f} hours")
    print(f"  Best mAP@0.5:0.95: {best_fitness:.4f}")
    print(f"  Results saved to: {save_dir}")
    
    writer.close()
    
    return model, save_dir


def validate(
    
    model: nn.Module,
    dataloader,
    compute_loss,
    device,
    num_classes: int,
    conf_thres: float = 0.001,
    iou_thres: float = 0.7,
    amp: bool = True
) -> Dict[str, float]:
    """
    Validate model.
    
    Returns:
        Dictionary with metrics
    """
    model.eval()
    # Add at START of validate function:
    print("\n" + "="*50)
    print("VALIDATION DEBUG")
    print("="*50)
    
    # Initialize stats collection
    stats = []
    total_loss = 0
    n_batches = 0
    
    # IoU thresholds for mAP@0.5:0.95
    iouv = np.linspace(0.5, 0.95, 10)
    niou = iouv.shape[0]
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['images'].to(device).float()
            depths = batch['depths'].to(device).float()
            illuminations = batch['illuminations'].to(device).float()
            targets = batch['targets'].to(device)
            
            nb, _, img_h, img_w = images.shape
            
            # Forward
            with autocast('cuda', enabled=amp):
                outputs = model(images, depths, illuminations)
            
            # ===== Handle model output format =====
            if isinstance(outputs, tuple):
                inference_out, training_out = outputs
                loss, _ = compute_loss(training_out, targets)
                pred_tensor = inference_out
            else:
                loss, _ = compute_loss(outputs, targets)
                pred_tensor = torch.cat([o.view(o.shape[0], -1, o.shape[-1]) for o in outputs], dim=1)
            
            # Safe loss accumulation
            loss_val = loss.item()
            if not (math.isnan(loss_val) or math.isinf(loss_val)):
                total_loss += loss_val
                n_batches += 1
            
            pred_np = pred_tensor.cpu().numpy()
            
            # Process each image in batch
            for si in range(nb):
                pred = pred_np[si]
                mask = targets[:, 0] == si
                t = targets[mask].cpu().numpy()
                
                det = apply_nms(pred, conf_thres, iou_thres, img_w, img_h)
                
                nl = len(t)
                tcls = t[:, 1].astype(int).tolist() if nl else []
                
                if len(det) == 0:
                    if nl:
                        stats.append((np.zeros((0, niou), dtype=bool), np.array([]), np.array([]), tcls))
                    continue
                
                # Predictions
                det_box = det[:, :4]
                det_conf = det[:, 4]
                det_cls = det[:, 5].astype(int)
                
                if nl:
                    labels = t.copy()
                    labels_xyxy = np.zeros((nl, 5))
                    labels_xyxy[:, 0] = labels[:, 1]
                    labels_xyxy[:, 1] = (labels[:, 2] - labels[:, 4] / 2) * img_w
                    labels_xyxy[:, 2] = (labels[:, 3] - labels[:, 5] / 2) * img_h
                    labels_xyxy[:, 3] = (labels[:, 2] + labels[:, 4] / 2) * img_w
                    labels_xyxy[:, 4] = (labels[:, 3] + labels[:, 5] / 2) * img_h
                    
                    iou_mat = box_iou(det_box, labels_xyxy[:, 1:])
                    
                    correct = np.zeros((len(det), niou), dtype=bool)
                    matched_gt = np.zeros((nl, niou), dtype=bool)
                    for j in range(len(det)):
                        max_iou = 0
                        best_k = -1
                        for k in range(nl):
                            if det_cls[j] == int(labels_xyxy[k, 0]):
                                if iou_mat[j, k] > max_iou:
                                    max_iou = iou_mat[j, k]
                                    best_k = k
                        
                        if best_k >= 0:
                            # For each IoU threshold, if max_iou >= threshold and it hasn't been matched yet
                            for th_idx, th in enumerate(iouv):
                                if max_iou >= th and not matched_gt[best_k, th_idx]:
                                    correct[j, th_idx] = True
                                    matched_gt[best_k, th_idx] = True
                    
                    stats.append((correct, det_conf, det_cls, tcls))
                else:
                    stats.append((np.zeros((len(det), niou), dtype=bool), det_conf, det_cls, tcls))
    
    # Compute mAP
    metrics = compute_map(stats, num_classes)
    metrics['loss'] = total_loss / max(n_batches, 1)
    
    model.train()
    return metrics


def apply_nms(
    pred: np.ndarray,
    conf_thres: float,
    iou_thres: float,
    img_w: int,
    img_h: int,
    max_det: int = 300
) -> np.ndarray:
    """
    Apply NMS to predictions (fast version using torchvision).
    
    Args:
        pred: (num_predictions, 5+nc) - x, y, w, h, obj, cls...
        conf_thres: Confidence threshold
        iou_thres: IoU threshold for NMS
        img_w, img_h: Image dimensions for scaling
        max_det: Maximum detections to keep
    
    Returns:
        Detections (n, 6) - x1, y1, x2, y2, conf, cls
    """
    if pred.shape[0] == 0:
        return np.zeros((0, 6))
    
    # Extract boxes and scores
    boxes = pred[:, :4].copy()  # xywh
    obj_conf = pred[:, 4]
    cls_conf = pred[:, 5:] if pred.shape[1] > 5 else np.ones((len(pred), 1))
    
    # Get class with highest confidence
    cls_scores = obj_conf[:, None] * cls_conf
    confidences = np.max(cls_scores, axis=1)
    class_ids = np.argmax(cls_scores, axis=1)
    
    # Filter by confidence
    mask = confidences > conf_thres
    if not mask.any():
        return np.zeros((0, 6))
    
    boxes = boxes[mask]
    confidences = confidences[mask]
    class_ids = class_ids[mask]
    
    # Cap candidates to top-scoring to avoid slow NMS
    if len(confidences) > max_det * 10:
        top_idx = np.argpartition(confidences, -max_det * 10)[-max_det * 10:]
        boxes = boxes[top_idx]
        confidences = confidences[top_idx]
        class_ids = class_ids[top_idx]
    
    # Convert xywh to xyxy
    boxes_xyxy = np.zeros_like(boxes)
    boxes_xyxy[:, 0] = boxes[:, 0] - boxes[:, 2] / 2  # x1
    boxes_xyxy[:, 1] = boxes[:, 1] - boxes[:, 3] / 2  # y1
    boxes_xyxy[:, 2] = boxes[:, 0] + boxes[:, 2] / 2  # x2
    boxes_xyxy[:, 3] = boxes[:, 1] + boxes[:, 3] / 2  # y2
    
    # Use torchvision NMS (much faster, GPU-capable)
    try:
        import torchvision
        boxes_t = torch.from_numpy(boxes_xyxy).float()
        scores_t = torch.from_numpy(confidences).float()
        # Class-aware NMS: offset boxes by class to prevent inter-class suppression
        class_offset = torch.from_numpy(class_ids).float() * 4096
        boxes_offset = boxes_t + class_offset[:, None]
        keep = torchvision.ops.nms(boxes_offset, scores_t, iou_thres)
        keep = keep[:max_det]  # Limit detections
        
        detections = np.column_stack([
            boxes_xyxy[keep.numpy()],
            confidences[keep.numpy()],
            class_ids[keep.numpy()]
        ])
        return detections
    except Exception:
        # Fallback: simple top-k without full NMS
        order = np.argsort(-confidences)[:max_det]
        return np.column_stack([
            boxes_xyxy[order],
            confidences[order],
            class_ids[order]
        ])


def compute_map(stats: list, num_classes: int) -> Dict[str, float]:
    """
    Compute mAP from collected stats.
    
    Args:
        stats: List of (correct, conf, pred_cls, target_cls) tuples.
               correct is (N, 10) boolean array for IoU thresholds 0.5:0.05:0.95
    """
    if len(stats) == 0:
        return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0, 'f1': 0}
    
    # Concatenate all stats
    correct = np.concatenate([s[0] for s in stats], axis=0) if stats[0][0].size > 0 else np.zeros((0, 10))
    confidences = np.concatenate([s[1] for s in stats]) if len(stats[0][1]) > 0 else np.array([])
    pred_classes = np.concatenate([s[2] for s in stats]) if len(stats[0][2]) > 0 else np.array([])
    target_classes = np.concatenate([np.array(s[3]) for s in stats if len(s[3]) > 0]) if any(len(s[3]) > 0 for s in stats) else np.array([])
    
    if len(confidences) == 0:
        return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0, 'f1': 0}
    
    # Sort by confidence
    i = np.argsort(-confidences)
    confidences = confidences[i]
    pred_classes = pred_classes[i]
    correct = correct[i]
    
    unique_classes = np.unique(np.concatenate([target_classes, pred_classes])) if len(target_classes) > 0 else np.unique(pred_classes)
    
    ap, ap_class, p, r = [], [], [], []
    for cls in unique_classes:
        cls_mask = pred_classes == cls
        n_pred = cls_mask.sum()
        n_gt = (target_classes == cls).sum()
        
        if n_gt == 0 or n_pred == 0:
            continue
            
        # Accumulate metrics across all 10 thresholds
        ap_per_thresh = []
        # TP and FP for IoU > 0.5 (column 0)
        tp_50 = correct[cls_mask, 0].cumsum()
        fp_50 = (~correct[cls_mask, 0]).cumsum()
        
        recall_50 = tp_50 / (n_gt + 1e-16)
        precision_50 = tp_50 / (tp_50 + fp_50 + 1e-16)
        
        # Calculate AP for each threshold
        for j in range(correct.shape[1]):
            tp = correct[cls_mask, j].cumsum()
            fp = (~correct[cls_mask, j]).cumsum()
            recall = tp / (n_gt + 1e-16)
            precision = tp / (tp + fp + 1e-16)
            ap_per_thresh.append(compute_ap(recall, precision))
            
        ap.append(ap_per_thresh)
        ap_class.append(cls)
        p.append(precision_50[-1] if len(precision_50) else 0.0)
        r.append(recall_50[-1] if len(recall_50) else 0.0)
    
    if len(ap) == 0:
        return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0, 'f1': 0}
        
    # ap is list of shapes [10] representing thresholds
    ap = np.array(ap) # shape (n_classes, 10 thresholds)
    
    mAP50 = ap[:, 0].mean()
    mAP_all = ap.mean() # mean over all classes and all thresholds
    precision_mean = np.mean(p)
    recall_mean = np.mean(r)
    f1 = 2 * precision_mean * recall_mean / (precision_mean + recall_mean + 1e-6)
    
    return {
        'mAP50': mAP50,
        'mAP50-95': mAP_all,
        'precision': precision_mean,
        'recall': recall_mean,
        'f1': f1
    }


def compute_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """Compute AP using all-point interpolation."""
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    
    # Ensure precision is monotonically decreasing
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    
    # Find points where recall changes
    i = np.where(mrec[1:] != mrec[:-1])[0]
    
    # Sum areas under the curve
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    
    return ap


def save_checkpoint(state: dict, path: Path):
    """Save training checkpoint."""
    torch.save(state, path)
    print(f"[SAVED] Checkpoint: {path}")


def load_pretrained_weights(model: nn.Module, weights_path: str, device, surgical_transfer: bool = False):
    """Load pretrained weights from standard YOLOv5 checkpoints into YOLOv5-3D."""
    print(f"Loading pretrained weights: {weights_path}")
    
    try:
        ckpt = torch.load(weights_path, map_location=device, weights_only=False)
    except Exception as e:
        print(f"  [WARN] Could not load checkpoint: {e}")
        return
    
    # Extract state dict from checkpoint
    if 'model' in ckpt:
        src = ckpt['model']
        if hasattr(src, 'state_dict'):
            state_dict = src.float().state_dict()
        elif isinstance(src, dict):
            state_dict = src
        else:
            state_dict = {}
    elif 'state_dict' in ckpt:
        state_dict = ckpt['state_dict']
    elif isinstance(ckpt, dict):
        state_dict = ckpt
    else:
        if hasattr(ckpt, 'state_dict'):
            state_dict = ckpt.float().state_dict()
        else:
            print("  [WARN] Could not extract state_dict from checkpoint")
            return
    
    # Match compatible weights by shape
    model_state = model.state_dict()
    matched = 0
    
    for k, v in state_dict.items():
        # Strip common prefixes
        new_k = k
        for prefix in ('module.', 'model.'):
            if new_k.startswith(prefix):
                new_k = new_k[len(prefix):]
        
        # PHASE 7 SURGICAL TRANSFER:
        # Deliberately skip loading the old 'fusion' arrays so the new 
        # CrossAttention modules fall back to random initialization.
        if surgical_transfer and 'fusion' in new_k:
            continue
            
        if new_k in model_state and model_state[new_k].shape == v.shape:
            model_state[new_k] = v
            matched += 1
    
    model.load_state_dict(model_state)
    print(f"  Loaded {matched}/{len(model_state)} weights ({matched*100//len(model_state)}% matched)")


def print_args(args: dict):
    """Print arguments."""
    print(f"\n{colorstr('yellow')}Arguments:{colorstr('reset')}")
    for k, v in sorted(args.items()):
        print(f"  {k}: {v}")


def main():
    parser = argparse.ArgumentParser(description='YOLOv5-3D Training')
    
    # Data
    parser.add_argument('--data-dir', type=str, required=True, help='Data directory path')
    
    # Model
    parser.add_argument('--weights', type=str, default='', help='Pretrained weights path')
    parser.add_argument('--pretrained', action='store_true', help='Use pretrained backbone')
    parser.add_argument('--freeze-backbone', action='store_true', default=False, help='Freeze backbone')
    parser.add_argument('--surgical-transfer', action='store_true', help='Phase 7: Transfer weights but forcefully skip the fusion modules to initialize new Cross-Attention')
    parser.add_argument('--freeze-illum-fraction', type=float, default=0.33, help='Fraction of epochs to freeze illumination branch (Phase 3)')
    
    # Training
    parser.add_argument('--epochs', type=int, default=300, help='Number of epochs')
    parser.add_argument('--batch-size', type=int, default=8, help='Batch size')
    parser.add_argument('--imgsz', type=int, default=640, help='Image size')
    parser.add_argument('--patience', type=int, default=50, help='Early stopping patience (epochs without improvement)')
    
    # Optimizer
    parser.add_argument('--lr0', type=float, default=0.01, help='Initial learning rate')
    parser.add_argument('--lrf', type=float, default=0.2, help='Final LR multiplier')
    parser.add_argument('--momentum', type=float, default=0.937, help='SGD momentum')
    parser.add_argument('--weight-decay', type=float, default=0.0005, help='Weight decay')
    parser.add_argument('--warmup-epochs', type=int, default=1, help='Warmup epochs')
    parser.add_argument('--optimizer', type=str, default='AdamW', choices=['SGD', 'AdamW'])
    parser.add_argument('--cos-lr', action='store_true', default=True, help='Cosine LR schedule')
    
    # Device
    parser.add_argument('--device', type=str, default='', help='Device (cuda:0, cpu)')
    parser.add_argument('--workers', type=int, default=4, help='DataLoader workers')
    
    # Output
    parser.add_argument('--project', type=str, default='runs/train', help='Project directory')
    parser.add_argument('--name', type=str, default='exp', help='Experiment name')
    parser.add_argument('--exist-ok', action='store_true', help='Overwrite existing experiment')
    
    # Validation
    parser.add_argument('--conf-thres', type=float, default=0.001, help='Confidence threshold (0.001 for mAP eval)')
    parser.add_argument('--iou-thres', type=float, default=0.7, help='IoU threshold')
    
    # Mixed precision
    parser.add_argument('--amp', action='store_true', default=True, help='Use AMP')
    
    # Misc
    parser.add_argument('--save-period', type=int, default=-1, help='Save checkpoint every x epochs')
    parser.add_argument('--verbose', action='store_true', help='Verbose output')
    
    args = parser.parse_args()
    
    # Train
    train(**vars(args))


if __name__ == '__main__':
    main()


### val.py
Definition and code for `val.py`.


In [ ]:
%%writefile val.py
"""
YOLOv5-3D Validation Script
Comprehensive evaluation with detailed metrics and visualization.
"""

import os
import sys
import argparse
import json
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
from torch.amp import autocast
from tqdm import tqdm

# Add project root to path
ROOT = Path(__file__).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.yolo import YOLOv5_3D
from utils.datasets import create_dataloaders
from utils.metrics import MetricTracker, ConfusionMatrix, non_max_suppression, ap_per_class
from utils.plots import plot_confusion_matrix, plot_pr_curve, plot_detection_results
from utils.general import colorstr


def validate(
    model,
    dataloader,
    device,
    conf_thres: float = 0.001,
    iou_thres: float = 0.7,
    save_dir: str = 'runs/val',
    save_json: bool = False,
    plots: bool = True,
    verbose: bool = True
) -> Dict:
    """
    Validate model on dataset.
    
    Args:
        model: Model to validate
        dataloader: Validation dataloader
        device: Device
        conf_thres: Confidence threshold
        iou_thres: IoU threshold for NMS
        save_dir: Directory to save results
        save_json: Save results as JSON
        plots: Generate plots
        verbose: Print detailed results
    
    Returns:
        Dictionary with validation results
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    model.eval()
    
    # Get class names
    class_names = dataloader.dataset.classes
    num_classes = dataloader.dataset.num_classes
    
    # Metrics
    metric_tracker = MetricTracker(num_classes)
    confusion_matrix = ConfusionMatrix(num_classes, iou_threshold=0.5)
    
    # Stats collection
    all_predictions = []
    all_targets = []
    all_confidences = []
    all_pred_classes = []
    all_target_classes = []
    
    total_loss = 0
    n_batches = 0
    
    print(f"\n{colorstr('blue')}Running validation...{colorstr('reset')}")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validating'):
            images = batch['images'].to(device).float()
            depths = batch['depths'].to(device).float()
            illuminations = batch['illuminations'].to(device).float()
            targets = batch['targets'].to(device)
            
            # Forward
            with autocast('cuda', enabled=True):
                outputs = model(images, depths, illuminations)
            
            # NMS
            for i, out in enumerate(outputs):
                pred_np = out.cpu().numpy()
                det = non_max_suppression(pred_np, conf_thres, iou_thres)
                
                # Get targets for this image
                mask = targets[:, 0] == i
                t = targets[mask].cpu().numpy() if mask.any() else np.zeros((0, 6))
                
                # Scale
                img_h, img_w = images.shape[2:]
                
                for d in det:
                    if d.shape[0] > 0:
                        d[:, [0, 2]] *= img_w
                        d[:, [1, 3]] *= img_h
                
                # Update metrics
                metric_tracker.update(det, t, [(img_h, img_w)] * len(images))
                
                # Collect for PR curve
                if len(det) > 0:
                    all_predictions.extend(det[:, :4].tolist())
                    all_confidences.extend(det[:, 4].tolist())
                    all_pred_classes.extend(det[:, 5].astype(int).tolist())
                
                if t.shape[0] > 0:
                    all_target_classes.extend(t[:, 1].astype(int).tolist())
                
                # Update confusion matrix
                if t.shape[0] > 0:
                    labels = t[:, 1:].copy()
                    labels[:, 1] = (t[:, 1] - t[:, 3] / 2) * img_w
                    labels[:, 2] = (t[:, 2] - t[:, 4] / 2) * img_h
                    labels[:, 3] = (t[:, 1] + t[:, 3] / 2) * img_w
                    labels[:, 4] = (t[:, 2] + t[:, 4] / 2) * img_h
                    confusion_matrix.process_batch(det[0] if len(det) > 0 else np.zeros((0, 6)), labels)
    
    # Compute metrics
    metrics = metric_tracker.compute_metrics()
    
    # Print results
    print(f"\n{colorstr('green')}Validation Results:{colorstr('reset')}")
    print(f"  mAP@0.5: {metrics['mAP50']:.4f}")
    print(f"  mAP@0.5:0.95: {metrics['mAP50-95']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1: {metrics['f1']:.4f}")
    
    # Per-class results
    if verbose and 'ap_per_class' in metrics and metrics['ap_per_class'] is not None:
        print(f"\n{colorstr('blue')}Per-class AP@0.5:{colorstr('reset')}")
        ap = metrics['ap_per_class']
        for i, name in enumerate(class_names):
            if i < ap.shape[0]:
                print(f"  {name}: {ap[i, 0]:.4f}")
    
    # Generate plots
    if plots:
        print(f"\n{colorstr('blue')}Generating plots...{colorstr('reset')}")
        
        # Confusion matrix
        plot_confusion_matrix(
            confusion_matrix.get_matrix(),
            class_names,
            save_dir,
            normalize=True
        )
        
        # Save PR curve if we have data
        if len(all_confidences) > 0:
            # Create dummy PR data (would need proper calculation for accurate curve)
            print(f"  PR curve generation requires full stats calculation")
    
    # Save JSON
    if save_json:
        results = {
            'mAP50': float(metrics['mAP50']),
            'mAP50-95': float(metrics['mAP50-95']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'f1': float(metrics['f1']),
            'classes': class_names,
            'conf_thres': conf_thres,
            'iou_thres': iou_thres
        }
        
        with open(save_dir / 'results.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"{colorstr('green')}Saved results to {save_dir / 'results.json'}{colorstr('reset')}")
    
    # Confusion matrix stats
    tp, fp, fn = confusion_matrix.tp_fp_fn()
    print(f"\n{colorstr('blue')}Detection Stats:{colorstr('reset')}")
    for i, name in enumerate(class_names):
        if i < len(tp):
            print(f"  {name}: TP={int(tp[i])}, FP={int(fp[i])}, FN={int(fn[i])}")
    
    return metrics


def evaluate_model(
    weights: str,
    data_dir: str,
    batch_size: int = 8,
    imgsz: int = 640,
    conf_thres: float = 0.001,
    iou_thres: float = 0.7,
    device: str = '',
    save_dir: str = 'runs/val',
    plots: bool = True,
    verbose: bool = True
):
    """
    Evaluate model from weights file.
    
    Args:
        weights: Path to model weights
        data_dir: Data directory
        batch_size: Batch size
        imgsz: Image size
        conf_thres: Confidence threshold
        iou_thres: IoU threshold
        device: Device
        save_dir: Save directory
        plots: Generate plots
        verbose: Print detailed results
    """
    # Device
    device = torch.device(device if device else ('cuda' if torch.cuda.is_available() else 'cpu'))
    
    # Load model
    print(f"{colorstr('blue')}Loading model from {weights}...{colorstr('reset')}")
    ckpt = torch.load(weights, map_location=device)
    
    if 'model' in ckpt:
        state_dict = ckpt['model']
        if hasattr(state_dict, 'state_dict'):
            state_dict = state_dict.state_dict()
        nc = ckpt.get('nc', 6)
    else:
        state_dict = ckpt
        nc = 6
    
    model = YOLOv5_3D(nc=nc).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    
    print(f"{colorstr('green')}Model loaded!{colorstr('reset')}")
    
    # Create dataloader
    print(f"\n{colorstr('blue')}Loading data from {data_dir}...{colorstr('reset')}")
    _, val_loader, _ = create_dataloaders(
        data_dir=data_dir,
        batch_size=batch_size,
        img_size=imgsz,
        num_workers=4
    )
    
    print(f"Validation samples: {len(val_loader.dataset)}")
    
    # Validate
    results = validate(
        model=model,
        dataloader=val_loader,
        device=device,
        conf_thres=conf_thres,
        iou_thres=iou_thres,
        save_dir=save_dir,
        save_json=True,
        plots=plots,
        verbose=verbose
    )
    
    return results


def main():
    parser = argparse.ArgumentParser(description='YOLOv5-3D Validation')
    
    parser.add_argument('--weights', type=str, required=True, 
                       help='Model weights path')
    parser.add_argument('--data', type=str, required=True, 
                       help='Data directory')
    parser.add_argument('--batch-size', type=int, default=8, 
                       help='Batch size')
    parser.add_argument('--imgsz', type=int, default=640, 
                       help='Image size')
    parser.add_argument('--conf-thres', type=float, default=0.001, 
                       help='Confidence threshold')
    parser.add_argument('--iou-thres', type=float, default=0.7, 
                       help='IoU threshold for NMS')
    parser.add_argument('--device', type=str, default='', 
                       help='Device')
    parser.add_argument('--save-dir', type=str, default='runs/val', 
                       help='Save directory')
    parser.add_argument('--plots', action='store_true', default=True, 
                       help='Generate plots')
    parser.add_argument('--verbose', action='store_true', default=True, 
                       help='Print detailed results')
    
    args = parser.parse_args()
    
    evaluate_model(
        weights=args.weights,
        data_dir=args.data,
        batch_size=args.batch_size,
        imgsz=args.imgsz,
        conf_thres=args.conf_thres,
        iou_thres=args.iou_thres,
        device=args.device,
        save_dir=args.save_dir,
        plots=args.plots,
        verbose=args.verbose
    )


if __name__ == '__main__':
    main()



### models/common.py
Definition and code for `models/common.py`.


In [ ]:
%%writefile models/common.py
"""
YOLOv5-3D Common Modules
Core building blocks for the YOLOv5-3D architecture with illumination-depth fusion.
"""

import math
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F


def autopad(k, p=None, d=1):
    """Pad to 'same' shape outputs."""
    if d > 1:
        k = d * (k - 1) + 1 if isinstance(k, int) else [d * (x - 1) + 1 for x in k]
    if p is None:
        p = k // 2 if isinstance(k, int) else [x // 2 for x in k]
    return p


class Conv(nn.Module):
    """Standard convolution with BatchNorm and SiLU activation."""
    
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        """Initialize Conv layer.
        
        Args:
            c1: Input channels
            c2: Output channels
            k: Kernel size
            s: Stride
            p: Padding
            g: Groups
            d: Dilation
            act: Activation function (True for SiLU, False for None, or nn.Module)
        """
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU() if act is True else (act if isinstance(act, nn.Module) else nn.Identity())

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def forward_fuse(self, x):
        """Forward for fused model (BN folded into conv)."""
        return self.act(self.conv(x))


class DWConv(nn.Module):
    """Depth-wise convolution."""
    
    def __init__(self, c1, c2, k=1, s=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c1, k, s, autopad(k, None, d), groups=c1, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c1)
        self.act = nn.SiLU() if act is True else (act if isinstance(act, nn.Module) else nn.Identity())

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    """Standard bottleneck block."""
    
    def __init__(self, c1, c2, shortcut=True, g=1, e=0.5):
        """Initialize Bottleneck.
        
        Args:
            c1: Input channels
            c2: Output channels
            shortcut: Whether to use shortcut connection
            g: Groups
            e: Expansion ratio
        """
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_, c2, 3, 1, g=g)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))


class C3(nn.Module):
    """CSP Bottleneck with 3 convolutions."""
    
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):
        """Initialize C3 module.
        
        Args:
            c1: Input channels
            c2: Output channels
            n: Number of Bottleneck blocks
            shortcut: Use shortcut connections
            g: Groups
            e: Expansion ratio
        """
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, e=1.0) for _ in range(n)))

    def forward(self, x):
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))


class SPPF(nn.Module):
    """Spatial Pyramid Pooling - Fast version."""
    
    def __init__(self, c1, c2, k=5):
        """Initialize SPPF.
        
        Args:
            c1: Input channels
            c2: Output channels
            k: Kernel size for max pooling
        """
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c_ * 4, c2, 1, 1)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        return self.cv2(torch.cat([x, y1, y2, self.m(y2)], 1))


class Concat(nn.Module):
    """Concatenate a list of tensors along dimension."""
    
    def __init__(self, dimension=1):
        super().__init__()
        self.d = dimension

    def forward(self, x):
        return torch.cat(x, self.d)


class Focus(nn.Module):
    """Focus wh information into channel space."""
    
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, act=True):
        super().__init__()
        self.conv = Conv(c1 * 4, c2, k, s, p, g, act=act)

    def forward(self, x):
        # x(b,c,w,h) -> y(b,4c,w/2,h/2)
        return self.conv(torch.cat([
            x[..., ::2, ::2],
            x[..., 1::2, ::2],
            x[..., ::2, 1::2],
            x[..., 1::2, 1::2]
        ], 1))


# ============================================================================
# ILLUMINATION-DEPTH FUSION MODULES
# ============================================================================

class IlluminationDepthEncoder(nn.Module):
    """
    Encoder for Illumination (3ch) and Depth (1ch) inputs.
    Processes both modalities and produces fused features.
    
    Architecture:
    - Depth Branch: 1ch -> 16ch -> 32ch (lightweight)
    - Illumination Branch: 3ch -> 32ch -> 64ch
    - Fusion: Concatenation + Conv
    """
    
    def __init__(self, depth_ch=1, illum_ch=3, out_ch=64):
        super().__init__()
        
        # Depth encoder (lightweight - depth has less information)
        self.depth_encoder = nn.Sequential(
            Conv(depth_ch, 16, 3, 2),   # /2 downsample
            Conv(16, 32, 3, 2),          # /4 downsample
            Conv(32, 32, 3, 1),          # Keep features
        )
        
        # Illumination encoder
        self.illum_encoder = nn.Sequential(
            Conv(illum_ch, 32, 3, 2),    # /2 downsample
            Conv(32, 64, 3, 2),          # /4 downsample
            Conv(64, 64, 3, 1),          # Keep features
        )
        
        # Fusion layer
        self.fusion = Conv(32 + 64, out_ch, 1, 1)
        
    def forward(self, depth, illumination):
        """
        Args:
            depth: (B, 1, H, W) - Depth map
            illumination: (B, 3, H, W) - Illumination map
        
        Returns:
            fused: (B, out_ch, H/4, W/4) - Fused features
        """
        d_feat = self.depth_encoder(depth)
        i_feat = self.illum_encoder(illumination)
        
        # Ensure spatial dimensions match
        if d_feat.shape[2:] != i_feat.shape[2:]:
            d_feat = F.interpolate(d_feat, size=i_feat.shape[2:], mode='bilinear', align_corners=False)
        
        fused = torch.cat([d_feat, i_feat], dim=1)
        return self.fusion(fused)


class IlluminationGuidedAttention(nn.Module):
    """
    Illumination-Guided Attention Module (IGAM).
    Uses illumination information to modulate RGB features.
    
    This helps the model focus on defect regions that may be
    obscured by inconsistent lighting conditions.
    
    Parameters: ~50K (negligible overhead)
    """
    
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        
        # Channel attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
        )
        
        # Spatial attention
        self.spatial = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.BatchNorm2d(1),
        )
        
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x, illumination_hint=None):
        """
        Args:
            x: (B, C, H, W) - Input features
            illumination_hint: Optional illumination guidance
        
        Returns:
            Attention-weighted features
        """
        B, C, H, W = x.shape
        
        # Channel attention
        avg_out = self.fc(self.avg_pool(x).view(B, C))
        max_out = self.fc(self.max_pool(x).view(B, C))
        channel_att = self.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        x = x * channel_att
        
        # Spatial attention
        avg_spatial = torch.mean(x, dim=1, keepdim=True)
        max_spatial, _ = torch.max(x, dim=1, keepdim=True)
        spatial_att = self.sigmoid(self.spatial(torch.cat([avg_spatial, max_spatial], dim=1)))
        
        return x * spatial_att


class MultiModalFusion(nn.Module):
    """
    CBAM-Enhanced Multi-Modal Fusion Module.
    
    Replaces simple weighted sum with full CBAM attention after
    concatenating RGB backbone features with Illumination-Depth features.
    
    Flow: concat → 1x1 reduce → CBAM(channel + spatial) → residual add
    """
    
    def __init__(self, rgb_ch, id_ch, out_ch):
        super().__init__()
        
        # Channel alignment
        self.rgb_proj = Conv(rgb_ch, out_ch, 1, 1)
        self.id_proj = Conv(id_ch, out_ch, 1, 1)
        
        # Reduce concatenated channels: 2*out_ch → out_ch
        self.reduce = Conv(out_ch * 2, out_ch, 1, 1)
        
        # ---- CBAM: Channel Attention ----
        reduction = max(out_ch // 16, 4)
        self.ca_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.ca_max_pool = nn.AdaptiveMaxPool2d(1)
        self.ca_fc = nn.Sequential(
            nn.Conv2d(out_ch, reduction, 1, bias=False),
            nn.SiLU(inplace=True),
            nn.Conv2d(reduction, out_ch, 1, bias=False),
        )
        
        # ---- CBAM: Spatial Attention ----
        self.sa_conv = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.BatchNorm2d(1),
        )
        
        self.sigmoid = nn.Sigmoid()
        
        # Final refinement
        self.refine = Conv(out_ch, out_ch, 3, 1)
        
    def forward(self, rgb_feat, id_feat):
        """
        Args:
            rgb_feat: RGB backbone features (B, rgb_ch, H, W)
            id_feat: Illumination-Depth features (B, id_ch, H, W)
        Returns:
            Fused features (B, out_ch, H, W)
        """
        # Align channels
        rgb = self.rgb_proj(rgb_feat)
        id_ = self.id_proj(id_feat)
        
        # Spatial alignment
        if id_.shape[2:] != rgb.shape[2:]:
            id_ = F.interpolate(id_, size=rgb.shape[2:], mode='bilinear', align_corners=False)
        
        # Concatenate and reduce
        fused = self.reduce(torch.cat([rgb, id_], dim=1))
        
        # ---- Channel Attention: "What" channels matter ----
        ca_avg = self.ca_fc(self.ca_avg_pool(fused))
        ca_max = self.ca_fc(self.ca_max_pool(fused))
        channel_att = self.sigmoid(ca_avg + ca_max)
        fused = fused * channel_att
        
        # ---- Spatial Attention: "Where" to look ----
        sa_avg = torch.mean(fused, dim=1, keepdim=True)
        sa_max, _ = torch.max(fused, dim=1, keepdim=True)
        spatial_att = self.sigmoid(self.sa_conv(torch.cat([sa_avg, sa_max], dim=1)))
        fused = fused * spatial_att
        
        # Residual connection with RGB (preserves backbone gradients)
        fused = fused + rgb
        
        return self.refine(fused)

class CrossAttentionFusion(nn.Module):
    """
    Phase 7: Parameter-Efficient Cross-Attention Transformer Layer.
    
    Instead of concatenating, this uses the 1-channel Depth/Illumination feature
    map as a spatial Query (Q) to modulate the 3-channel RGB feature map (Key/Value).
    It avoids the O(N^2) memory explosion of standard Vision Transformers by 
    calculating a bilinear depth-guided spatial excitation map, minimizing parameters.
    """
    
    def __init__(self, rgb_ch, id_ch, out_ch):
        super().__init__()
        
        # Squeeze-Excitation dimensions to save massive numbers of parameters
        embed_dim = max(out_ch // 4, 16)
        
        # Q = Depth/Illumination, K = RGB, V = RGB
        self.q_proj = Conv(id_ch, embed_dim, 1, 1)
        self.k_proj = Conv(rgb_ch, embed_dim, 1, 1)
        self.v_proj = Conv(rgb_ch, out_ch, 1, 1)
        
        # Attention scale factor
        self.scale = embed_dim ** -0.5
        
        # Residual alignment for the RGB backbone to allow surgical load skipping
        self.rgb_align = Conv(rgb_ch, out_ch, 1, 1) if rgb_ch != out_ch else nn.Identity()
        
    def forward(self, rgb_feat, id_feat):
        # 1. Spatial alignment (in case ID map was processed at a different stride)
        if id_feat.shape[2:] != rgb_feat.shape[2:]:
            id_feat = F.interpolate(id_feat, size=rgb_feat.shape[2:], mode='bilinear', align_corners=False)
            
        # 2. Project inputs to lower-embedding space
        q = self.q_proj(id_feat)  # Query (Depth context)
        k = self.k_proj(rgb_feat) # Key (RGB context)
        v = self.v_proj(rgb_feat) # Value (RGB features)
        
        # 3. Bilinear Spatial Cross-Attention 
        # We element-wise multiply Q and K, then sum across channels to create 
        # a heavily correlated (B, 1, H, W) spatial attention gate.
        # This tells the network WHERE the depth map finds an anomaly on the RGB map.
        spatial_attn = torch.sum(q * k, dim=1, keepdim=True) * self.scale
        spatial_attn = torch.sigmoid(spatial_attn) 
        
        # 4. Modulate the Value (RGB) layer with the spatial topological gate
        attended_v = v * spatial_attn
        
        # 5. Residual Connection
        # Guarantees the network can fall back perfectly on its pretrained 2D weights
        return attended_v + self.rgb_align(rgb_feat)


# ============================================================================
# ADDITIONAL UTILITY MODULES
# ============================================================================

class Contract(nn.Module):
    """Contract width-height into channels."""
    
    def __init__(self, g=1):
        super().__init__()
        self.g = g

    def forward(self, x):
        b, c, h, w = x.shape
        s = 2  # contraction factor
        x = x.view(b, c, h // s, s, w // s, s)  # x(b,c,h/s,s,w/s,s)
        x = x.permute(0, 3, 5, 1, 2, 4).contiguous()  # x(b,s,s,c,h/s,w/s)
        return x.view(b, c * s * s, h // s, w // s)  # x(b,c*s*s,h/s,w/s)


class Expand(nn.Module):
    """Expand channels into width-height."""
    
    def __init__(self, g=1):
        super().__init__()
        self.g = g

    def forward(self, x):
        b, c, h, w = x.shape
        s = 2  # expansion factor
        x = x.view(b, s, s, c // s ** 2, h, w)  # x(b,s,s,c/s^2,h,w)
        x = x.permute(0, 3, 4, 1, 5, 2).contiguous()  # x(b,c/s^2,h,s,w,s)
        return x.view(b, c // s ** 2, h * s, w * s)  # x(b,c/s^2,h*s,w*s)


class ImplicitA(nn.Module):
    """Implicit knowledge addition."""
    
    def __init__(self, channel, mean=0., std=.02):
        super().__init__()
        self.channel = channel
        self.mean = mean
        self.std = std
        self.implicit = nn.Parameter(torch.zeros(1, channel, 1, 1))
        nn.init.normal_(self.implicit, mean=mean, std=std)

    def forward(self, x):
        return self.implicit + x


class ImplicitM(nn.Module):
    """Implicit knowledge multiplication."""
    
    def __init__(self, channel, mean=1., std=.02):
        super().__init__()
        self.channel = channel
        self.mean = mean
        self.std = std
        self.implicit = nn.Parameter(torch.ones(1, channel, 1, 1))
        nn.init.normal_(self.implicit, mean=mean, std=std)

    def forward(self, x):
        return self.implicit * x


def initialize_weights(model):
    """Initialize model weights."""
    for m in model.modules():
        t = type(m)
        if t is nn.Conv2d:
            pass  # nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif t is nn.BatchNorm2d:
            m.eps = 1e-3
            m.momentum = 0.03
        elif t in [nn.Hardswish, nn.LeakyReLU, nn.ReLU, nn.ReLU6, nn.SiLU]:
            m.inplace = True


def fuse_conv_and_bn(conv, bn):
    """Fuse Conv2d() and BatchNorm2d() layers."""
    fusedconv = nn.Conv2d(
        conv.in_channels,
        conv.out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        groups=conv.groups,
        bias=True
    ).requires_grad_(False).to(conv.weight.device)

    # Fuse weights
    w_conv = conv.weight.clone().view(conv.out_channels, -1)
    w_bn = torch.diag(bn.weight.div(torch.sqrt(bn.eps + bn.running_var)))
    fusedconv.weight.copy_(torch.mm(w_bn, w_conv).view(fusedconv.weight.shape))

    # Fuse bias
    b_conv = torch.zeros(conv.weight.size(0), device=conv.weight.device) if conv.bias is None else conv.bias
    b_bn = bn.bias - bn.weight.mul(bn.running_mean).div(torch.sqrt(bn.running_var + bn.eps))
    fusedconv.bias.copy_(torch.mm(w_bn, b_conv.reshape(-1, 1)).reshape(-1) + b_bn)

    return fusedconv



### models/yolo.py
Definition and code for `models/yolo.py`.


In [ ]:
%%writefile models/yolo.py
"""
YOLOv5-3D Model Definition
Enhanced YOLOv5 with Illumination-Depth fusion for PCB defect detection.
"""

import math
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from copy import deepcopy

from .common import (
    Conv, C3, SPPF, Concat, Bottleneck,
    IlluminationDepthEncoder, IlluminationGuidedAttention, CrossAttentionFusion, MultiModalFusion,
    DWConv, Focus, Contract, Expand,
    initialize_weights, fuse_conv_and_bn
)


def make_divisible(x, divisor):
    """Returns nearest x divisible by divisor."""
    if isinstance(divisor, torch.Tensor):
        divisor = divisor.item()
    return math.ceil(x / divisor) * divisor


def check_anchor_order(m):
    """Check anchor order against stride order."""
    a = m.anchors.prod(-1).mean(-1).view(-1)  # mean anchor area per output layer
    da = a[-1] - a[0]  # delta a
    ds = m.stride[-1] - m.stride[0]  # delta s
    if da.sign() != ds.sign():  # same order?
        m.anchors[:] = m.anchors.flip(0)


def parse_model(d, ch):
    """Parse model dict and create model."""
    anchors, nc, gd, gw = d['anchors'], d['nc'], d['depth_multiple'], d['width_multiple']
    na = (len(anchors[0]) // 2) if isinstance(anchors, list) else anchors
    no = na * (nc + 5)  # number of outputs per anchor

    layers, save, c2 = [], [], ch[-1]  # layers, savelist, ch out
    for i, (f, n, m, args) in enumerate(d['backbone'] + d['head']):
        m = eval(m) if isinstance(m, str) else m  # eval strings
        for j, a in enumerate(args):
            try:
                args[j] = eval(a) if isinstance(a, str) else a
            except NameError:
                pass

        n = n_ = max(round(n * gd), 1) if n > 1 else n  # depth gain
        if m in [Conv, C3, DWConv, SPPF, Focus, Bottleneck]:
            c1, c2 = ch[f], args[0]
            c2 = make_divisible(c2 * gw, 8) if c2 != no else c2
            args = [c1, c2, *args[1:]]
            if m is C3:
                args.insert(2, n)  # number of repeats
                n = 1
        elif m is nn.BatchNorm2d:
            args = [ch[f]]
        elif m is Concat:
            c2 = sum(ch[x] for x in f)
        elif m is Contract:
            c2 = ch[f] * args[0] ** 2
        elif m is Expand:
            c2 = ch[f] // args[0] ** 2
        else:
            c2 = ch[f]

        m_ = nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
        t = str(m)[8:-2].replace('__main__.', '')
        np = sum(x.numel() for x in m_.parameters())
        m_.i, m_.f, m_.type, m_.np = i, f, t, np
        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
        layers.append(m_)
        ch.append(c2)
    
    return nn.Sequential(*layers), sorted(save)


class Detections(nn.Module):
    """Detection output layer."""
    
    def __init__(self, nc=80, anchors=(), ch=(), inplace=True):
        super().__init__()
        self.nc = nc  # number of classes
        self.no = nc + 5  # number of outputs per anchor
        self.nl = len(anchors)  # number of detection layers
        self.na = len(anchors[0]) // 2  # number of anchors
        self.grid = [torch.zeros(1)] * self.nl
        self.anchor_grid = [torch.zeros(1)] * self.nl
        self.register_buffer('anchors', torch.tensor(anchors).float().view(self.nl, -1, 2))
        self.register_buffer('stride', torch.tensor([8.0, 16.0, 32.0]))
        self.inplace = inplace
        
        # Convolutional layers for detection
        self.m = nn.ModuleList(nn.Conv2d(x, self.no * self.na, 1) for x in ch)
        
    def forward(self, x):
        z = []
        for i in range(self.nl):
            x[i] = self.m[i](x[i])
            bs, _, ny, nx = x[i].shape
            x[i] = x[i].view(bs, self.na, self.no, ny, nx).permute(0, 1, 3, 4, 2).contiguous()
            
            if not self.training:
                if self.grid[i].shape[2:4] != x[i].shape[2:4]:
                    self.grid[i], self.anchor_grid[i] = self._make_grid(nx, ny, i)
                
                y = x[i].sigmoid()
                if self.inplace:
                    y[..., 0:2] = (y[..., 0:2] * 2 + self.grid[i]) * self.stride[i]
                    y[..., 2:4] = (y[..., 2:4] * 2) ** 2 * self.anchor_grid[i]
                else:
                    xy = (y[..., 0:2] * 2 + self.grid[i]) * self.stride[i]
                    wh = (y[..., 2:4] * 2) ** 2 * self.anchor_grid[i]
                    y = torch.cat((xy, wh, y[..., 4:]), -1)
                z.append(y.view(bs, -1, self.no))
        
        return x if self.training else (torch.cat(z, 1), x)
    
    def _make_grid(self, nx=20, ny=20, i=0):
        d = self.anchors[i].device
        t = self.anchors[i].dtype
        shape = 1, self.na, ny, nx, 2
        y, x = torch.arange(ny, device=d, dtype=t), torch.arange(nx, device=d, dtype=t)
        yv, xv = torch.meshgrid(y, x, indexing='ij')
        grid = torch.stack((xv, yv), 2).expand(shape) - 0.5
        anchor_grid = (self.anchors[i] * self.stride[i]).view((1, self.na, 1, 1, 2)).expand(shape)
        return grid, anchor_grid


class Model(nn.Module):
    """
    YOLOv5-3D Model with Illumination-Depth Fusion.
    
    This model extends the standard YOLOv5 architecture with:
    1. Illumination-Depth Encoder branch
    2. Multi-modal feature fusion
    3. Illumination-guided attention
    
    Args:
        cfg: Model configuration dict
        ch: Input channels (default 3 for RGB, supports 7 for RGB+Illum+Depth)
        nc: Number of classes
        anchors: Anchor sizes
        use_illumination_branch: Whether to use the illumination-depth branch
    """
    
    def __init__(self, cfg='yolo5_3d.yaml', ch=3, nc=None, anchors=None, 
                 use_illumination_branch=True):
        super().__init__()
        self.use_illumination_branch = use_illumination_branch
        
        # Default configuration for YOLOv5s-3D
        if isinstance(cfg, dict):
            self.yaml = cfg
        else:
            import yaml
            from pathlib import Path
            if not Path(cfg).exists():
                # Use default inline config
                self.yaml = self._get_default_config()
            else:
                with open(cfg, encoding='ascii', errors='ignore') as f:
                    self.yaml = yaml.safe_load(f)
        
        # Override model parameters
        if nc:
            self.yaml['nc'] = nc
        if anchors:
            self.yaml['anchors'] = anchors
        
        # Store input channels for reference
        self.yaml['ch'] = ch
        
        # Build model
        self.model, self.save = parse_model(self.yaml, [ch])
        self.names = [str(i) for i in range(self.yaml['nc'])]
        self.inplace = True
        
        # Initialize strides and biases
        m = self.model[-1]  # Detect()
        if isinstance(m, Detections):
            s = 256  # 2x min stride
            m.inplace = self.inplace
            m.stride = torch.tensor([8., 16., 32.])  # forward
            check_anchor_order(m)
            m.anchors /= m.stride.view(-1, 1, 1)
            self.stride = m.stride
            self._initialize_biases()
        
        # Initialize weights
        initialize_weights(self)
        self.info()
    
    def _get_default_config(self):
        """Get default YOLOv5s-3D configuration."""
        return {
            'nc': 6,  # number of classes (default PCB defect types)
            'depth_multiple': 0.33,
            'width_multiple': 0.50,
            'anchors': [
                [10, 13, 16, 30, 33, 23],  # P3/8
                [30, 61, 62, 45, 59, 119],  # P4/16
                [116, 90, 156, 198, 373, 326]  # P5/32
            ],
            'backbone': [
                [-1, 1, 'Conv', [64, 6, 2, 2]],  # 0-P1/2
                [-1, 1, 'Conv', [128, 3, 2]],    # 1-P2/4
                [-1, 3, 'C3', [128]],
                [-1, 1, 'Conv', [256, 3, 2]],    # 3-P3/8
                [-1, 6, 'C3', [256]],
                [-1, 1, 'Conv', [512, 3, 2]],    # 5-P4/16
                [-1, 9, 'C3', [512]],
                [-1, 1, 'Conv', [1024, 3, 2]],   # 7-P5/32
                [-1, 3, 'C3', [1024]],
                [-1, 1, 'SPPF', [1024, 5]],      # 9
            ],
            'head': [
                [-1, 1, 'Conv', [512, 1, 1]],
                [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
                [[-1, 6], 1, 'Concat', [1]],
                [-1, 3, 'C3', [512, False]],     # 13
                [-1, 1, 'Conv', [256, 1, 1]],
                [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
                [[-1, 4], 1, 'Concat', [1]],
                [-1, 3, 'C3', [256, False]],     # 17 (P3/8-small)
                [-1, 1, 'Conv', [256, 3, 2]],
                [[-1, 14], 1, 'Concat', [1]],
                [-1, 3, 'C3', [512, False]],     # 20 (P4/16-medium)
                [-1, 1, 'Conv', [512, 3, 2]],
                [[-1, 10], 1, 'Concat', [1]],
                [-1, 3, 'C3', [1024, False]],    # 23 (P5/32-large)
                [[17, 20, 23], 1, 'Detections', ['nc', 'anchors']],  # Detect(P3, P4, P5)
            ]
        }
    
    def _initialize_biases(self, cf=None):
        """Initialize biases for detection heads."""
        m = self.model[-1]  # Detect() module
        for mi, s in zip(m.m, m.stride):
            b = mi.bias.view(m.na, -1)
            b.data[:, 4] += math.log(8 / (640 / s) ** 2)
            b.data[:, 5:] += math.log(0.6 / (m.nc - 0.99)) if cf is None else torch.log(cf / cf.sum())
            mi.bias = nn.Parameter(b.view(-1), requires_grad=True)
    
    def forward(self, x, depth=None, illumination=None):
        """
        Forward pass.
        
        Args:
            x: Input RGB image (B, 3, H, W)
            depth: Optional depth map (B, 1, H, W)
            illumination: Optional illumination map (B, 3, H, W)
        
        Returns:
            During training: raw predictions
            During inference: (predictions, training_output)
        """
        # If using 7-channel input directly
        if x.shape[1] == 7:
            # Split channels: RGB(3) + Illumination(3) + Depth(1)
            rgb = x[:, :3, :, :]
            illumination = x[:, 3:6, :, :]
            depth = x[:, 6:7, :, :]
            x = rgb
        
        return self._forward_once(x)
    
    def _forward_once(self, x):
        """Single forward pass through the model."""
        y, dt = [], []  # outputs
        for m in self.model:
            if m.f != -1:  # if not from previous layer
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x if m.i in self.save else None)
        return x
    
    def _forward_once_with_fusion(self, x, depth, illumination):
        """Forward pass with illumination-depth fusion."""
        # This will be implemented in the extended model
        pass
    
    def info(self, verbose=False, img_size=640):
        """Print model information."""
        n_p = sum(x.numel() for x in self.parameters())
        n_g = sum(x.numel() for x in self.parameters() if x.requires_grad)
        
        print(f"\n{'='*60}")
        print(f"YOLOv5-3D Model Summary")
        print(f"{'='*60}")
        print(f"  Layers: {len(list(self.modules()))}")
        print(f"  Parameters: {n_p:,}")
        print(f"  Gradients: {n_g:,}")
        print(f"  Image size: {img_size}")
        print(f"{'='*60}\n")


class YOLOv5_3D(nn.Module):
    """
    Complete YOLOv5-3D Model with Illumination-Depth Branch.
    
    This is the main model class that combines:
    1. YOLOv5 backbone (pretrained weights supported)
    2. Illumination-Depth encoder branch
    3. Multi-modal fusion module
    4. YOLO detection head
    
    Designed for:
    - PCB defect detection
    - Handling inconsistent lighting conditions
    - Edge deployment (efficient architecture)
    """
    
    def __init__(self, nc=6, input_size=640, pretrained=False, 
                 freeze_backbone=False, use_illum_attention=True):
        """
        Initialize YOLOv5-3D.
        
        Args:
            nc: Number of classes
            input_size: Input image size
            pretrained: Load pretrained YOLOv5s weights
            freeze_backbone: Freeze backbone layers
            use_illum_attention: Use illumination-guided attention
        """
        super().__init__()
        
        self.nc = nc
        self.input_size = input_size
        self.use_illum_attention = use_illum_attention
        
        # ============ Backbone (YOLOv5s CSPDarknet) ============
        # Stage 1: Conv + C3
        self.stem = Conv(3, 32, 6, 2, 2)  # P1/2
        self.stage1 = nn.Sequential(
            Conv(32, 64, 3, 2),  # P2/4
            C3(64, 64, 1),
        )
        
        # Stage 2: P3/8
        self.stage2 = nn.Sequential(
            Conv(64, 128, 3, 2),
            C3(128, 128, 3),
        )
        
        # Stage 3: P4/16
        self.stage3 = nn.Sequential(
            Conv(128, 256, 3, 2),
            C3(256, 256, 6),
        )
        
        # Stage 4: P5/32
        self.stage4 = nn.Sequential(
            Conv(256, 512, 3, 2),
            C3(512, 512, 9),
            SPPF(512, 512, 5),
        )
        
        # ============ Illumination-Depth Branch ============
        self.id_encoder = IlluminationDepthEncoder(
            depth_ch=1, 
            illum_ch=3, 
            out_ch=64
        )
        
        # Fusion modules at different scales utilizing Phase 7 Spatial Cross-Attention
        # x2 (P2/4): 64ch, x3 (P3/8): 128ch, x4 (P4/16): 256ch, x5 (P5/32): 512ch
        self.fusion_p4 = CrossAttentionFusion(256, 64, 256)  # x4 has 256 channels
        self.fusion_p3 = CrossAttentionFusion(128, 64, 128)  # x3 has 128 channels
        self.fusion_p2 = CrossAttentionFusion(64, 64, 64)    # x2 has 64 channels
        
        # Illumination-guided attention (matching fusion output channels)
        if use_illum_attention:
            self.illum_att_p4 = IlluminationGuidedAttention(256, reduction=16)
            self.illum_att_p3 = IlluminationGuidedAttention(128, reduction=16)
            self.illum_att_p2 = IlluminationGuidedAttention(64, reduction=16)
        
        # ============ Neck (FPN + PAN) ============
        # Top-down pathway (FPN)
        # x5 (512ch) → upsample → 256ch, concat with x4 (256ch) → 512ch input
        self.up_p5_p4 = nn.Sequential(
            Conv(512, 256, 1, 1),
            nn.Upsample(scale_factor=2, mode='nearest'),
        )
        self.c3_p4 = C3(512, 256, 3, False)  # input: 256+256=512, output: 256
        
        # p4_td (256ch) → upsample → 128ch, concat with x3 (128ch) → 256ch input
        self.up_p4_p3 = nn.Sequential(
            Conv(256, 128, 1, 1),
            nn.Upsample(scale_factor=2, mode='nearest'),
        )
        self.c3_p3 = C3(256, 128, 3, False)  # input: 128+128=256, output: 128
        
        # Bottom-up pathway (PAN)
        # p3_td (128ch) → downsample → 128ch, concat with p4_td (256ch) → 384ch input
        self.down_p3_p4 = Conv(128, 128, 3, 2)
        self.c3_p4_pan = C3(384, 256, 3, False)  # input: 128+256=384, output: 256
        
        # p4_bu (256ch) → downsample → 256ch, concat with x5 (512ch) → 768ch input
        self.down_p4_p5 = Conv(256, 256, 3, 2)
        self.c3_p5_pan = C3(768, 512, 3, False)  # input: 256+512=768, output: 512
        
        # ============ Detection Head ============
        self.detect = Detections(
            nc=nc,
            anchors=[
                [10, 13, 16, 30, 33, 23],      # P3/8 (small objects)
                [30, 61, 62, 45, 59, 119],     # P4/16 (medium objects)
                [116, 90, 156, 198, 373, 326]  # P5/32 (large objects)
            ],
            ch=[128, 256, 512]
        )
        
        # Stride
        self.stride = torch.tensor([8., 16., 32.])
        
        # CRITICAL: Normalize anchors by stride so _make_grid produces correct
        # pixel-space anchor sizes during inference (anchor_grid = anchors * stride).
        # Without this, inference boxes are 8-32x too large and mAP never improves.
        self.detect.anchors /= self.detect.stride.view(-1, 1, 1)
        check_anchor_order(self.detect)
        
        # Freeze backbone if specified
        if freeze_backbone:
            self._freeze_backbone()
        
        # Initialize weights
        initialize_weights(self)
        
        # Load pretrained weights
        if pretrained:
            self._load_pretrained(pretrained)
        
        self._print_model_info()
    
    def _freeze_backbone(self):
        """Freeze backbone layers for transfer learning."""
        freeze_modules = [
            self.stem, self.stage1, self.stage2
        ]
        for module in freeze_modules:
            for param in module.parameters():
                param.requires_grad = False
        print("[INFO] Backbone layers (stem, stage1, stage2) frozen.")
    
    def _load_pretrained(self, weights_path):
        """Load pretrained YOLOv5s weights with layer-name mapping.
        
        Maps standard YOLOv5s sequential `model.N` naming to our
        named architecture (`stem`, `stage1`, etc.).
        
        Transfers: backbone, neck, and detection head (if shapes match).
        Skips: illumination/depth encoder, fusion modules, attention modules.
        """
        import pathlib
        _orig = getattr(pathlib, 'PosixPath', None)
        pathlib.PosixPath = pathlib.WindowsPath  # Fix Linux→Windows
        
        try:
            # Add stub classes for unpickling
            import models.yolo as _ym
            import models.common as _cm
            for _n in ['C3', 'Conv', 'SPPF', 'Bottleneck', 'Concat']:
                if hasattr(_cm, _n) and not hasattr(_ym, _n):
                    setattr(_ym, _n, getattr(_cm, _n))
            if not hasattr(_ym, 'Detect'):
                setattr(_ym, 'Detect', Detections)
            
            ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
            if 'model' in ckpt:
                src = ckpt['model']
                if hasattr(src, 'state_dict'):
                    pretrained_sd = src.float().state_dict()
                elif isinstance(src, dict):
                    pretrained_sd = src
                else:
                    pretrained_sd = {}
            else:
                pretrained_sd = ckpt
            
            # ===== Layer-name mapping: YOLOv5s model.N → our named modules =====
            # YOLOv5s backbone: model.0=stem, model.1+2=stage1, model.3+4=stage2,
            #   model.5+6=stage3, model.7+8+9=stage4(conv+sppf+c3)
            # YOLOv5s neck: model.10=up, model.11=cat, model.12=c3, model.13=up,
            #   model.14=cat, model.15=c3, model.16=down, model.17=cat, model.18=c3,
            #   model.19=down, model.20=cat, model.21=c3
            # YOLOv5s head: model.24=detect
            prefix_map = {
                'model.0.': 'stem.',                # Focus/Conv P1/2
                'model.1.': 'stage1.0.',             # Conv P2/4
                'model.2.': 'stage1.1.',             # C3
                'model.3.': 'stage2.0.',             # Conv P3/8
                'model.4.': 'stage2.1.',             # C3
                'model.5.': 'stage3.0.',             # Conv P4/16
                'model.6.': 'stage3.1.',             # C3
                'model.7.': 'stage4.0.',             # Conv P5/32
                'model.8.': 'stage4.1.',             # SPPF
                'model.9.': 'stage4.2.',             # C3
                # Neck
                'model.12.': 'neck_c3_1.',           # C3 after first concat
                'model.15.': 'neck_c3_2.',           # C3 after second concat
                'model.18.': 'neck_c3_3.',           # C3 after third concat
                'model.21.': 'neck_c3_4.',           # C3 after fourth concat
                'model.10.': 'lateral_conv1.',        # 1x1 conv for FPN
                'model.13.': 'lateral_conv2.',        # 1x1 conv for FPN
                'model.16.': 'downsample1.',          # Downsample conv
                'model.19.': 'downsample2.',          # Downsample conv
                # Detection head
                'model.24.': 'detect.',               # Detect layer
            }
            
            self_sd = self.state_dict()
            transferred = 0
            skipped_shape = 0
            skipped_nomap = 0
            
            for src_key, src_val in pretrained_sd.items():
                # Try each prefix mapping
                dst_key = None
                for src_prefix, dst_prefix in prefix_map.items():
                    if src_key.startswith(src_prefix):
                        dst_key = dst_prefix + src_key[len(src_prefix):]
                        break
                
                if dst_key is None:
                    skipped_nomap += 1
                    continue
                
                if dst_key in self_sd:
                    if src_val.shape == self_sd[dst_key].shape:
                        self_sd[dst_key] = src_val
                        transferred += 1
                    else:
                        skipped_shape += 1
                else:
                    skipped_nomap += 1
            
            self.load_state_dict(self_sd)
            
            total = len(self_sd)
            print(f"\n{'='*60}")
            print(f"  PRETRAINED WEIGHT TRANSFER")
            print(f"{'='*60}")
            print(f"  Source: {weights_path}")
            print(f"  Transferred: {transferred}/{total} layers")
            print(f"  Shape mismatch (skipped): {skipped_shape}")
            print(f"  No mapping (skipped): {skipped_nomap}")
            print(f"  Randomly initialized: {total - transferred}")
            print(f"  (ID encoder, fusion, attention = random init)")
            print(f"{'='*60}\n")
            
        except Exception as e:
            print(f"[WARNING] Could not load pretrained weights: {e}")
        finally:
            if _orig is not None:
                pathlib.PosixPath = _orig
    
    def _print_model_info(self):
        """Print model information."""
        n_p = sum(x.numel() for x in self.parameters())
        n_g = sum(x.numel() for x in self.parameters() if x.requires_grad)
        
        print(f"\n{'='*60}")
        print(f"  YOLOv5-3D Model Summary")
        print(f"{'='*60}")
        print(f"  Classes: {self.nc}")
        print(f"  Input size: {self.input_size}")
        print(f"  Total parameters: {n_p:,}")
        print(f"  Trainable parameters: {n_g:,}")
        print(f"  Illumination attention: {self.use_illum_attention}")
        print(f"{'='*60}\n")
    
    def forward(self, rgb, depth=None, illumination=None):
        """
        Forward pass.
        
        Args:
            rgb: RGB image (B, 3, H, W)
            depth: Depth map (B, 1, H, W)
            illumination: Illumination map (B, 3, H, W)
        
        Returns:
            Detection outputs
        """
        # Process depth and illumination if provided
        id_features = None
        if depth is not None and illumination is not None:
            id_features = self.id_encoder(depth, illumination)
        
        # ============ Backbone ============
        x1 = self.stem(rgb)        # P1/2: 320
        x2 = self.stage1(x1)       # P2/4: 160
        x3 = self.stage2(x2)       # P3/8: 80
        x4 = self.stage3(x3)       # P4/16: 40
        x5 = self.stage4(x4)       # P5/32: 20
        
        # ============ Illumination-Depth Fusion ============
        if id_features is not None:
            # Upsample id_features to match different scales
            id_p4 = F.interpolate(id_features, size=x4.shape[2:], mode='bilinear', align_corners=False)
            id_p3 = F.interpolate(id_features, size=x3.shape[2:], mode='bilinear', align_corners=False)
            id_p2 = F.interpolate(id_features, size=x2.shape[2:], mode='bilinear', align_corners=False)
            
            # Apply fusion
            x4 = self.fusion_p4(x4, id_p4)
            x3 = self.fusion_p3(x3, id_p3)
            x2 = self.fusion_p2(x2, id_p2)
            
            # Apply illumination-guided attention
            if self.use_illum_attention:
                x4 = self.illum_att_p4(x4)
                x3 = self.illum_att_p3(x3)
                x2 = self.illum_att_p2(x2)
        
        # ============ Neck (FPN + PAN) ============
        # Top-down (FPN)
        p5_up = self.up_p5_p4(x5)
        p4_td = self.c3_p4(torch.cat([p5_up, x4], dim=1))
        
        p4_up = self.up_p4_p3(p4_td)
        p3_td = self.c3_p3(torch.cat([p4_up, x3], dim=1))
        
        # Bottom-up (PAN)
        p3_down = self.down_p3_p4(p3_td)
        p4_bu = self.c3_p4_pan(torch.cat([p3_down, p4_td], dim=1))
        
        p4_down = self.down_p4_p5(p4_bu)
        p5_bu = self.c3_p5_pan(torch.cat([p4_down, x5], dim=1))
        
        # ============ Detection Head ============
        outputs = self.detect([p3_td, p4_bu, p5_bu])
        
        return outputs


def attempt_load(weights, map_location=None, inplace=True, fuse=True):
    """Load model from weights file."""
    from pathlib import Path
    
    # Single model
    if isinstance(weights, (str, Path)):
        ckpt = torch.load(weights, map_location=map_location, weights_only=False)
        model = ckpt['ema' if ckpt.get('ema') else 'model'].float()
        
        # Fuse
        if fuse:
            model = fuse_model(model)
        
        # Compatibility updates
        for m in model.modules():
            if type(m) in [nn.Hardswish, nn.LeakyReLU, nn.ReLU, nn.ReLU6, nn.SiLU]:
                m.inplace = inplace
            elif type(m) is nn.Upsample:
                m.recompute_scale_factor = None
        
        model.eval()
        return model


def fuse_model(model):
    """Fuse Conv2d + BatchNorm2d layers."""
    for m in model.modules():
        if hasattr(m, 'conv') and hasattr(m, 'bn'):
            m.conv = fuse_conv_and_bn(m.conv, m.bn)
            delattr(m, 'bn')
            m.forward = m.forward_fuse
    return model


# ── Stubs for pickle compatibility with standard YOLOv5 checkpoints ──
# Standard yolov5s.pt files contain pickled 'DetectionModel' objects.
# These stubs allow torch.load to unpickle them without errors.
class DetectionModel(nn.Module):
    """Stub: allows torch.load to unpickle standard YOLOv5 checkpoints."""
    def __init__(self, *args, **kwargs):
        super().__init__()

class _ModelStub(DetectionModel):
    """Alias for DetectionModel (some older checkpoints use this name)."""
    pass

# Keep 'Model' in module namespace for pickle compat, but don't shadow the real Model class above
import sys as _sys
_sys.modules[__name__].__dict__.setdefault('DetectionModel', DetectionModel)



### models/yolo5_3d.yaml
Definition and code for `models/yolo5_3d.yaml`.


In [ ]:
%%writefile models/yolo5_3d.yaml
# YOLOv5-3D Model Configuration
# Enhanced with Illumination-Depth Branch for PCB Defect Detection

# Model parameters
nc: 6  # number of classes (update based on your dataset)
depth_multiple: 0.33  # model depth multiple
width_multiple: 0.50  # layer channel multiple

# Anchors for 3 detection scales
anchors:
  - [10, 13, 16, 30, 33, 23]       # P3/8   - small objects
  - [30, 61, 62, 45, 59, 119]      # P4/16  - medium objects
  - [116, 90, 156, 198, 373, 326]  # P5/32  - large objects

# Backbone
# Input: 640x640x3 (RGB) or 640x640x7 (RGB+Illum+Depth)
backbone:
  # [from, number, module, args]
  # P1/2 - stem
  - [-1, 1, Conv, [64, 6, 2, 2]]  # 0: 320x320
  
  # P2/4
  - [-1, 1, Conv, [128, 3, 2]]    # 1: 160x160
  - [-1, 3, C3, [128]]            # 2: C3 block
  
  # P3/8
  - [-1, 1, Conv, [256, 3, 2]]    # 3: 80x80
  - [-1, 6, C3, [256]]            # 4: C3 block
  
  # P4/16
  - [-1, 1, Conv, [512, 3, 2]]    # 5: 40x40
  - [-1, 9, C3, [512]]            # 6: C3 block
  
  # P5/32
  - [-1, 1, Conv, [1024, 3, 2]]   # 7: 20x20
  - [-1, 3, C3, [1024]]           # 8: C3 block
  - [-1, 1, SPPF, [1024, 5]]      # 9: SPPF

# Head
head:
  # FPN - Top-down pathway
  - [-1, 1, Conv, [512, 1, 1]]                    # 10
  - [-1, 1, nn.Upsample, [null, 2, 'nearest']]    # 11
  - [[-1, 6], 1, Concat, [1]]                     # 12: concat with P4
  - [-1, 3, C3, [512, false]]                     # 13: P4 fusion
  
  - [-1, 1, Conv, [256, 1, 1]]                    # 14
  - [-1, 1, nn.Upsample, [null, 2, 'nearest']]    # 15
  - [[-1, 4], 1, Concat, [1]]                     # 16: concat with P3
  - [-1, 3, C3, [256, false]]                     # 17: P3/8-small
  
  # PAN - Bottom-up pathway
  - [-1, 1, Conv, [256, 3, 2]]                    # 18
  - [[-1, 14], 1, Concat, [1]]                    # 19: concat
  - [-1, 3, C3, [512, false]]                     # 20: P4/16-medium
  
  - [-1, 1, Conv, [512, 3, 2]]                    # 21
  - [[-1, 10], 1, Concat, [1]]                    # 22: concat
  - [-1, 3, C3, [1024, false]]                    # 23: P5/32-large
  
  # Detect
  - [[17, 20, 23], 1, Detections, [nc, anchors]] # 24: Detect(P3, P4, P5)



### models/__init__.py
Definition and code for `models/__init__.py`.


In [ ]:
%%writefile models/__init__.py
# YOLOv5-3D Models Package
from .common import (
    Conv, C3, SPPF, Bottleneck, Concat, Focus, DWConv,
    Contract, Expand,
    IlluminationDepthEncoder, IlluminationGuidedAttention, MultiModalFusion,
    initialize_weights, fuse_conv_and_bn
)
from .yolo import YOLOv5_3D, Model, Detections, attempt_load, fuse_model

__all__ = [
    'Conv', 'C3', 'SPPF', 'Bottleneck', 'Concat', 'Focus', 'DWConv',
    'Contract', 'Expand',
    'IlluminationDepthEncoder', 'IlluminationGuidedAttention', 'MultiModalFusion',
    'initialize_weights', 'fuse_conv_and_bn',
    'YOLOv5_3D', 'Model', 'Detections', 'attempt_load', 'fuse_model'
]



### utils/datasets.py
Definition and code for `utils/datasets.py`.


In [ ]:
%%writefile utils/datasets.py
"""
PCB Dataset with Illumination-Depth Support
Handles RGB images, YOLO format labels, and .npy files for depth and illumination.
"""

import os
import glob
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from typing import Optional, List, Tuple, Dict, Any
import warnings

# Suppress harmless Albumentations warnings when applying Color Jitters to grayscale Depth Maps
warnings.filterwarnings('ignore', category=UserWarning, module='albucore.decorators')
warnings.filterwarnings('ignore', category=UserWarning, module='albumentations.check_version')


class PCBDataset(Dataset):
    """
    PCB Defect Detection Dataset with Illumination-Depth Support.
    
    Expected directory structure:
    data/
    ├── train/
    │   ├── images/          # RGB images (.jpg, .png)
    │   ├── labels/          # YOLO format labels (.txt)
    │   ├── depth/           # Depth maps (.npy)
    │   └── illumination/    # Illumination maps (.npy)
    ├── valid/
    │   ├── images/
    │   ├── labels/
    │   ├── depth/
    │   └── illumination/
    └── test/
        ├── images/
        ├── labels/
        ├── depth/
        └── illumination/
    
    YOLO label format: class x_center y_center width height (normalized 0-1)
    
    .npy files:
    - depth: (H, W) or (H, W, 1) - single channel depth map
    - illumination: (H, W, 3) - 3 channel illumination map
    """
    
    def __init__(
        self,
        data_dir: str,
        split: str = 'train',
        img_size: int = 640,
        augment: bool = True,
        normalize: bool = True,
        cache: bool = False,
        classes: Optional[List[str]] = None
    ):
        """
        Initialize PCB Dataset.
        
        Args:
            data_dir: Root data directory
            split: 'train', 'valid', or 'test'
            img_size: Target image size
            augment: Apply data augmentation
            normalize: Normalize images to [0, 1]
            cache: Cache images in memory
            classes: List of class names
        """
        super().__init__()
        
        self.data_dir = Path(data_dir)
        self.split_dir = self.data_dir / split
        self.img_size = img_size
        self.augment = augment and (split == 'train')
        self.normalize = normalize
        self.cache = cache
        
        # Setup paths
        self.images_dir = self.split_dir / 'images'
        self.labels_dir = self.split_dir / 'labels'
        self.depth_dir = self.split_dir / 'depth'
        self.illum_dir = self.split_dir / 'illumination'
        
        # Get all image files
        self.image_files = self._get_image_files()
        print(f"[{split.upper()}] Found {len(self.image_files)} images")
        
        # Class names
        self.classes = classes or self._auto_detect_classes()
        self.num_classes = len(self.classes)
        print(f"[{split.upper()}] Classes: {self.classes}")
        
        # Data augmentation pipeline
        self.transform = self._build_transforms()
        
        # Phase 7: Targeted minority class pool for Copy-Paste
        # Indices: Excess_solder (8), Insufficient_solder (9), Spike (11)
        self.minority_classes = [8, 9, 11]
        self.minority_patches = []
        if self.augment:
            print(f"[{split.upper()}] Pre-caching minority 3D topologies into RAM for fast Copy-Paste...")
            for i, img_path in enumerate(self.image_files):
                label_path = self._get_label_path(img_path)
                if label_path.exists():
                    has_minority = False
                    with open(label_path, 'r') as f:
                        for line in f:
                            if line.strip() and int(line.split()[0]) in self.minority_classes:
                                has_minority = True
                                break
                    
                    if has_minority:
                        try:
                            # Load full image arrays once over disk
                            data = self._load_data(img_path)
                            d_image = data['image']
                            d_bboxes = data['bboxes']
                            d_labels = data['class_labels']
                            d_depth = data['depth']
                            d_illum = data['illumination']
                            
                            # Scale the Depth/Illuminations into identical coordinate meshes
                            dH_rgb, dW_rgb = d_image.shape[:2]
                            if d_depth.shape[:2] != (dH_rgb, dW_rgb):
                                d_depth = cv2.resize(d_depth, (dW_rgb, dH_rgb))
                                if d_depth.ndim == 2:
                                    d_depth = np.expand_dims(d_depth, axis=-1)
                            if d_illum.shape[:2] != (dH_rgb, dW_rgb):
                                d_illum = cv2.resize(d_illum, (dW_rgb, dH_rgb))
                                if d_illum.ndim == 2:
                                    d_illum = np.stack([d_illum]*3, axis=-1)
                            
                            # Slice out ONLY the minority 3D topology coordinate meshes and RAM Cache them
                            for d_box, d_lbl in zip(d_bboxes, d_labels):
                                if d_lbl in self.minority_classes:
                                    dx_c, dy_c, dw, dh = d_box
                                    x1 = max(0, int((dx_c - dw/2) * dW_rgb))
                                    y1 = max(0, int((dy_c - dh/2) * dH_rgb))
                                    x2 = min(dW_rgb, int((dx_c + dw/2) * dW_rgb))
                                    y2 = min(dH_rgb, int((dy_c + dh/2) * dH_rgb))
                                    
                                    if x2 > x1 + 2 and y2 > y1 + 2:
                                        self.minority_patches.append({
                                            'image': d_image[y1:y2, x1:x2].copy(),
                                            'depth': d_depth[y1:y2, x1:x2].copy(),
                                            'illumination': d_illum[y1:y2, x1:x2].copy(),
                                            'label': d_lbl
                                        })
                        except Exception:
                            pass
            print(f"[{split.upper()}] RAM Cached {len(self.minority_patches)} 3D minority patches.")
        
        # Cache
        self.cached_data = {}
        if cache:
            self._cache_data()
    
    def _get_image_files(self) -> List[Path]:
        """Get all image file paths."""
        extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']
        image_files = []
        for ext in extensions:
            image_files.extend(self.images_dir.glob(ext))
            image_files.extend(self.images_dir.glob(ext.upper()))
        return sorted(image_files)
    
    def _auto_detect_classes(self) -> List[str]:
        """Auto-detect classes from labels."""
        # Try to read from data.yaml or classes.txt
        classes_file = self.data_dir / 'classes.txt'
        yaml_file = self.data_dir / 'data.yaml'
        
        if classes_file.exists():
            with open(classes_file, 'r') as f:
                classes = [line.strip() for line in f if line.strip()]
            return classes
        
        if yaml_file.exists():
            import yaml
            with open(yaml_file, 'r') as f:
                data = yaml.safe_load(f)
                if 'names' in data:
                    return data['names']
        
        # Auto-detect from labels
        class_ids = set()
        for img_path in self.image_files[:100]:  # Check first 100
            label_path = self._get_label_path(img_path)
            if label_path.exists():
                with open(label_path, 'r') as f:
                    for line in f:
                        if line.strip():
                            class_ids.add(int(line.split()[0]))
        
        if class_ids:
            return [f'class_{i}' for i in range(max(class_ids) + 1)]
        
        return ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
    
    def _build_transforms(self) -> A.Compose:
        """Build augmentation pipeline."""
        if self.augment:
            transform = A.Compose([
                # Geometric transforms
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(
                    translate_percent={'x': (-0.1, 0.1), 'y': (-0.1, 0.1)},
                    scale=(0.8, 1.2),
                    rotate=(-15, 15),
                    border_mode=cv2.BORDER_CONSTANT,
                    fill=0,
                    p=0.5
                ),
                
                # Color transforms
                A.RandomBrightnessContrast(
                    brightness_limit=0.2,
                    contrast_limit=0.2,
                    p=0.5
                ),
                A.HueSaturationValue(
                    hue_shift_limit=10,
                    sat_shift_limit=20,
                    val_shift_limit=20,
                    p=0.3
                ),
                
                # Noise and blur
                A.GaussNoise(p=0.3),
                A.GaussianBlur(blur_limit=3, p=0.2),
                
                # Cutout (CoarseDropout) forces model to look at whole object
                A.CoarseDropout(
                    num_holes_range=(1, 8),
                    hole_height_range=(8, 32),
                    hole_width_range=(8, 32),
                    fill=0, 
                    p=0.3
                ),
                
                # Resize
                A.LongestMaxSize(max_size=self.img_size),
                A.PadIfNeeded(
                    min_height=self.img_size,
                    min_width=self.img_size,
                    border_mode=cv2.BORDER_CONSTANT,
                    fill=0
                ),
                
                # Normalize
                A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255.0),
                ToTensorV2()
            ], additional_targets={'depth': 'image', 'illumination': 'image'},
            bbox_params=A.BboxParams(
                format='yolo',
                min_visibility=0.1,
                label_fields=['class_labels'],
                min_area=1  # Filter out tiny boxes
            ))
        else:
            transform = A.Compose([
                A.LongestMaxSize(max_size=self.img_size),
                A.PadIfNeeded(
                    min_height=self.img_size,
                    min_width=self.img_size,
                    border_mode=cv2.BORDER_CONSTANT,
                    fill=0
                ),
                A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255.0),
                ToTensorV2()
            ], additional_targets={'depth': 'image', 'illumination': 'image'},
            bbox_params=A.BboxParams(
                format='yolo',
                min_visibility=0.1,
                label_fields=['class_labels'],
                min_area=1
            ))
        
        return transform
    
    def _cache_data(self):
        """Cache all data in memory."""
        print(f"[{self.split.upper()}] Caching data...")
        for i, img_path in enumerate(self.image_files):
            self.cached_data[i] = self._load_data(img_path)
            if (i + 1) % 100 == 0:
                print(f"  Cached {i + 1}/{len(self.image_files)}")
    
    def _load_data(self, img_path: Path) -> Dict[str, Any]:
        """Load all data for a single sample."""
        # Load RGB image
        image = cv2.imread(str(img_path))
        if image is None:
            raise ValueError(f"Could not load image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        # Load label
        label_path = self._get_label_path(img_path)
        bboxes, class_labels = self._load_labels(label_path, (h, w))
        
        # Load depth
        depth_path = self._get_depth_path(img_path)
        depth = self._load_npy(depth_path, 'depth')
        
        # Load illumination
        illum_path = self._get_illumination_path(img_path)
        illumination = self._load_npy(illum_path, 'illumination')
        
        return {
            'image': image,
            'bboxes': bboxes,
            'class_labels': class_labels,
            'depth': depth,
            'illumination': illumination,
            'image_path': str(img_path),
            'original_shape': (h, w)
        }
    
    def _get_label_path(self, img_path: Path) -> Path:
        """Get corresponding label path."""
        return self.labels_dir / f"{img_path.stem}.txt"
    
    def _get_depth_path(self, img_path: Path) -> Path:
        """Get corresponding depth path."""
        return self.depth_dir / f"{img_path.stem}.npy"
    
    def _get_illumination_path(self, img_path: Path) -> Path:
        """Get corresponding illumination path."""
        return self.illum_dir / f"{img_path.stem}.npy"
    
    def _load_labels(self, label_path: Path, img_shape: Tuple[int, int]) -> Tuple[List, List]:
        """Load YOLO format labels."""
        bboxes = []
        class_labels = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        parts = line.split()
                        if len(parts) >= 5:
                            class_id = int(parts[0])
                            x_center = float(parts[1])
                            y_center = float(parts[2])
                            width = float(parts[3])
                            height = float(parts[4])
                            
                            # Clamp values to valid range [0, 1] with small epsilon
                            eps = 1e-6
                            x_center = max(eps, min(1.0 - eps, x_center))
                            y_center = max(eps, min(1.0 - eps, y_center))
                            width = max(eps, min(1.0 - eps, width))
                            height = max(eps, min(1.0 - eps, height))
                            
                            bboxes.append([x_center, y_center, width, height])
                            class_labels.append(class_id)
        
        return bboxes, class_labels
    
    def _load_npy(self, npy_path: Path, data_type: str) -> np.ndarray:
        """Load numpy file — handles uint8 (compact) or float32 (legacy) format."""
        if npy_path.exists():
            data = np.load(str(npy_path))
            
            # Convert uint8 [0,255] -> float32 [0,1]
            if data.dtype == np.uint8:
                data = data.astype(np.float32) / 255.0
            else:
                data = data.astype(np.float32)
            
            # Handle different formats
            if data_type == 'depth':
                # Ensure (H, W, 1)
                if data.ndim == 2:
                    data = data[:, :, np.newaxis]  # (H, W) -> (H, W, 1)
                elif data.ndim == 3:
                    if data.shape[0] == 1:
                        data = np.transpose(data, (1, 2, 0))  # (1, H, W) -> (H, W, 1)
                    elif data.shape[2] > 1:
                        data = data[:, :, :1]
                # Re-normalize in case of float32 legacy files
                if data.max() > 1.0:
                    data = data / data.max()
                    
            elif data_type == 'illumination':
                # Ensure (H, W, 3)
                if data.ndim == 2:
                    data = np.stack([data] * 3, axis=-1)
                elif data.ndim == 3:
                    if data.shape[0] == 3:
                        data = np.transpose(data, (1, 2, 0))
                    elif data.shape[2] < 3:
                        data = np.concatenate([data] * 3, axis=-1)[:, :, :3]
                if data.max() > 1.0:
                    data = data / data.max()
            
            return data.astype(np.float32)
        
        else:
            # Return zeros if file doesn't exist
            if data_type == 'depth':
                return np.zeros((self.img_size, self.img_size, 1), dtype=np.float32)
            else:
                return np.zeros((self.img_size, self.img_size, 3), dtype=np.float32)
    
    def __len__(self) -> int:
        return len(self.image_files)
    
    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        """Get a single sample."""
        # Load or get cached data
        if self.cache:
            data = self.cached_data[index]
        else:
            data = self._load_data(self.image_files[index])
        
        image = data['image'].copy()
        bboxes = list(data['bboxes'])
        class_labels = list(data['class_labels'])
        depth = data['depth'].copy()
        illumination = data['illumination'].copy()
        
        # FIX: Ensure depth and illumination are exactly the same size as RGB BEFORE any processing
        H_rgb, W_rgb = image.shape[:2]
        if depth.shape[:2] != (H_rgb, W_rgb):
            depth = cv2.resize(depth, (W_rgb, H_rgb))
            if depth.ndim == 2:
                depth = np.expand_dims(depth, axis=-1)
        if illumination.shape[:2] != (H_rgb, W_rgb):
            illumination = cv2.resize(illumination, (W_rgb, H_rgb))
            if illumination.ndim == 2:
                illumination = np.stack([illumination]*3, axis=-1)
        
        # ====================================================================
        # PHASE 7: TARGETED COPY-PASTE FOR 3D MINORITY CLASSES
        # ====================================================================
        # Aggressively combat the dataset imbalance by hunting down Spike, 
        # Excess, and Insufficient solder topologies and physically multiplying 
        # their occurrences across random background patches, 35% of the time.
        if self.augment and getattr(self, 'minority_patches', None) and len(self.minority_patches) > 0:
            if random.random() < 0.35:
                # Pick 1 to 3 random defect patches
                num_pastes = random.randint(1, 3)
                for _ in range(num_pastes):
                    patch_data = random.choice(self.minority_patches)
                    patch_img = patch_data['image']
                    patch_depth = patch_data['depth']
                    patch_illum = patch_data['illumination']
                    d_lbl = patch_data['label']
                    
                    ph, pw = patch_img.shape[:2]
                    tH, tW = image.shape[:2]
                    
                    if tW > pw and tH > ph:
                        paste_x = random.randint(0, tW - pw)
                        paste_y = random.randint(0, tH - ph)
                        
                        # Hard-paste directly into the target modal arrays
                        image[paste_y:paste_y+ph, paste_x:paste_x+pw] = patch_img
                        depth[paste_y:paste_y+ph, paste_x:paste_x+pw] = patch_depth
                        illumination[paste_y:paste_y+ph, paste_x:paste_x+pw] = patch_illum
                        
                        # Calculate new target box coordinates
                        new_x_c = (paste_x + pw/2) / tW
                        new_y_c = (paste_y + ph/2) / tH
                        
                        bboxes.append([new_x_c, new_y_c, pw/tW, ph/tH])
                        class_labels.append(d_lbl)
        # ====================================================================

        # Sanitize bboxes BEFORE transform
        # Must be strictly within [0, 1] range with safe margins
        eps = 1e-3  # Use larger epsilon for more safety margin
        valid_bboxes = []
        valid_labels = []
        
        for i, bbox in enumerate(bboxes):
            x_c = float(bbox[0])
            y_c = float(bbox[1])
            w = float(bbox[2])
            h = float(bbox[3])
            
            # Clamp to valid range
            x_c = max(eps, min(1.0 - eps, x_c))
            y_c = max(eps, min(1.0 - eps, y_c))
            w = max(eps, min(1.0 - eps, w))
            h = max(eps, min(1.0 - eps, h))
            
            # Ensure box doesn't extend beyond image
            half_w = w / 2
            half_h = h / 2
            x_c = max(half_w + eps, min(1.0 - half_w - eps, x_c))
            y_c = max(half_h + eps, min(1.0 - half_h - eps, y_c))
            
            # Only keep valid boxes
            if w > eps and h > eps and x_c > 0 and y_c > 0:
                valid_bboxes.append([x_c, y_c, w, h])
                if i < len(class_labels):
                    valid_labels.append(class_labels[i])
        
        bboxes = valid_bboxes
        class_labels = valid_labels
        
        # Apply transforms with error handling
        try:
            if len(bboxes) > 0:
                transformed = self.transform(
                    image=image,
                    depth=depth,
                    illumination=illumination,
                    bboxes=bboxes,
                    class_labels=class_labels
                )
            else:
                transformed = self.transform(
                    image=image,
                    depth=depth,
                    illumination=illumination,
                    bboxes=[],
                    class_labels=[]
                )
        except (ValueError, RuntimeError) as e:
            # Fallback: apply only resize and normalize without augmentation
            import albumentations as A
            fallback_transform = A.Compose([
                A.LongestMaxSize(max_size=self.img_size),
                A.PadIfNeeded(
                    min_height=self.img_size,
                    min_width=self.img_size,
                    border_mode=cv2.BORDER_CONSTANT,
                    fill=0
                ),
                A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255.0),
                ToTensorV2()
            ], additional_targets={'depth': 'image', 'illumination': 'image'},
            bbox_params=A.BboxParams(
                format='yolo',
                min_visibility=0.0,
                label_fields=['class_labels']
            ))
            
            if len(bboxes) > 0:
                transformed = fallback_transform(
                    image=image,
                    depth=depth,
                    illumination=illumination,
                    bboxes=bboxes,
                    class_labels=class_labels
                )
            else:
                transformed = fallback_transform(
                    image=image,
                    depth=depth,
                    illumination=illumination,
                    bboxes=[],
                    class_labels=[]
                )
        
        image_tensor = transformed['image']
        depth_tensor = transformed['depth']
        illum_tensor = transformed['illumination']
        bboxes_transformed = list(transformed.get('bboxes', []))
        class_labels_transformed = list(transformed.get('class_labels', []))
        
        # Final sanitization after transforms
        final_bboxes = []
        final_labels = []
        for i, bbox in enumerate(bboxes_transformed):
            x_c = max(eps, min(1.0 - eps, float(bbox[0])))
            y_c = max(eps, min(1.0 - eps, float(bbox[1])))
            w = max(eps, min(1.0 - eps, float(bbox[2])))
            h = max(eps, min(1.0 - eps, float(bbox[3])))
            
            if w > eps and h > eps:
                final_bboxes.append([x_c, y_c, w, h])
                if i < len(class_labels_transformed):
                    final_labels.append(class_labels_transformed[i])
        
        bboxes_transformed = final_bboxes
        class_labels_transformed = final_labels
        
        # Transform depth and illumination (Convert Albumentations outputs to C,H,W tensors)
        # Depth might be (H,W) or (H,W,1), ensure it's (1,H,W)
        if isinstance(depth_tensor, torch.Tensor):
            depth = depth_tensor
        else:
            depth = torch.from_numpy(depth_tensor)
            
        if depth.ndim == 2:
            depth = depth.unsqueeze(0)
        elif depth.ndim == 3:
            if depth.shape[0] != 1 and depth.shape[2] == 1:
                depth = depth.permute(2, 0, 1)
                
        # Illumination might be (H,W,3), ensure it's (3,H,W)
        if isinstance(illum_tensor, torch.Tensor):
            illumination = illum_tensor
        else:
            illumination = torch.from_numpy(illum_tensor)
            
        if illumination.ndim == 3 and illumination.shape[2] == 3:
            illumination = illumination.permute(2, 0, 1)
        elif illumination.ndim == 2:
            illumination = illumination.unsqueeze(0).repeat(3, 1, 1)
        
        # Create target tensor
        num_boxes = len(bboxes_transformed)
        targets = torch.zeros(num_boxes, 6)  # [batch_idx, class, x, y, w, h]
        
        if num_boxes > 0:
            for i, (bbox, cls) in enumerate(zip(bboxes_transformed, class_labels_transformed)):
                targets[i, 0] = 0  # batch index (will be set in collate)
                targets[i, 1] = cls
                targets[i, 2] = bbox[0]  # x_center
                targets[i, 3] = bbox[1]  # y_center
                targets[i, 4] = bbox[2]  # width
                targets[i, 5] = bbox[3]  # height
        
        return {
            'image': image_tensor.float(),
            'depth': depth.float(),
            'illumination': illumination.float(),
            'targets': targets,
            'image_path': data['image_path'],
            'num_boxes': num_boxes
        }


def collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """
    Custom collate function for batching.
    Handles variable number of boxes per image.
    """
    images = []
    depths = []
    illuminations = []
    targets = []
    image_paths = []
    
    for i, item in enumerate(batch):
        images.append(item['image'])
        depths.append(item['depth'])
        illuminations.append(item['illumination'])
        
        # Update batch index in targets
        t = item['targets'].clone()
        if t.shape[0] > 0:
            t[:, 0] = i
        targets.append(t)
        
        image_paths.append(item['image_path'])
    
    return {
        'images': torch.stack(images),
        'depths': torch.stack(depths),
        'illuminations': torch.stack(illuminations),
        'targets': torch.cat(targets, dim=0),
        'image_paths': image_paths
    }


def create_dataloaders(
    data_dir: str,
    batch_size: int = 8,
    img_size: int = 640,
    num_workers: int = 4,
    cache: bool = False,
    classes: Optional[List[str]] = None
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train, validation, and test dataloaders.
    
    Args:
        data_dir: Root data directory
        batch_size: Batch size
        img_size: Image size
        num_workers: Number of workers for data loading
        cache: Cache images in memory
        classes: List of class names
    
    Returns:
        train_loader, val_loader, test_loader
    """
    # Create datasets
    train_dataset = PCBDataset(
        data_dir=data_dir,
        split='train',
        img_size=img_size,
        augment=True,
        cache=cache,
        classes=classes
    )
    
    val_dataset = PCBDataset(
        data_dir=data_dir,
        split='valid',
        img_size=img_size,
        augment=False,
        cache=cache,
        classes=train_dataset.classes
    )
    
    test_dataset = PCBDataset(
        data_dir=data_dir,
        split='test',
        img_size=img_size,
        augment=False,
        cache=cache,
        classes=train_dataset.classes
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True if num_workers > 0 else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader


class DataHandler:
    """
    Data Handler class for managing datasets and data loading.
    Provides a unified interface for training and evaluation.
    """
    
    def __init__(
        self,
        data_dir: str,
        batch_size: int = 8,
        img_size: int = 640,
        num_workers: int = 4,
        cache: bool = False
    ):
        """
        Initialize DataHandler.
        
        Args:
            data_dir: Root data directory
            batch_size: Batch size
            img_size: Image size
            num_workers: Number of workers
            cache: Cache images in memory
        """
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.num_workers = num_workers
        
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
        self.classes = None
        self.num_classes = None
        
        self._setup()
    
    def _setup(self):
        """Setup datasets and dataloaders."""
        print("\n" + "="*60)
        print("Setting up Data Handler")
        print("="*60)
        
        # Create dataloaders
        self.train_loader, self.val_loader, self.test_loader = create_dataloaders(
            data_dir=self.data_dir,
            batch_size=self.batch_size,
            img_size=self.img_size,
            num_workers=self.num_workers
        )
        
        # Get class info
        self.classes = self.train_loader.dataset.classes
        self.num_classes = self.train_loader.dataset.num_classes
        
        print(f"\nData Summary:")
        print(f"  Train batches: {len(self.train_loader)}")
        print(f"  Val batches: {len(self.val_loader)}")
        print(f"  Test batches: {len(self.test_loader)}")
        print(f"  Classes ({self.num_classes}): {self.classes}")
        print("="*60 + "\n")
    
    def get_sample_batch(self) -> Dict[str, torch.Tensor]:
        """Get a sample batch for visualization or testing."""
        for batch in self.train_loader:
            return batch
    
    def get_class_weights(self) -> torch.Tensor:
        """Calculate class weights for balanced training."""
        class_counts = torch.zeros(self.num_classes)
        
        for batch in self.train_loader:
            targets = batch['targets']
            if targets.shape[0] > 0:
                classes = targets[:, 1].long()
                for c in classes:
                    class_counts[c] += 1
        
        # Inverse frequency weighting
        class_weights = 1.0 / (class_counts + 1e-6)
        class_weights = class_weights / class_weights.sum() * self.num_classes
        
        return class_weights


if __name__ == "__main__":
    # Test the dataset
    import argparse
    
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', type=str, required=True, help='Data directory')
    parser.add_argument('--batch', type=int, default=4, help='Batch size')
    parser.add_argument('--imgsz', type=int, default=640, help='Image size')
    args = parser.parse_args()
    
    # Create handler
    handler = DataHandler(
        data_dir=args.data,
        batch_size=args.batch,
        img_size=args.imgsz
    )
    
    # Test batch
    batch = handler.get_sample_batch()
    
    print("\nSample Batch:")
    print(f"  Images shape: {batch['images'].shape}")
    print(f"  Depths shape: {batch['depths'].shape}")
    print(f"  Illuminations shape: {batch['illuminations'].shape}")
    print(f"  Targets shape: {batch['targets'].shape}")
    print(f"  Image paths: {batch['image_paths'][:2]}...")



### utils/general.py
Definition and code for `utils/general.py`.


In [ ]:
%%writefile utils/general.py
"""
General Utilities for YOLOv5-3D
File operations, logging, and helper functions.
"""

import os
import sys
import time
import logging
import random
import math
import re
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Union
from functools import lru_cache

import numpy as np
import torch
import torch.nn as nn


# ANSI color codes
COLORS = {
    'red': '\033[91m',
    'green': '\033[92m',
    'yellow': '\033[93m',
    'blue': '\033[94m',
    'magenta': '\033[95m',
    'cyan': '\033[96m',
    'white': '\033[97m',
    'reset': '\033[0m',
}


def colorstr(*input_str):
    """
    Colorize string.
    
    Usage:
        print(colorstr('blue', 'hello world'))
        print(colorstr('red', 'bold', 'error message'))
    """
    *args, string = input_str if len(input_str) > 1 else ('green', input_str[0])
    colors = {'red', 'green', 'yellow', 'blue', 'magenta', 'cyan', 'white'}
    style = ''.join([COLORS.get(arg, '') for arg in args if arg in colors])
    return f'{style}{string}{COLORS["reset"]}'


def set_logging(verbose=True):
    """Setup logging."""
    level = logging.DEBUG if verbose else logging.INFO
    logging.basicConfig(
        level=level,
        format='%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )


@lru_cache(maxsize=None)
def get_user_config_dir():
    """Get user config directory."""
    if sys.platform == 'win32':
        return Path(os.environ.get('APPDATA', ''))
    elif sys.platform == 'darwin':
        return Path.home() / 'Library' / 'Application Support'
    else:
        return Path.home() / '.config'


def increment_path(path: Path, exist_ok: bool = False, sep: str = '') -> Path:
    """
    Increment path to avoid overwriting.
    
    Args:
        path: Base path
        exist_ok: If True, don't increment
        sep: Separator between name and number
    
    Returns:
        Incremented path
    
    Example:
        runs/exp --> runs/exp{sep}0, runs/exp{sep}1, etc.
    """
    path = Path(path)
    if exist_ok or not path.exists():
        return path
    
    # Increment
    dirs = sorted([p for p in path.parent.glob(f'{path.stem}{sep}*') if p.is_dir()])
    matches = [re.match(rf'{path.stem}{sep}(\d+)', d.name) for d in dirs]
    idx = max([int(m.group(1)) for m in matches if m] + [-1]) + 1
    
    return path.parent / f'{path.stem}{sep}{idx}'


def get_latest_run(search_dir: Path):
    """Get latest run directory."""
    runs = list(search_dir.glob('exp*/'))
    return max(runs, key=lambda x: x.stat().st_mtime) if runs else ''


def one_cycle(y1: float, y2: float, steps: int) -> callable:
    """
    One cycle learning rate scheduler.
    
    Returns a function that maps [0, steps] -> [y1, y2] -> [y2, y1].
    """
    def f(x):
        return ((1 - math.cos(x * math.pi / steps)) / 2) * (y2 - y1) + y1
    return f


def check_dataset(data_dir: str) -> Dict:
    """
    Check dataset structure and return info.
    
    Args:
        data_dir: Dataset directory path
    
    Returns:
        Dataset info dict
    """
    data_dir = Path(data_dir)
    
    required_dirs = ['images', 'labels']
    optional_dirs = ['depth', 'illumination']
    
    info = {
        'path': str(data_dir),
        'valid': True,
        'splits': {},
        'missing': []
    }
    
    for split in ['train', 'valid', 'test']:
        split_dir = data_dir / split
        if split_dir.exists():
            info['splits'][split] = {}
            
            # Check required dirs
            for d in required_dirs:
                if (split_dir / d).exists():
                    files = list((split_dir / d).glob('*'))
                    info['splits'][split][d] = len(files)
                else:
                    info['splits'][split][d] = 0
                    info['missing'].append(f'{split}/{d}')
            
            # Check optional dirs
            for d in optional_dirs:
                if (split_dir / d).exists():
                    files = list((split_dir / d).glob('*.npy'))
                    info['splits'][split][d] = len(files)
                else:
                    info['splits'][split][d] = 0
        else:
            info['splits'][split] = None
    
    # Check for classes file
    if (data_dir / 'classes.txt').exists():
        with open(data_dir / 'classes.txt') as f:
            info['classes'] = [line.strip() for line in f if line.strip()]
    elif (data_dir / 'data.yaml').exists():
        import yaml
        with open(data_dir / 'data.yaml') as f:
            data = yaml.safe_load(f)
            info['classes'] = data.get('names', [])
    else:
        info['classes'] = None
    
    return info


def weights_init(model):
    """Initialize model weights."""
    for m in model.modules():
        t = type(m)
        if t is nn.Conv2d:
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif t is nn.BatchNorm2d:
            m.eps = 1e-3
            m.momentum = 0.03
        elif t in [nn.Hardswish, nn.LeakyReLU, nn.ReLU, nn.ReLU6, nn.SiLU]:
            m.inplace = True


def time_sync():
    """Get synchronized time."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.time()


def profile(input, ops, n=10, device=None):
    """
    Profile model layers.
    
    Args:
        input: Input tensor
        ops: List of operations
        n: Number of iterations
        device: Device to use
    """
    print(f"{'Params':>12s}{'GFLOPs':>12s}{'GPU_mem (GB)':>14s}{'forward (ms)':>14s}"
          f"{'backward (ms)':>14s}{'input':>24s}{'output':>24s}")
    
    for op in ops if isinstance(ops, list) else [ops]:
        op = op.to(device) if device else op
        
        # Params
        p = sum(x.numel() for x in op.parameters())
        
        # FLOPs
        flops = 0
        try:
            from thop import profile as thop_profile
            flops, _ = thop_profile(op, inputs=(input,), verbose=False)
        except:
            pass
        
        # GPU memory
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
        
        # Time
        start = time_sync()
        for _ in range(n):
            op(input)
        forward_time = (time_sync() - start) * 1000 / n
        
        if input.requires_grad:
            start = time_sync()
            for _ in range(n):
                input.grad = None
                op(input).sum().backward()
            backward_time = (time_sync() - start) * 1000 / n
        else:
            backward_time = 0
        
        # Memory
        mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        
        print(f'{p:12.2f}{flops/1e9:12.4f}{mem:14.4f}{forward_time:14.4f}'
              f'{backward_time:14.4f}{str(list(input.shape)):>24s}'
              f'{str(list(op(input).shape)):>24s}')


def copy_attr(a, b, include=(), exclude=()):
    """Copy attributes from b to a."""
    for k, v in b.__dict__.items():
        if (len(include) and k not in include) or k.startswith('_') or k in exclude:
            continue
        setattr(a, k, v)


def intersect_dicts(da, db, exclude=()):
    """
    Intersect dictionaries.
    
    Returns dict with matching keys and shapes, excluding 'exclude' keys.
    """
    return {k: v for k, v in da.items() 
            if k in db and v.shape == db[k].shape and not any(x in k for x in exclude)}


def de_parallel(model):
    """De-parallelize a model (returns single-GPU model)."""
    return model.module if hasattr(model, 'module') else model


def initialize_weights(model):
    """Initialize model weights."""
    for m in model.modules():
        t = type(m)
        if t is nn.Conv2d:
            pass
        elif t is nn.BatchNorm2d:
            m.eps = 1e-3
            m.momentum = 0.03
        elif t in [nn.Hardswish, nn.LeakyReLU, nn.ReLU, nn.ReLU6, nn.SiLU]:
            m.inplace = True


def make_divisible(x, divisor):
    """Returns nearest x divisible by divisor."""
    if isinstance(divisor, torch.Tensor):
        divisor = int(divisor.max())
    return math.ceil(x / divisor) * divisor


def check_file(file: str):
    """Check if file exists and return path."""
    if Path(file).is_file() or file == '':
        return file
    else:
        files = list(Path('').rglob(file))
        if not files:
            raise FileNotFoundError(f'File not found: {file}')
        if len(files) > 1:
            warnings.warn(f'Multiple files found: {files}')
        return files[0]


def check_img_size(imgsz, s=32):
    """
    Verify image size is multiple of stride s.
    
    Args:
        imgsz: Image size (int or list)
        s: Stride
    
    Returns:
        New size
    """
    if isinstance(imgsz, int):
        new_size = max(make_divisible(imgsz, int(s)), s)
        if new_size != imgsz:
            warnings.warn(f'Image size {imgsz} not divisible by stride {s}. Using {new_size}')
        return new_size
    else:
        return [check_img_size(x, s) for x in imgsz]


def clip_coords(boxes, shape):
    """
    Clip bounding box coordinates to image shape.
    
    Args:
        boxes: Boxes (n, 4) [x1, y1, x2, y2]
        shape: Image shape (h, w)
    """
    if isinstance(boxes, torch.Tensor):
        boxes[:, 0].clamp_(0, shape[1])  # x1
        boxes[:, 1].clamp_(0, shape[0])  # y1
        boxes[:, 2].clamp_(0, shape[1])  # x2
        boxes[:, 3].clamp_(0, shape[0])  # y2
    else:
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, shape[1])
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, shape[0])


def scale_coords(img1_shape, coords, img0_shape, ratio_pad=None):
    """
    Rescale coords from img1_shape to img0_shape.
    
    Args:
        img1_shape: Model input shape (h, w)
        coords: Coordinates to rescale
        img0_shape: Original image shape
        ratio_pad: Padding ratio
    """
    if ratio_pad is None:
        gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])
        pad = (img1_shape[1] - img0_shape[1] * gain) / 2, \
              (img1_shape[0] - img0_shape[0] * gain) / 2
    else:
        gain = ratio_pad[0][0]
        pad = ratio_pad[1]
    
    coords[:, [0, 2]] -= pad[0]
    coords[:, [1, 3]] -= pad[1]
    coords[:, :4] /= gain
    clip_coords(coords, img0_shape)
    return coords


def xyxy2xywh(x):
    """Convert boxes from xyxy to xywh format."""
    y = x.clone() if isinstance(x, torch.Tensor) else x.copy()
    y[..., 0] = (x[..., 0] + x[..., 2]) / 2  # x center
    y[..., 1] = (x[..., 1] + x[..., 3]) / 2  # y center
    y[..., 2] = x[..., 2] - x[..., 0]  # width
    y[..., 3] = x[..., 3] - x[..., 1]  # height
    return y


def xywh2xyxy(x):
    """Convert boxes from xywh to xyxy format."""
    y = x.clone() if isinstance(x, torch.Tensor) else x.copy()
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # x1
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # y1
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # x2
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # y2
    return y


def xywhn2xyxy(x, w=640, h=640, padw=0, padh=0):
    """Convert normalized xywh to pixel xyxy format."""
    y = x.clone() if isinstance(x, torch.Tensor) else x.copy()
    y[..., 0] = w * (x[..., 0] - x[..., 2] / 2) + padw  # x1
    y[..., 1] = h * (x[..., 1] - x[..., 3] / 2) + padh  # y1
    y[..., 2] = w * (x[..., 0] + x[..., 2] / 2) + padw  # x2
    y[..., 3] = h * (x[..., 1] + x[..., 3] / 2) + padh  # y2
    return y


def xyn2xy(x, w=640, h=640, padw=0, padh=0):
    """Convert normalized segments to pixel segments."""
    y = x.clone() if isinstance(x, torch.Tensor) else x.copy()
    y[..., 0] = w * x[..., 0] + padw  # x
    y[..., 1] = h * x[..., 1] + padh  # y
    return y


def segment2box(segment, width=640, height=640):
    """Convert segment to bounding box."""
    x, y = segment.T
    inside = (x >= 0) & (y >= 0) & (x <= width) & (y <= height)
    x, y = x[inside], y[inside]
    return np.array([x.min(), y.min(), x.max(), y.max()]) if any(x) else np.zeros(4)


def segments2boxes(segments):
    """Convert segments to bounding boxes."""
    boxes = []
    for s in segments:
        x, y = s.T
        boxes.append([x.min(), y.min(), x.max(), y.max()])
    return np.array(boxes)


def resample_segments(segments, n=1000):
    """Resample segments to n points each."""
    for i, s in enumerate(segments):
        x = np.linspace(0, len(s) - 1, n)
        xp = np.arange(len(s))
        segments[i] = np.concatenate(
            [np.interp(x, xp, s[:, i]) for i in range(2)]
        ).reshape(2, -1).T
    return segments


def clean_str(s):
    """Clean string by replacing special characters with underscore."""
    return re.sub(pattern='[|@#!¡·$€%&()=?¿^*;:,¨´><+]', repl='_', string=s)


def print_mutation(key, results, hyp, save_dir):
    """Print mutation results."""
    hyp = hyp.copy()
    hyp = {k: round(v, 5) for k, v in hyp.items()}
    keys = ('metrics/mAP50-95', 'metrics/mAP50', 'metrics/precision', 
            'metrics/recall', 'val/box_loss', 'val/obj_loss', 'val/cls_loss')
    
    print(f'{key}: {results}\n')
    with open(save_dir / 'hyp_evolve.yaml', 'a') as f:
        f.write(f'{key}: {results}\n')
        f.write(f'hyp: {hyp}\n')


def select_device(device: str = '', batch_size: int = 0) -> torch.device:
    """
    Select device for training/inference.
    
    Args:
        device: Device string ('cpu', 'cuda', 'cuda:0', etc.)
        batch_size: Batch size (used for multi-GPU check)
    
    Returns:
        torch.device
    """
    # Parse device string
    if device.lower() == 'cpu':
        return torch.device('cpu')
    
    elif device.lower().startswith('cuda'):
        if not torch.cuda.is_available():
            print(f"{colorstr('yellow')}CUDA not available, using CPU{colorstr('reset')}")
            return torch.device('cpu')
        
        # Get device index
        if ':' in device:
            device_idx = int(device.split(':')[1])
        else:
            device_idx = 0
        
        if device_idx >= torch.cuda.device_count():
            print(f"{colorstr('yellow')}CUDA device {device_idx} not available, using cuda:0{colorstr('reset')}")
            device_idx = 0
        
        # Print device info
        gpu_name = torch.cuda.get_device_name(device_idx)
        gpu_mem = torch.cuda.get_device_properties(device_idx).total_memory / 1e9
        print(f"{colorstr('green')}Using CUDA device {device_idx}: {gpu_name} ({gpu_mem:.1f} GB){colorstr('reset')}")
        
        return torch.device(f'cuda:{device_idx}')
    
    else:
        # Auto-detect
        if torch.cuda.is_available():
            return torch.device('cuda:0')
        return torch.device('cpu')


class EarlyStopping:
    """Early stopping class to stop training when validation mAP stops improving."""
    
    def __init__(self, patience=30):
        """
        Initialize early stopping.
        
        Args:
            patience: Number of epochs to wait before stopping
        """
        self.best_fitness = 0.0
        self.best_epoch = 0
        self.patience = patience
        self.possible_stop = False
    
    def __call__(self, epoch, fitness):
        """
        Check if training should stop.
        
        Args:
            epoch: Current epoch
            fitness: Current fitness (mAP)
        
        Returns:
            True if training should stop
        """
        if fitness is None:
            return False
        
        if fitness > self.best_fitness:
            self.best_fitness = fitness
            self.best_epoch = epoch
        
        self.possible_stop = epoch - self.best_epoch >= self.patience
        
        return epoch - self.best_epoch >= self.patience



### utils/loss.py
Definition and code for `utils/loss.py`.


In [ ]:
%%writefile utils/loss.py
"""
Loss Functions for YOLOv5-3D
Implements CIoU loss, objectness loss, and classification loss.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def bbox_iou(box1, box2, xywh=True, GIoU=False, DIoU=False, CIoU=False, eps=1e-7):
    """
    Calculate IoU of box1 and box2.
    
    Args:
        box1: Tensor of shape (N, 4) or (4,)
        box2: Tensor of shape (M, 4) or (4,)
        xywh: Whether boxes are in xywh format
        GIoU: Use GIoU loss
        DIoU: Use DIoU loss
        CIoU: Use CIoU loss
        eps: Small constant for numerical stability
    
    Returns:
        IoU value
    """
    # Transform xywh to xyxy
    if xywh:
        x1, y1, w1, h1 = box1.chunk(4, dim=-1)
        x2, y2, w2, h2 = box2.chunk(4, dim=-1)
        
        b1_x1, b1_x2 = x1 - w1 / 2, x1 + w1 / 2
        b1_y1, b1_y2 = y1 - h1 / 2, y1 + h1 / 2
        b2_x1, b2_x2 = x2 - w2 / 2, x2 + w2 / 2
        b2_y1, b2_y2 = y2 - h2 / 2, y2 + h2 / 2
    else:
        b1_x1, b1_y1, b1_x2, b1_y2 = box1.chunk(4, dim=-1)
        b2_x1, b2_y1, b2_x2, b2_y2 = box2.chunk(4, dim=-1)
        w1, h1 = b1_x2 - b1_x1, b1_y2 - b1_y1 + eps
        w2, h2 = b2_x2 - b2_x1, b2_y2 - b2_y1 + eps
    
    # Intersection area
    inter = (b1_x2.minimum(b2_x2) - b1_x1.maximum(b2_x1)).clamp_(0) * \
            (b1_y2.minimum(b2_y2) - b1_y1.maximum(b2_y1)).clamp_(0)
    
    # Union area
    union = w1 * h1 + w2 * h2 - inter + eps
    
    # IoU
    iou = inter / union
    
    if CIoU or DIoU or GIoU:
        cw = b1_x2.maximum(b2_x2) - b1_x1.minimum(b2_x1)  # convex width
        ch = b1_y2.maximum(b2_y2) - b1_y1.minimum(b2_y1)  # convex height
        
        if CIoU or DIoU:
            c2 = cw ** 2 + ch ** 2 + eps  # convex diagonal squared
            rho2 = ((b2_x1 + b2_x2 - b1_x1 - b1_x2) ** 2 + 
                    (b2_y1 + b2_y2 - b1_y1 - b1_y2) ** 2) / 4  # center dist squared
            
            if CIoU:
                v = (4 / (math.pi ** 2)) * (torch.atan(w2 / (h2 + eps)) - torch.atan(w1 / (h1 + eps))) ** 2
                with torch.no_grad():
                    alpha = v / (1 - iou + v + eps)
                return iou - (rho2 / c2 + v * alpha)  # CIoU
            return iou - rho2 / c2  # DIoU
        
        # GIoU
        c_area = cw * ch + eps
        return iou - (c_area - union) / c_area  # GIoU
    
    return iou


def smooth_BCE(eps=0.1):
    """
    Compute smoothed BCE targets.
    
    Returns positive and negative label smoothing BCE targets.
    """
    return 1.0 - 0.5 * eps, 0.5 * eps


class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance."""
    
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, pred, target):
        """Calculate focal loss."""
        bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ComputeLoss:
    """
    Compute Loss for YOLOv5-3D training.
    
    Combines:
    - Box loss (CIoU)
    - Objectness loss (BCE)
    - Classification loss (BCE with optional focal)
    """
    
    def __init__(self, model, hyp=None, class_weights=None):
        """
        Initialize loss computation.
        
        Args:
            model: YOLOv5-3D model
            hyp: Hyperparameters dict
            class_weights: Optional tensor of weights per class
        """
        device = next(model.parameters()).device
        
        # Default hyperparameters
        self.hyp = hyp or {
            'box': 0.05,          # box loss gain
            'cls': 0.5,           # cls loss gain
            'obj': 1.0,           # obj loss gain
            'cls_pw': 1.0,        # cls BCELoss positive_weight
            'obj_pw': 1.0,        # obj BCELoss positive_weight
            'fl_gamma': 0.0,      # focal loss gamma (0 = disabled)
            'anchor_t': 4.0,      # anchor-multiple threshold
            'label_smoothing': 0.0,
        }
        
        # Get detection layer
        m = model.model[-1] if hasattr(model, 'model') else model.detect
        self.device = device
        
        # Number of classes and outputs
        self.nc = m.nc
        self.nl = m.nl
        self.na = m.na
        self.no = m.no
        
        # Anchors
        self.anchors = m.anchors.to(device)
        self.stride = m.stride.to(device)
        
        reduction = 'none' if class_weights is not None else 'mean'
        # Define loss functions
        BCEcls = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.hyp['cls_pw']], device=device), reduction=reduction)
        BCEobj = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.hyp['obj_pw']], device=device))
        
        # Focal loss
        g = self.hyp['fl_gamma']
        if g > 0:
            BCEcls = FocalLoss(alpha=0.25, gamma=g, reduction=reduction)
            BCEobj = FocalLoss(alpha=0.25, gamma=g)
        
        self.BCEcls = BCEcls
        self.BCEobj = BCEobj
        self.class_weights = class_weights
        
        # Label smoothing
        self.cp, self.cn = smooth_BCE(eps=self.hyp['label_smoothing'])
        
        # Balance for different output scales
        self.balance = [4.0, 1.0, 0.4]
        
        # Metrics
        self.loss_items = None
    
    def __call__(self, preds, targets):
        """
        Compute total loss.
        
        Args:
            preds: Model predictions (list of outputs at different scales)
                   Each output can be:
                   - 5D: (batch, anchors, h, w, 5+nc) - training format
                   - 3D: (batch, anchors*h*w, 5+nc) - flattened format
            targets: Ground truth targets (N, 6) [batch_idx, class, x, y, w, h]
        
        Returns:
            total_loss, loss_items (box, obj, cls)
        """
        device = preds[0].device if isinstance(preds, (list, tuple)) else preds.device
        lcls = torch.zeros(1, device=device)
        lbox = torch.zeros(1, device=device)
        lobj = torch.zeros(1, device=device)
        
        # Build targets
        tcls, tbox, indices, anchors = self.build_targets(preds, targets)
        
        # Losses
        for i, pi in enumerate(preds):  # layer index, layer predictions
            b, a, gj, gi = indices[i]  # image, anchor, gridy, gridx
            tobj = torch.zeros_like(pi[..., 0])  # target obj (matches last dim of pi)
            
            n = b.shape[0]  # number of targets
            if n:
                # Get prediction subset based on output format
                if len(pi.shape) == 5:
                    # 5D: (batch, anchors, h, w, 5+nc)
                    pred_subset = pi[b, a, gj, gi]
                else:
                    # 3D: (batch, anchors*h*w, 5+nc) - need to compute flat index
                    # Flatten anchor and grid indices
                    flat_idx = a * (pi.shape[1] // self.na) + gj * gi  # simplified
                    # Alternative: use the indices directly
                    pred_subset = pi[b]  # (n, num_predictions, 5+nc)
                    # Get specific predictions
                    pred_subset = pi[b, flat_idx] if len(flat_idx.shape) > 0 else pi[b].squeeze(0)
                
                # Decode predictions to match target space (standard YOLOv5 formula)
                pxy = pred_subset[:, :2].sigmoid() * 2 - 0.5   # [-0.5, 1.5] grid offset
                pwh = (pred_subset[:, 2:4].sigmoid() * 2) ** 2 * anchors[i]  # grid-space wh
                pbox = torch.cat((pxy, pwh), 1)  # predicted box in grid-space
                
                # IoU loss
                iou = bbox_iou(pbox, tbox[i], CIoU=True).squeeze()
                lbox += (1.0 - iou).mean()
                
                # Objectness target
                iou = iou.detach().clamp(0).type(tobj.dtype)
                if len(pi.shape) == 5:
                    tobj[b, a, gj, gi] = iou  # IoU as target
                else:
                    # For 3D output, set objectness at flat indices
                    tobj[b, flat_idx] = iou
                
                # Classification
                if self.nc > 1:
                    t = torch.full_like(pred_subset[:, 5:], self.cn, device=device)
                    t[range(n), tcls[i]] = self.cp
                    
                    # Compute unreduced BCE loss
                    bce_loss = self.BCEcls(pred_subset[:, 5:], t)
                    
                    # Apply class weights if provided
                    if self.class_weights is not None:
                        # Extract weights for the matched target classes
                        c_weights = self.class_weights[tcls[i]]
                        # For BCEWithLogitsLoss output (N, nc), we weigh each row by its target class weight
                        # Note: This is an approximation since BCE considers all classes, but we emphasize
                        # the correct class row weight.
                        bce_loss = bce_loss.mean(1) * c_weights
                        lcls += bce_loss.mean()
                    else:
                        lcls += bce_loss.mean() if len(bce_loss.shape) > 0 else bce_loss
            
            # Objectness loss
            obji = self.BCEobj(pi[..., 4], tobj)
            lobj += obji * self.balance[i]
        
        # Apply gains
        lbox *= self.hyp['box']
        lobj *= self.hyp['obj']
        lcls *= self.hyp['cls']
        
        # Total loss
        loss = lbox + lobj + lcls
        
        # Loss items for logging
        self.loss_items = torch.cat((lbox.detach(), lobj.detach(), lcls.detach())).detach()
        
        return loss * 3, self.loss_items
    
    def build_targets(self, p, targets):
        """
        Build targets for loss computation.
        """
        na, nt = self.na, targets.shape[0]
        tcls, tbox, indices, anch = [], [], [], []
        
        gain = torch.ones(7, device=self.device)
        ai = torch.arange(na, device=self.device).float().view(na, 1).repeat(1, nt)
        targets = torch.cat((targets.repeat(na, 1, 1), ai[..., None]), 2)
        
        g = 0.5
        off = torch.tensor(
            [[0, 0], [1, 0], [0, 1], [-1, 0], [0, -1]],
            device=self.device
        ).float() * g
        
        for i in range(self.nl):
            # Anchors are already stride-normalized in YOLOv5_3D.__init__
            anchors = self.anchors[i]
            
            shape = p[i].shape
            
            if len(shape) == 5:
                gain[2:6] = torch.tensor(shape)[[3, 2, 3, 2]]
            elif len(shape) == 4:
                gain[2:6] = torch.tensor(shape)[[2, 1, 2, 1]]
            elif len(shape) == 3:
                num_predictions = shape[1]
                grid_size = int((num_predictions / na) ** 0.5)
                gain[2:6] = torch.tensor([grid_size, grid_size, grid_size, grid_size],
                                        device=self.device, dtype=torch.float)
            
            t = targets * gain
            if nt:
                r = t[..., 4:6] / anchors[:, None]
                j = torch.max(r, 1 / r).max(2)[0] < self.hyp['anchor_t']
                t = t[j]
            else:
                t = targets[0]
            
            if t.shape[0] == 0:
                indices.append((torch.tensor([], dtype=torch.long, device=self.device),
                              torch.tensor([], dtype=torch.long, device=self.device),
                              torch.tensor([], dtype=torch.long, device=self.device),
                              torch.tensor([], dtype=torch.long, device=self.device)))
                tbox.append(torch.zeros((0, 4), device=self.device))
                anch.append(torch.zeros((0, 2), device=self.device))
                tcls.append(torch.tensor([], dtype=torch.long, device=self.device))
                continue
            
            gxy = t[:, 2:4]
            gxi = gain[[2, 3]] - gxy
            j, k = ((gxy % 1 < g) & (gxy > 1)).T
            l, m = ((gxi % 1 < g) & (gxi > 1)).T
            j = torch.stack((torch.ones_like(j), j, k, l, m))
            t = t.repeat((5, 1, 1))[j]
            offsets = (torch.zeros_like(gxy)[None] + off[:, None])[j]
            
            bc, gxy, gwh, a = t.chunk(4, 1)
            a, (b, c) = a.long().view(-1), bc.long().T
            gij = (gxy - offsets).long()
            gi, gj = gij.T
            
            max_h = int(gain[3].item())
            max_w = int(gain[2].item())
            gj = gj.clamp_(0, max_h - 1)
            gi = gi.clamp_(0, max_w - 1)
            
            indices.append((b, a, gj, gi))
            tbox.append(torch.cat((gxy - gij, gwh), 1))
            anch.append(self.anchors[i][a])
            tcls.append(c)
        
        return tcls, tbox, indices, anch

class ComputeLossAux:
    """
    Compute Loss for YOLOv5-3D with auxiliary illumination-depth loss.
    
    Adds auxiliary losses for the illumination-depth branch.
    """
    
    def __init__(self, model, hyp=None):
        """Initialize with auxiliary losses."""
        self.main_loss = ComputeLoss(model, hyp)
        self.device = next(model.parameters()).device
        
        # Auxiliary loss weights
        self.hyp = hyp or {}
        self.aux_weight = self.hyp.get('aux_weight', 0.1)
        
        # Reconstruction loss for depth/illumination consistency
        self.recon_loss = nn.MSELoss()
    
    def __call__(self, preds, targets, aux_outputs=None):
        """
        Compute total loss with auxiliary terms.
        
        Args:
            preds: Model predictions
            targets: Ground truth targets
            aux_outputs: Auxiliary outputs from illumination-depth branch
        
        Returns:
            total_loss, loss_items
        """
        # Main detection loss
        main_loss, loss_items = self.main_loss(preds, targets)
        
        # Auxiliary losses (if any)
        aux_loss = torch.zeros(1, device=self.device)
        
        if aux_outputs is not None:
            # Could add reconstruction consistency loss here
            pass
        
        total_loss = main_loss + self.aux_weight * aux_loss
        
        return total_loss, torch.cat([loss_items, aux_loss.detach().unsqueeze(0)])


### utils/metrics.py
Definition and code for `utils/metrics.py`.


In [ ]:
%%writefile utils/metrics.py
"""
Metrics and Evaluation Utilities for YOLOv5-3D
Implements mAP, Precision, Recall, F1, and Confusion Matrix.
"""

import numpy as np
import torch
from typing import List, Tuple, Dict, Optional
from collections import defaultdict


def box_iou(box1: np.ndarray, box2: np.ndarray) -> np.ndarray:
    """
    Calculate IoU between two sets of boxes.
    
    Args:
        box1: (N, 4) xyxy format
        box2: (M, 4) xyxy format
    
    Returns:
        IoU matrix (N, M)
    """
    def box_area(box):
        return (box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1])
    
    area1 = box_area(box1)
    area2 = box_area(box2)
    
    # Intersection
    lt = np.maximum(box1[:, None, :2], box2[:, :2])
    rb = np.minimum(box1[:, None, 2:], box2[:, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    
    # IoU
    iou = inter / (area1[:, None] + area2 - inter + 1e-6)
    return iou


def xywh2xyxy(x: np.ndarray) -> np.ndarray:
    """Convert boxes from xywh to xyxy format."""
    y = x.copy()
    y[..., 0] = x[..., 0] - x[..., 2] / 2  # x1
    y[..., 1] = x[..., 1] - x[..., 3] / 2  # y1
    y[..., 2] = x[..., 0] + x[..., 2] / 2  # x2
    y[..., 3] = x[..., 1] + x[..., 3] / 2  # y2
    return y


def non_max_suppression(
    prediction: np.ndarray,
    conf_threshold: float = 0.25,
    iou_threshold: float = 0.45,
    max_det: int = 300
) -> List[np.ndarray]:
    """
    Non-Maximum Suppression.
    
    Args:
        prediction: Model output (N, num_boxes, 5+nc)
        conf_threshold: Confidence threshold
        iou_threshold: IoU threshold for NMS
        max_det: Maximum detections per image
    
    Returns:
        List of detections per image: (n, 6) [x1, y1, x2, y2, conf, cls]
    """
    if isinstance(prediction, torch.Tensor):
        prediction = prediction.cpu().detach().numpy()
        
    bs = prediction.shape[0]  # batch size
    nc = prediction.shape[2] - 5  # number of classes
    output = [np.zeros((0, 6))] * bs
    
    for xi, x in enumerate(prediction):  # image index, image inference
        # Filter by confidence
        x = x.T  # (5+nc, num_boxes)
        box = x[:4].T  # (num_boxes, 4)
        conf = x[4:5].T * x[5:].T  # (num_boxes, nc) - conf = obj_conf * cls_conf
        conf = np.max(conf, axis=1, keepdims=True)
        cls = np.argmax(x[5:].T, axis=1, keepdims=True)
        
        # Concatenate
        x = np.concatenate([box, conf, cls], axis=1)
        
        # Filter by confidence threshold
        mask = x[:, 4] > conf_threshold
        x = x[mask]
        
        if x.shape[0] == 0:
            continue
        
        # Convert to xyxy
        x[:, :4] = xywh2xyxy(x[:, :4])
        
        # NMS per class
        c = x[:, 5:6] * 4096  # classes scaled for separate NMS
        boxes, scores = x[:, :4] + c, x[:, 4]
        
        # Simple NMS implementation
        ious = box_iou(boxes, boxes)
        
        # Greedy NMS
        keep = []
        order = scores.argsort()[::-1]
        
        while order.size > 0:
            i = order[0]
            keep.append(i)
            
            if len(keep) >= max_det:
                break
            
            # Remove boxes with high IoU
            mask = ious[i, order[1:]] <= iou_threshold
            order = order[1:][mask]
        
        # Save results
        output[xi] = x[keep]
    
    return output


def ap_per_class(
    tp: np.ndarray,
    conf: np.ndarray,
    pred_cls: np.ndarray,
    target_cls: np.ndarray,
    eps: float = 1e-16
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute AP per class.
    
    Args:
        tp: True positives
        conf: Confidence scores
        pred_cls: Predicted classes
        target_cls: Target classes
        eps: Small constant
    
    Returns:
        p, r, f1, ap, unique_classes
    """
    # Sort by confidence
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]
    
    # Unique classes
    unique_classes = np.unique(target_cls)
    nc = unique_classes.shape[0]
    
    # Create precision-recall curve
    px = np.linspace(0, 1, 1000)
    ap = np.zeros((nc, tp.shape[1]))
    p = np.zeros((nc, 1000))
    r = np.zeros((nc, 1000))
    
    for ci, c in enumerate(unique_classes):
        i = pred_cls == c
        n_l = (target_cls == c).sum()  # number of labels
        n_p = i.sum()  # number of predictions
        
        if n_p == 0 or n_l == 0:
            continue
        
        # Accumulate FPs and TPs
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        
        # Recall
        recall = tpc / (n_l + eps)
        r[ci] = np.interp(-px, -conf[i], recall[:, 0], left=0)
        
        # Precision
        precision = tpc / (tpc + fpc)
        p[ci] = np.interp(-px, -conf[i], precision[:, 0], left=1)
        
        # AP from recall-precision curve
        for j in range(tp.shape[1]):
            ap[ci, j] = compute_ap(recall[:, j], precision[:, j])
    
    # F1
    f1 = 2 * p * r / (p + r + eps)
    
    # CRITICAL FALLBACK: Mathematically ensure nothing exceeds 1.0 
    # due to edge cases in FP/TP accumulation or interpolation
    p = np.clip(p, 0.0, 1.0)
    r = np.clip(r, 0.0, 1.0)
    f1 = np.clip(f1, 0.0, 1.0)
    ap = np.clip(ap, 0.0, 1.0)
    
    return p, r, f1, ap, unique_classes


def compute_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """
    Compute AP using 11-point or all-point interpolation.
    """
    # Append sentinel values
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    
    # Ensure precision is monotonically decreasing
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    
    # Find points where recall changes
    i = np.where(mrec[1:] != mrec[:-1])[0]
    
    # Sum areas under the curve
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    
    return ap


class MetricTracker:
    """
    Track and compute metrics during training and evaluation.
    """
    
    def __init__(self, num_classes: int, iou_threshold: float = 0.5):
        """
        Initialize MetricTracker.
        
        Args:
            num_classes: Number of classes
            iou_threshold: IoU threshold for evaluation
        """
        self.num_classes = num_classes
        self.iou_threshold = iou_threshold
        
        # Store predictions and targets
        self.predictions = []
        self.targets = []
        
        # Stats
        self.stats = []
    
def update(self, preds: List[np.ndarray], target_batch: np.ndarray, img_shapes: List[Tuple[int, int]]):
    """
    Update tracker with predictions and targets.
    """
    # ===== FIX: Use np.unique for numpy arrays =====
    batch_indices = np.unique(target_batch[:, 0]) if target_batch.shape[0] > 0 else np.array([])
    # ===== END FIX =====
    
    for i, det in enumerate(preds):
        # Get targets for this image
        if len(batch_indices) > 0:
            mask = target_batch[:, 0] == i
            t = target_batch[mask]
            labels = t[:, 1:] if t.shape[0] > 0 else np.zeros((0, 5))
        else:
            labels = np.zeros((0, 5))
        
        # Scale boxes to original image size
        if det.shape[0] > 0 and i < len(img_shapes):
            h, w = img_shapes[i]
            det[:, [0, 2]] *= w
            det[:, [1, 3]] *= h
        
        if labels.shape[0] > 0:
            # Convert labels from xywh to xyxy
            labels_xyxy = labels.copy()
            labels_xyxy[:, 1:] = xywh2xyxy(labels[:, 1:])
            labels_xyxy[:, 1] *= img_shapes[i][1] if i < len(img_shapes) else 1
            labels_xyxy[:, 3] *= img_shapes[i][1] if i < len(img_shapes) else 1
            labels_xyxy[:, 2] *= img_shapes[i][0] if i < len(img_shapes) else 1
            labels_xyxy[:, 4] *= img_shapes[i][0] if i < len(img_shapes) else 1
        else:
            labels_xyxy = np.zeros((0, 5))
        
        # Store for evaluation
        self.predictions.append(det)
        self.targets.append(labels_xyxy)
        
        # Compute stats
        nl = len(labels_xyxy)
        tcls = labels_xyxy[:, 0].tolist() if nl else []
        
        if len(det) == 0:
            if nl:
                self.stats.append((np.zeros((0, 1), dtype=bool), 
                                  np.array([]), 
                                  np.array([]), 
                                  tcls))
            continue
        
        # Predictions
        det_label = det[:, 5].astype(int)
        det_conf = det[:, 4]
        det_box = det[:, :4]
        
        if nl:
            # Compute IoU
            correct = np.zeros((len(det), 1), dtype=bool)
            matched_gt = np.zeros(nl, dtype=bool)
            iou_matrix = box_iou(det_box, labels_xyxy[:, 1:])
            
            # Match predictions to labels
            for j in range(len(det)):
                max_iou = 0
                best_k = -1
                for k in range(nl):
                    if det_label[j] == int(labels_xyxy[k, 0]):
                        if iou_matrix[j, k] > max_iou:
                            max_iou = iou_matrix[j, k]
                            best_k = k
                
                if best_k >= 0 and max_iou > self.iou_threshold and not matched_gt[best_k]:
                    correct[j] = True
                    matched_gt[best_k] = True
        
            self.stats.append((correct, det_conf, det_label, tcls))
        else:
            self.stats.append((np.zeros((len(det), 1), dtype=bool), 
                              det_conf, det_label, tcls))
    
    def compute_metrics(self) -> Dict[str, float]:
        """
        Compute all metrics.
        
        Returns:
            Dictionary with metrics: mAP@0.5, mAP@0.5:0.95, precision, recall, f1
        """
        if not self.stats:
            return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0, 'f1': 0}
        
        # Concatenate stats
        stats = [np.concatenate(x, 0) for x in zip(*self.stats)]
        
        if len(stats) == 0 or stats[0].shape[0] == 0:
            return {'mAP50': 0, 'mAP50-95': 0, 'precision': 0, 'recall': 0, 'f1': 0}
        
        # Compute AP
        p, r, f1, ap, unique_classes = ap_per_class(*stats)
        
        # mAP@0.5
        mAP50 = ap[:, 0].mean() if ap.shape[0] > 0 else 0
        
        # mAP@0.5:0.95
        mAP50_95 = ap.mean() if ap.size > 0 else 0
        
        # Mean precision and recall at conf threshold
        precision = p.mean() if p.size > 0 else 0
        recall = r.mean() if r.size > 0 else 0
        f1_score = f1.mean() if f1.size > 0 else 0
        
        return {
            'mAP50': mAP50,
            'mAP50-95': mAP50_95,
            'precision': precision,
            'recall': recall,
            'f1': f1_score,
            'ap_per_class': ap
        }
    
    def reset(self):
        """Reset tracker."""
        self.predictions = []
        self.targets = []
        self.stats = []


class ConfusionMatrix:
    """
    Confusion Matrix for object detection evaluation.
    """
    
    def __init__(self, num_classes: int, iou_threshold: float = 0.5):
        """
        Initialize Confusion Matrix.
        
        Args:
            num_classes: Number of classes
            iou_threshold: IoU threshold for matching
        """
        self.num_classes = num_classes
        self.iou_threshold = iou_threshold
        self.matrix = np.zeros((num_classes + 1, num_classes + 1))
    
    def process_batch(self, detections: np.ndarray, labels: np.ndarray):
        """
        Process a batch of detections and labels.
        
        Args:
            detections: (N, 6) [x1, y1, x2, y2, conf, cls]
            labels: (M, 5) [cls, x1, y1, x2, y2]
        """
        if detections.shape[0] == 0:
            if labels.shape[0] > 0:
                for label in labels:
                    self.matrix[int(label[0]), self.num_classes] += 1  # FN
            return
        
        if labels.shape[0] == 0:
            for det in detections:
                self.matrix[self.num_classes, int(det[5])] += 1  # FP
            return
        
        # Compute IoU
        iou_matrix = box_iou(detections[:, :4], labels[:, 1:])
        
        # Match detections to labels
        matched = np.zeros(labels.shape[0], dtype=bool)
        
        for i, det in enumerate(detections):
            det_cls = int(det[5])
            
            # Find best matching label
            if iou_matrix.shape[1] > 0:
                j = iou_matrix[i].argmax()
                iou = iou_matrix[i, j]
                
                if iou > self.iou_threshold and not matched[j]:
                    label_cls = int(labels[j, 0])
                    if det_cls == label_cls:
                        self.matrix[label_cls, det_cls] += 1  # TP
                    else:
                        self.matrix[label_cls, det_cls] += 1  # Wrong class
                        self.matrix[self.num_classes, det_cls] += 1  # FP
                    matched[j] = True
                else:
                    self.matrix[self.num_classes, det_cls] += 1  # FP
            else:
                self.matrix[self.num_classes, det_cls] += 1  # FP
        
        # Unmatched labels are FN
        for j, m in enumerate(matched):
            if not m:
                self.matrix[int(labels[j, 0]), self.num_classes] += 1  # FN
    
    def get_matrix(self) -> np.ndarray:
        """Get the confusion matrix."""
        return self.matrix
    
    def tp_fp_fn(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Get TP, FP, FN per class."""
        tp = np.diag(self.matrix)[:self.num_classes]
        fp = self.matrix[self.num_classes, :self.num_classes]
        fn = self.matrix[:self.num_classes, self.num_classes]
        return tp, fp, fn
    
    def reset(self):
        """Reset the matrix."""
        self.matrix = np.zeros((self.num_classes + 1, self.num_classes + 1))


def compute_class_weights(dataset) -> torch.Tensor:
    """
    Compute class weights based on class frequency efficiently.
    Directly parses the text labels to bypass massive augmentation overheads.
    """
    class_counts = np.zeros(dataset.num_classes)
    
    for img_path in dataset.image_files:
        label_path = dataset._get_label_path(img_path)
        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = int(parts[0])
                        if cls_id < dataset.num_classes:
                            class_counts[cls_id] += 1
    
    # Inverse frequency weighting
    class_weights = 1.0 / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    
    return torch.from_numpy(class_weights).float()



### utils/plots.py
Definition and code for `utils/plots.py`.


In [ ]:
%%writefile utils/plots.py
"""
Visualization and Plotting Utilities for YOLOv5-3D
Training curves, confusion matrix, detection visualization.
"""

import os
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import cv2


# Color palette for classes
COLORS = [
    (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
    (255, 0, 255), (0, 255, 255), (128, 0, 0), (0, 128, 0),
    (0, 0, 128), (128, 128, 0), (128, 0, 128), (0, 128, 128),
    (64, 0, 0), (0, 64, 0), (0, 0, 64), (64, 64, 0),
]


def plot_training_curves(
    results: Dict[str, List[float]],
    save_dir: str,
    epochs: int
):
    """
    Plot training curves (loss, mAP, precision, recall).
    
    Args:
        results: Dictionary with metric histories
        save_dir: Directory to save plots
        epochs: Total number of epochs
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Create figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('YOLOv5-3D Training Results', fontsize=14, fontweight='bold')
    
    epochs_range = list(range(1, epochs + 1))
    
    # Plot 1: Losses
    ax1 = axes[0, 0]
    if 'train_loss' in results:
        ax1.plot(epochs_range, results['train_loss'], label='Train Loss', color='blue', linewidth=2)
    if 'val_loss' in results:
        ax1.plot(epochs_range, results['val_loss'], label='Val Loss', color='red', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    
    # Plot 2: mAP
    ax2 = axes[0, 1]
    if 'mAP50' in results:
        ax2.plot(epochs_range, results['mAP50'], label='mAP@0.5', color='green', linewidth=2)
    if 'mAP50_95' in results:
        ax2.plot(epochs_range, results['mAP50_95'], label='mAP@0.5:0.95', color='orange', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('mAP')
    ax2.set_title('Mean Average Precision')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2.set_ylim([0, 1])
    
    # Plot 3: Precision & Recall
    ax3 = axes[1, 0]
    if 'precision' in results:
        ax3.plot(epochs_range, results['precision'], label='Precision', color='purple', linewidth=2)
    if 'recall' in results:
        ax3.plot(epochs_range, results['recall'], label='Recall', color='cyan', linewidth=2)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Score')
    ax3.set_title('Precision & Recall')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax3.set_ylim([0, 1])
    
    # Plot 4: F1 Score
    ax4 = axes[1, 1]
    if 'f1' in results:
        ax4.plot(epochs_range, results['f1'], label='F1 Score', color='magenta', linewidth=2)
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('F1')
    ax4.set_title('F1 Score')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax4.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(save_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SAVED] Training curves: {save_dir / 'training_curves.png'}")


def plot_confusion_matrix(
    matrix: np.ndarray,
    class_names: List[str],
    save_dir: str,
    normalize: bool = True
):
    """
    Plot confusion matrix.
    
    Args:
        matrix: Confusion matrix array
        class_names: List of class names
        save_dir: Directory to save
        normalize: Whether to normalize the matrix
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    # Remove background class if present
    if matrix.shape[0] > len(class_names):
        matrix = matrix[:len(class_names), :len(class_names)]
    
    if normalize:
        matrix = matrix.astype('float') / (matrix.sum(axis=1, keepdims=True) + 1e-6)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot heatmap
    sns.heatmap(
        matrix,
        annot=True,
        fmt='.2f' if normalize else 'd',
        cmap='Blues',
        xticklabels=class_names + ['FP'],
        yticklabels=class_names + ['FN'],
        ax=ax,
        cbar_kws={'label': 'Normalized Count' if normalize else 'Count'}
    )
    
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title('Confusion Matrix' + (' (Normalized)' if normalize else ''), fontsize=14)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SAVED] Confusion matrix: {save_dir / 'confusion_matrix.png'}")


def plot_pr_curve(
    precision: np.ndarray,
    recall: np.ndarray,
    class_names: List[str],
    save_dir: str,
    ap: Optional[np.ndarray] = None
):
    """
    Plot Precision-Recall curves for each class.
    
    Args:
        precision: Precision values per class
        recall: Recall values per class
        class_names: List of class names
        save_dir: Directory to save
        ap: AP values per class
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot PR curve for each class
    for i, name in enumerate(class_names):
        if i < precision.shape[0]:
            label = f'{name}'
            if ap is not None and i < ap.shape[0]:
                label += f' (AP={ap[i, 0]:.3f})'
            ax.plot(recall[i], precision[i], label=label, linewidth=2)
    
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title('Precision-Recall Curves', fontsize=14)
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(save_dir / 'pr_curve.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SAVED] PR curve: {save_dir / 'pr_curve.png'}")


def plot_f1_confidence(
    f1: np.ndarray,
    confidence: np.ndarray,
    class_names: List[str],
    save_dir: str
):
    """
    Plot F1 score vs confidence threshold.
    
    Args:
        f1: F1 values per class
        confidence: Confidence thresholds
        class_names: List of class names
        save_dir: Directory to save
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for i, name in enumerate(class_names):
        if i < f1.shape[0]:
            ax.plot(confidence, f1[i], label=name, linewidth=2)
    
    ax.set_xlabel('Confidence Threshold', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score vs Confidence Threshold', fontsize=14)
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(save_dir / 'f1_confidence.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SAVED] F1-confidence curve: {save_dir / 'f1_confidence.png'}")


def plot_detection_results(
    image: np.ndarray,
    detections: np.ndarray,
    class_names: List[str],
    save_path: str,
    conf_threshold: float = 0.25
):
    """
    Visualize detection results on image.
    
    Args:
        image: RGB image (H, W, 3)
        detections: (N, 6) [x1, y1, x2, y2, conf, cls]
        class_names: List of class names
        save_path: Path to save visualization
        conf_threshold: Confidence threshold
    """
    # Create figure
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    # Draw each detection
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        
        if conf < conf_threshold:
            continue
        
        cls = int(cls)
        color = COLORS[cls % len(COLORS)]
        color_normalized = (color[0] / 255, color[1] / 255, color[2] / 255)
        
        # Draw box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color_normalized, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw label
        label = f'{class_names[cls]} {conf:.2f}'
        ax.text(
            x1, y1 - 5, label,
            fontsize=10, color='white',
            bbox=dict(facecolor=color_normalized, alpha=0.7, edgecolor='none'),
            verticalalignment='bottom'
        )
    
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0.1)
    plt.close()


def plot_sample_batch(
    images: np.ndarray,
    depths: np.ndarray,
    illuminations: np.ndarray,
    save_path: str,
    num_samples: int = 4
):
    """
    Visualize sample batch with RGB, depth, and illumination.
    
    Args:
        images: Batch of images (B, C, H, W)
        depths: Batch of depth maps (B, C, H, W)
        illuminations: Batch of illumination maps (B, C, H, W)
        save_path: Path to save visualization
        num_samples: Number of samples to visualize
    """
    num_samples = min(num_samples, images.shape[0])
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    for i in range(num_samples):
        # RGB image
        img = images[i].transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
        if img.max() > 1:
            img = img / 255.0
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'RGB Image {i+1}')
        axes[i, 0].axis('off')
        
        # Depth map
        depth = depths[i, 0]  # Single channel
        axes[i, 1].imshow(depth, cmap='viridis')
        axes[i, 1].set_title(f'Depth Map {i+1}')
        axes[i, 1].axis('off')
        
        # Illumination map
        illum = illuminations[i].transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
        if illum.max() > 1:
            illum = illum / illum.max()
        axes[i, 2].imshow(illum)
        axes[i, 2].set_title(f'Illumination {i+1}')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"[SAVED] Sample batch: {save_path}")


def plot_loss_components(
    box_loss: List[float],
    obj_loss: List[float],
    cls_loss: List[float],
    save_dir: str
):
    """
    Plot individual loss components.
    
    Args:
        box_loss: Box loss history
        obj_loss: Objectness loss history
        cls_loss: Classification loss history
        save_dir: Directory to save
    """
    save_dir = Path(save_dir)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    epochs = range(1, len(box_loss) + 1)
    
    ax.plot(epochs, box_loss, label='Box Loss', linewidth=2)
    ax.plot(epochs, obj_loss, label='Objectness Loss', linewidth=2)
    ax.plot(epochs, cls_loss, label='Classification Loss', linewidth=2)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss Components')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'loss_components.png', dpi=150, bbox_inches='tight')
    plt.close()


def plot_lr_schedule(
    lrs: List[float],
    save_dir: str
):
    """
    Plot learning rate schedule.
    
    Args:
        lrs: Learning rate history
        save_dir: Directory to save
    """
    save_dir = Path(save_dir)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    
    ax.plot(range(1, len(lrs) + 1), lrs, linewidth=2, color='green')
    
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Learning Rate Schedule')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(save_dir / 'lr_schedule.png', dpi=150, bbox_inches='tight')
    plt.close()


def save_results_csv(
    results: Dict[str, List[float]],
    save_path: str
):
    """
    Save training results to CSV file.
    
    Args:
        results: Dictionary with metric histories
        save_path: Path to save CSV
    """
    import csv
    
    save_path = Path(save_path)
    
    # Get all keys and find max length
    keys = list(results.keys())
    max_len = max(len(results[k]) for k in keys)
    
    with open(save_path, 'w', newline='') as f:
        writer = csv.writer(f)
        
        # Header
        writer.writerow(['epoch'] + keys)
        
        # Data
        for i in range(max_len):
            row = [i + 1]
            for k in keys:
                if i < len(results[k]):
                    row.append(f'{results[k][i]:.6f}')
                else:
                    row.append('')
            writer.writerow(row)
    
    print(f"[SAVED] Results CSV: {save_path}")


class TrainingVisualizer:
    """
    Class to manage all training visualizations.
    """
    
    def __init__(self, save_dir: str, class_names: List[str]):
        """
        Initialize TrainingVisualizer.
        
        Args:
            save_dir: Directory to save plots
            class_names: List of class names
        """
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.class_names = class_names
        
        # History
        self.results = {
            'train_loss': [],
            'val_loss': [],
            'mAP50': [],
            'mAP50_95': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'box_loss': [],
            'obj_loss': [],
            'cls_loss': [],
            'lr': []
        }
    
    def update(self, metrics: Dict[str, float], lr: float = None):
        """
        Update history with new metrics.
        
        Args:
            metrics: Dictionary with current epoch metrics
            lr: Current learning rate
        """
        for key in ['train_loss', 'val_loss', 'mAP50', 'mAP50_95', 
                    'precision', 'recall', 'f1', 'box_loss', 'obj_loss', 'cls_loss']:
            if key in metrics:
                self.results[key].append(metrics[key])
            else:
                self.results[key].append(0)
        
        if lr is not None:
            self.results['lr'].append(lr)
    
    def plot_all(self, epoch: int):
        """
        Generate all plots.
        
        Args:
            epoch: Current epoch number
        """
        # Training curves
        plot_training_curves(self.results, self.save_dir, epoch)
        
        # Loss components
        if any(self.results['box_loss']):
            plot_loss_components(
                self.results['box_loss'],
                self.results['obj_loss'],
                self.results['cls_loss'],
                self.save_dir
            )
        
        # LR schedule
        if self.results['lr']:
            plot_lr_schedule(self.results['lr'], self.save_dir)
        
        # Save CSV
        save_results_csv(self.results, self.save_dir / 'results.csv')
    
    def save_final(self):
        """Save all final results."""
        self.plot_all(len(self.results['train_loss']))
        print(f"\n[INFO] All visualizations saved to: {self.save_dir}")



### utils/__init__.py
Definition and code for `utils/__init__.py`.


In [ ]:
%%writefile utils/__init__.py
# YOLOv5-3D Utilities Package
from .datasets import PCBDataset, DataHandler, create_dataloaders, collate_fn
from .loss import ComputeLoss, ComputeLossAux, bbox_iou, FocalLoss
from .metrics import (
    MetricTracker, ConfusionMatrix,
    box_iou, xywh2xyxy, non_max_suppression,
    ap_per_class, compute_ap, compute_class_weights
)
from .plots import (
    TrainingVisualizer,
    plot_training_curves, plot_confusion_matrix,
    plot_pr_curve, plot_detection_results,
    plot_sample_batch
)
from .general import (
    colorstr, set_logging, increment_path,
    check_dataset, initialize_weights,
    select_device, check_img_size, scale_coords, make_divisible
)

__all__ = [
    # Datasets
    'PCBDataset', 'DataHandler', 'create_dataloaders', 'collate_fn',
    # Loss
    'ComputeLoss', 'ComputeLossAux', 'bbox_iou', 'FocalLoss',
    # Metrics
    'MetricTracker', 'ConfusionMatrix',
    'box_iou', 'xywh2xyxy', 'non_max_suppression',
    'ap_per_class', 'compute_ap', 'compute_class_weights',
    # Plots
    'TrainingVisualizer',
    'plot_training_curves', 'plot_confusion_matrix',
    'plot_pr_curve', 'plot_detection_results', 'plot_sample_batch',
    # General
    'colorstr', 'set_logging', 'increment_path',
    'check_dataset', 'initialize_weights',
    'select_device',
    'check_img_size', 'scale_coords', 'make_divisible'
]



### data/hyp.scratch.yaml
Definition and code for `data/hyp.scratch.yaml`.


In [ ]:
%%writefile data/hyp.scratch.yaml
# YOLOv5-3D Hyperparameters
# Optimized for PCB defect detection with illumination-depth fusion

# Learning rate
lr0: 0.01              # Initial learning rate (SGD=1E-2, Adam=1E-3)
lrf: 0.01              # Final learning rate (lr0 * lrf)
momentum: 0.937        # SGD momentum/Adam beta1
weight_decay: 0.0005   # Optimizer weight decay

# Warmup
warmup_epochs: 3       # Warmup epochs
warmup_momentum: 0.8   # Warmup initial momentum
warmup_bias_lr: 0.1    # Warmup initial bias learning rate

# Loss gains
box: 0.05              # Box loss gain
cls: 0.5               # Classification loss gain
cls_pw: 1.0            # Classification BCELoss positive_weight
obj: 1.0               # Objectness loss gain
obj_pw: 1.0            # Objectness BCELoss positive_weight

# Anchor settings
anchor_t: 4.0          # Anchor-multiple threshold

# Data augmentation
hsv_h: 0.015           # HSV-Hue augmentation
hsv_s: 0.7             # HSV-Saturation augmentation
hsv_v: 0.4             # HSV-Value augmentation
degrees: 0             # Rotation (+/- deg)
translate: 0.1         # Translation (+/- fraction)
scale: 0.9             # Scale (+/- gain)
shear: 0               # Shear (+/- deg)
perspective: 0         # Perspective (+/- fraction)
flipud: 0.0            # Flip up-down probability
fliplr: 0.5            # Flip left-right probability
mosaic: 1.0            # Mosaic augmentation probability
mixup: 0.1             # Mixup augmentation probability
copy_paste: 0.0        # Copy-paste augmentation probability

# Other
fl_gamma: 0.0          # Focal loss gamma (0 = disabled)
label_smoothing: 0.0   # Label smoothing (fraction)

# Illumination-Depth specific
aux_weight: 0.1        # Auxiliary loss weight for illumination-depth branch
fusion_weight: 0.3     # Fusion module learning rate multiplier



### data/pcb.yaml
Definition and code for `data/pcb.yaml`.


In [ ]:
%%writefile data/pcb.yaml
# PCB Defect Detection Dataset Configuration
# YOLOv5-3D with Illumination-Depth Support

# Dataset path
path: D:\Data_Tuning\PCB_final

# Split paths (relative to path)
train: train\images
val:   valid\images
test:  test\images

# Number of classes
nc: 12

# Class names
names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper
  6: Missing_solder
  7: Short_top
  8: Excess_solder
  9: Insufficient_solder
  10: Missing_component
  11: Spike

# ================================================================================================
# Illumination3D specific settings
# ================================================================================================
nc_illum: 3  # RGB illumination channels
nc_depth: 1  # Single depth channel



### data/__init__.py
Definition and code for `data/__init__.py`.


In [ ]:
%%writefile data/__init__.py
# YOLOv5-3D Scripts Package



## 3. Training Execution
Execute the training script.


In [ ]:
!python train.py --img 640 --batch 16 --epochs 100 --data data/pcb_3d.yaml --cfg models/yolov5_3d.yaml --weights yolov5s.pt --name pcb_defect_3d


## 4. Evaluation Execution
Evaluate the trained weights.


In [ ]:
!python kaggle_eval.py --data-dir D:/Data_Tuning/PCB_final --weights runs/train/pcb_defect_3d/weights/best.pt --imgsz 640
